In [1]:
# Clears out any variables, just to be safe.
%reset -f

import tkinter as tk
from tkinter import * # Keep this for constants like HORIZONTAL, W, etc.
from tkinter import ttk
from tkinter import filedialog
from tkinter.ttk import Progressbar
import pandas as pd
import matplotlib.pyplot as plt
from  matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
import plotly.graph_objects as go
import numpy as np
import math
import os
import re
import sys
import json
import ast
from time import perf_counter
from time import perf_counter_ns
import seaborn as sns
from scipy.optimize import curve_fit

In [2]:
# --- Global variables for file paths ---
data_filename = ""
temp_profile_file = ""
layout_file = ""
mating_table_file = ""
fur_file = ""
hps_file = ""
runData = [] # This will store all final data for plotting/export
runBool = False
imageViewer = None # For the main Moles/P/T plot
temp_viewer_window = None
baselineEdited = False
z_temp_cutoff_var = None
z_offset_var = None
p_calc_start_time_var = None
interpolate_temps_var = None
plot_fig = None      # Global figure reference for fit overlay
plot_ax2 = None      # Global pressure axis reference for fit overlay
plot_canvas = None   # Global canvas reference for fit overlay
fit_results_csv_path = ""   # Persistent path for appending fit results
dn_dt_start_var = None   # Widget var for dn/dt start time
dn_dt_end_var   = None   # Widget var for dn/dt end time
dn_dt_button    = None   # dn/dt Calculate button
dn_dt_csv_path = ""         # Persistent path for dn/dt results
dn_dt_csv_label = None      # Label widget showing dn/dt CSV path
dn_dt_interval_var = None  # Widget var for dn/dt interval width
run_id = ""             # RunID extracted from data file
calc_step_var = None       # Widget var for calculation step interval
output_dir = ""             # Output folder for auto-saved files
info_log = []               # Collects run info for the info window/file
info_window = None          # Toplevel for displaying run info
plot_ax4 = None            # 4th y-axis for dn/dt
dn_dt_results_label = None # Results label in dn/dt section
info_text_widget = None    # Direct ref to ScrolledText in info window
pcalc_button = None        # Calculate P_calc button
save_plot_button = None    # Save plot PNG/SVG button
export_fit_button = None   # Export fit CSV button
last_fit_result = None     # Stores most recent fit result for export
fit_csv_path = ""          # Persistent fit CSV path for this run
p_offset_var = None        # StringVar for P_calc pressure offset (MPa)
pcalc_info_label = None    # Label showing auto-calculated offset
p_calc_end_time_var = None   # End time for P_calc window
p_calc_step_var = None       # Step interval for P_calc loop
p_zero_start_var = None    # Vacuum window start time for P zero offset
p_zero_end_var   = None    # Vacuum window end time for P zero offset
p_zero_manual_var = None   # Manual P zero override (MPa)
line_thickness_var = None  # DoubleVar: curve linewidth

# --- Furniture table globals ---
fur_mass_entries    = {}    # UID -> StringVar (mass in grams)
fur_internal_checks = {}    # UID -> BooleanVar (is internal/subtractive)
fur_table_inner     = None  # Inner Frame holding table rows
_internal_uids_set  = set() # UIDs from MLD-FUR sheets
vol_info_label      = None  # Label showing vol before/after correction

# --- Chemical species globals ---
chem_species_var  = None  # StringVar: selected species
chem_mass_var     = None  # StringVar: mass in grams
chem_density_var  = None  # StringVar: custom density g/cm³

pressure_col_var  = None   # StringVar: selected PT pressure column


In [3]:
# --- Fit & Uncertainty Widgets ---
fit_start_time_var = None
fit_end_time_var = None
fit_results_label = None
calculateFitButton = None
sigma_p_var = None # <-- NEW
sigma_v_var = None # <-- NEW
sigma_t_var = None # <-- NEW
# ---

exportTempButton = None # Initialize button variables
tempViewButton = None
exportPlotButton = None
importTempButton = NoneexportHtmlButton = None  # Export HTML plot button
gas_species_var = None  # Gas species for Z lookup
texpansion_button = None

# fur_mass_var removed — replaced by fur_mass_entries table
_sorted_tc_idx   = None      # Pre-computed sorted TC index order for interpolation
_x_tc_sorted     = None      # Pre-computed sorted TC positions (mm)
_x_target_interp = None      # Pre-computed interp target array


In [4]:
# --- Pressure Trace Synthesizer globals ---
synth_p0_var         = None   # StringVar: initial pressure (MPa)
synth_t0_var         = None   # StringVar: initial temperature for n calc (degC)
synth_ambient_var    = None   # StringVar: ambient / cold-end temperature (degC)
synth_total_time_var = None   # StringVar: total simulation time (hr)
synth_dt_var         = None   # StringVar: time step (hr)
synth_profile_mode_var = None # StringVar: 'uniform' | 'gradient' | 'profile_csv' | 'loaded'
synth_spatial_file   = ''     # Path to 1-D spatial profile CSV (z_mm, T_C)
synth_spatial_label  = None   # Label widget showing spatial file name
synth_schedule_text  = None   # Text widget: furnace schedule (time, T pairs)
synth_n_label        = None   # Label showing derived n (mol)
synth_status_label   = None   # Status / results label
synth_run_button     = None   # Run Synthesizer button
synth_coarse_vol     = None   # Computed coarseVol (m3/bin) without raw data
synth_loaded_profile = None   # Tuple (z_arr, T_arr_K) for 'loaded' spatial mode
synth_vol_label      = None   # Label showing computed volume info
synth_lookup_table   = None   # pd.DataFrame: rows=TC setpoint (degC), cols=z_mm
synth_lookup_z_mm    = None   # np.array of z positions (mm) from CSV columns
synth_lookup_label   = None   # Label widget showing which profile CSV is loaded


In [5]:
# --- browseFiles: data + temp CSV only ---
def browseFiles():
    global data_filename, temp_profile_file, interpolate_temps_var

    data_filename = filedialog.askopenfilename(
        initialdir=os.path.expanduser(r'~\Lehigh University Dropbox'),
        title='1. Select Data CSV File',
        filetypes=(('CSV files', '*.csv*'), ('all files', '*.*'))
    )
    if data_filename:
        fileLabel.configure(text=f'Data: {os.path.basename(data_filename)}', fg='black')
        _populate_pt_dropdown(data_filename)
    else:
        fileLabel.configure(text='Data: (Not Selected)', fg='gray')

    start_dir = os.path.dirname(data_filename) if data_filename else os.path.expanduser('~')

    if not interpolate_temps_var.get():
        temp_profile_file = filedialog.askopenfilename(
            initialdir=start_dir,
            title='2. Select Temperature Profile CSV',
            filetypes=(('CSV files', '*.csv*'),)
        )
        if temp_profile_file:
            tempLabel.configure(text=f'Temp Profile: {os.path.basename(temp_profile_file)}',
                                state='normal', fg='black')
        else:
            tempLabel.configure(text='Temp Profile: (Not Selected)', state='normal', fg='gray')
    else:
        temp_profile_file = ''
        tempLabel.configure(text='Temp Profile: (Using Interpolation)', state='disabled', fg='gray')

    window.update_idletasks()


In [6]:
# --- PT column helper ---
def _populate_pt_dropdown(csv_path):
    """Read CSV headers, filter columns containing 'PT', update the pressure dropdown."""
    global pressure_col_var
    try:
        header = pd.read_csv(csv_path, nrows=0)
        pt_cols = [c for c in header.columns if 'PT' in c.upper()]
    except Exception as e:
        print(f'Warning: could not read PT columns: {e}')
        pt_cols = []

    if not pt_cols:
        pt_cols = ['(none found)']
        print('Warning: no PT columns found in data file.')
    else:
        print(f'PT columns found: {pt_cols}')

    if pressure_col_var is None:
        return   # GUI not yet built

    # Update OptionMenu choices — prefer A2 column if present
    _a2 = [c for c in pt_cols if 'A2' in c.upper()]
    pressure_col_var.set(_a2[0] if _a2 else pt_cols[0])
    menu = pt_menu['menu']
    menu.delete(0, 'end')
    for col in pt_cols:
        menu.add_command(label=col, command=lambda v=col: pressure_col_var.set(v))
    ptRow.configure(bg='white')
    ptLabel.configure(text=f'Pressure Col: ({len(pt_cols)} PT channel{"s" if len(pt_cols)>1 else ""} found)',
                      fg='black')


In [7]:
# --- Canonical geometry file paths ---
DROPBOX           = os.path.join(os.path.expanduser('~'), 'Lehigh University Dropbox')
CANON_MLD_DIR     = os.path.join(DROPBOX, r'ENG-MATSGroup/MATS - Collaboration/Data/Parsers/MolCalc/File Dependencies')
CANON_MOLCALC_DIR = os.path.join(DROPBOX, r'ENG-MATSGroup/MATS - Collaboration/Data\Parsers/MolCalc/File Dependencies')
CANON_LAYOUT_DIR  = CANON_MLD_DIR  # Layouts also live in File Dependencies

def _find_xlsx(directory, exact_stem):
    """Return the path to <exact_stem>.xlsx in directory (case-insensitive stem match),
    or None if the directory doesn't exist or the file isn't found."""
    if not os.path.isdir(directory):
        return None
    target = exact_stem.lower() + '.xlsx'
    for fname in os.listdir(directory):
        if fname.lower() == target:
            return os.path.join(directory, fname)
    return None

def _find_mating_table(directory):
    """Return the first .xlsx whose name contains 'mating', 'surface', or 'mate'."""
    if not os.path.isdir(directory):
        return None
    for kw in ['mating', 'surface', 'mate']:
        for fname in os.listdir(directory):
            if fname.lower().endswith('.xlsx') and kw in fname.lower():
                return os.path.join(directory, fname)
    return None

def _set_label_resolved(label, prefix, path):
    if path:
        label.configure(text=f'{prefix}: {os.path.basename(path)} (auto)', fg='darkgreen')
    else:
        label.configure(text=f'{prefix}: NOT FOUND — select manually', fg='red')
        from tkinter import messagebox
        messagebox.showwarning(
            'File Not Found',
            f'{prefix} was not found at the expected location:\n'
            f'{CANON_MLD_DIR}\n\n'
            f'Please use the Browse... button to locate the file manually.'
        )

def resolve_geometry_files():
    """Auto-resolve all geometry files from canonical paths; update globals + labels."""
    global fur_file, hps_file, mating_table_file, layout_file

    # MLD-FUR: look for exactly 'MLD-FUR.xlsx'
    fur_file = _find_xlsx(CANON_MLD_DIR, 'MLD - FUR')
    _set_label_resolved(furLabel, 'MLD-FUR', fur_file)
    global _internal_uids_set
    _internal_uids_set = set()
    if fur_file and os.path.isfile(fur_file):
        try:
            _xf2 = pd.ExcelFile(fur_file)
            for _sh2 in ['CRU', 'LID', 'PLU', 'VRE']:
                if _sh2 in _xf2.sheet_names:
                    _df2 = pd.read_excel(fur_file, sheet_name=_sh2, usecols=['UID'])
                    _internal_uids_set.update(_df2['UID'].dropna().astype(str).tolist())
            print(f'Internal UIDs detected: {_internal_uids_set}')
        except Exception as _e2:
            print(f'Warning: could not extract internal UIDs: {_e2}')

    # MLD-HPS: look for exactly 'MLD-HPS.xlsx'
    hps_file = _find_xlsx(CANON_MLD_DIR, 'MLD - HPS')
    _set_label_resolved(hpsLabel, 'MLD-HPS', hps_file)

    # Mating Table: search MolCalc root for any xlsx with 'mating', 'surface', or 'mate'
    mating_table_file = _find_mating_table(CANON_MOLCALC_DIR)
    _set_label_resolved(matingLabel, 'Mating Table', mating_table_file)

    # Layout — point dialog at the layouts folder; user always picks the file
    layout_dir = CANON_LAYOUT_DIR if os.path.isdir(CANON_LAYOUT_DIR) else os.path.expanduser('~')
    layout_file = filedialog.askopenfilename(
        initialdir=layout_dir,
        title='Select Autoclave Layout File',
        filetypes=(('Excel files', '*.xlsx*'),)
    )
    if layout_file:
        layoutLabel.configure(text=f'Layout: {os.path.basename(layout_file)}', fg='black')
    else:
        layoutLabel.configure(text='Layout: (Not Selected)', fg='gray')

    window.update_idletasks()
    if fur_table_inner is not None and layout_file:
        build_fur_table_from_layout()
    print(f'FUR:     {fur_file}')
    print(f'HPS:     {hps_file}')
    print(f'Mating:  {mating_table_file}')
    print(f'Layout:  {layout_file}')

# Individual override functions
def browse_fur():
    global fur_file
    p = filedialog.askopenfilename(
        initialdir=CANON_MLD_DIR if os.path.isdir(CANON_MLD_DIR) else os.path.expanduser('~'),
        title='Select MLD-FUR File', filetypes=(('Excel files', '*.xlsx*'),))
    if p:
        fur_file = p
        furLabel.configure(text=f'MLD-FUR: {os.path.basename(p)}', fg='black')
    window.update_idletasks()

def browse_hps():
    global hps_file
    p = filedialog.askopenfilename(
        initialdir=CANON_MLD_DIR if os.path.isdir(CANON_MLD_DIR) else os.path.expanduser('~'),
        title='Select MLD-HPS File', filetypes=(('Excel files', '*.xlsx*'),))
    if p:
        hps_file = p
        hpsLabel.configure(text=f'MLD-HPS: {os.path.basename(p)}', fg='black')
    window.update_idletasks()

def browse_mating():
    global mating_table_file
    p = filedialog.askopenfilename(
        initialdir=CANON_MOLCALC_DIR if os.path.isdir(CANON_MOLCALC_DIR) else os.path.expanduser('~'),
        title='Select Mating Table File', filetypes=(('Excel files', '*.xlsx*'),))
    if p:
        mating_table_file = p
        matingLabel.configure(text=f'Mating Table: {os.path.basename(p)}', fg='black')
    window.update_idletasks()

def browse_layout():
    global layout_file
    layout_dir = CANON_LAYOUT_DIR if os.path.isdir(CANON_LAYOUT_DIR) else os.path.expanduser('~')
    p = filedialog.askopenfilename(
        initialdir=layout_dir,
        title='Select Layout File', filetypes=(('Excel files', '*.xlsx*'),))
    if p:
        layout_file = p
        layoutLabel.configure(text=f'Layout: {os.path.basename(p)}', fg='black')
    window.update_idletasks()
    if fur_table_inner is not None and layout_file:
        build_fur_table_from_layout()


In [8]:
# --- Build Furniture Table from Layout ---
def build_fur_table_from_layout():
    """
    Populate the per-part furniture mass table from the currently loaded layout.
    Reads UIDs, marks internal parts (from _internal_uids_set), and creates
    one row per part with: UID label | material | Internal checkbox | mass entry.
    Safe to call repeatedly — clears and rebuilds each time.
    """
    global fur_mass_entries, fur_internal_checks, fur_table_inner

    if fur_table_inner is None:
        print('Warning: fur_table_inner not yet created — GUI not ready.'); return
    if not layout_file:
        from tkinter import messagebox
        messagebox.showwarning('No Layout', 'Load a Layout file first (via Resolve Geometry).'); return

    # Build UID → material map from whichever geometry files are available
    _uid_mat = {}
    for _path, _sheets, _mcol in [
        (fur_file, ['CRU','LID','PLU','VRE'], 'Chemicals'),
        (hps_file, ['AC','GKT','NZ','TU'],    'Material')
    ]:
        if _path and os.path.isfile(_path):
            try:
                _xf = pd.ExcelFile(_path)
                for _sh in _sheets:
                    if _sh in _xf.sheet_names:
                        _df = pd.read_excel(_path, sheet_name=_sh)
                        if 'UID' in _df.columns and _mcol in _df.columns:
                            for _, _r in _df[['UID', _mcol]].dropna().iterrows():
                                _uid_mat[str(_r['UID'])] = str(_r[_mcol])
            except Exception as _e:
                print(f'Warning: material read error from {os.path.basename(_path)}: {_e}')

    # Clear existing rows
    for w in fur_table_inner.winfo_children():
        w.destroy()
    fur_mass_entries.clear()
    fur_internal_checks.clear()

    try:
        _pl = pd.read_excel(layout_file)
    except Exception as _e:
        from tkinter import messagebox
        messagebox.showerror('Layout Error', f'Could not read layout: {_e}'); return

    _ROW_BG = ('#eef2f7', '#ffffff')
    _row_idx = 0
    for _, _row in _pl.iterrows():
        _uid = str(_row.get('UID', '')).strip()
        if not _uid or _uid.lower() == 'nan':
            continue
        _is_int  = _uid in _internal_uids_set
        _mat     = _uid_mat.get(_uid, '—')
        _bg      = _ROW_BG[_row_idx % 2]
        _row_idx += 1

        _f = tk.Frame(fur_table_inner, bg=_bg)
        _f.pack(fill=tk.X, padx=0, pady=0)

        tk.Label(_f, text=_uid,  width=16, anchor=W, bg=_bg,
                 font=('TkDefaultFont', 8)).pack(side=tk.LEFT, padx=(3,0))
        tk.Label(_f, text=_mat,  width=16, anchor=W, bg=_bg,
                 font=('TkDefaultFont', 8), fg='#555').pack(side=tk.LEFT)
        _int_var = tk.BooleanVar(value=_is_int)
        fur_internal_checks[_uid] = _int_var
        tk.Checkbutton(_f, variable=_int_var, bg=_bg).pack(side=tk.LEFT, padx=(2,6))
        _mass_sv = tk.StringVar(value='')
        fur_mass_entries[_uid] = _mass_sv
        tk.Entry(_f, textvariable=_mass_sv, width=9,
                 font=('TkDefaultFont', 8)).pack(side=tk.LEFT)

    fur_table_inner.update_idletasks()
    _n_int = sum(v.get() for v in fur_internal_checks.values())
    print(f'Furniture table: {len(fur_mass_entries)} parts ({_n_int} marked internal)')


In [9]:
# --- Toggle Temp File Label State ---
def toggle_temp_file_selection():
    global interpolate_temps_var, tempLabel, temp_profile_file
    if interpolate_temps_var.get():
        tempLabel.configure(text="Temp Profile: (Using Interpolation)", state='disabled')
        temp_profile_file = ""
    else:
        tempLabel.configure(text="Temp Profile: (Select Below)", state='normal')

In [10]:
# --- getData function ---
def getData(runDF_raw, EXP_ID):
    global graphLabel
    print("running getData")
    config_path = os.path.expanduser(r"~\Lehigh University Dropbox\ENG-MATSGroup\MATS\cRIO\RunConfig\EXPs\\") + EXP_ID + '.csv'
    print(config_path)
    try:
        TC_df = pd.read_csv(config_path, usecols=['Device Type', 'NI LV Channel','Shorthand Tag', 'IO Type'])
    except FileNotFoundError: graphLabel.configure(text=f"Error: TC config file not found for {EXP_ID}.csv"); return None, None, None
    except ValueError as e: graphLabel.configure(text=f"Error: Check columns in {EXP_ID}.csv"); return None, None, None
    original_rows = len(TC_df); TC_df.drop_duplicates(subset=['NI LV Channel'], keep='first', inplace=True)
    if len(TC_df) < original_rows: print(f"Removed {original_rows - len(TC_df)} duplicate NI LV Channels.")
    positions = []
    for name, type in zip(TC_df['Shorthand Tag'], TC_df['Device Type']):
        if type == 'TC':
            numbers_found = re.findall(r"\d+", str(name))
            if len(numbers_found) >= 2:
                try: positions.append(int(numbers_found[1]))
                except (ValueError, IndexError): print(f"Warn: Bad TC tag num: '{name}'."); positions.append(np.nan)
            else: print(f"Warn: <2 nums in TC tag: '{name}'."); positions.append(np.nan)
        else: positions.append(np.nan)
    TC_df['X Position'] = positions; TC_df = TC_df.dropna(subset=['X Position'])
    try: TC_df['X Position'] = TC_df['X Position'].astype(int)
    except ValueError: print("Warn: X Pos not int.")
    TC_df = TC_df.sort_values(by='X Position'); TC_positions=TC_df['X Position']
    print("TC_positions (valid & sorted):", TC_positions.tolist()); print("TC_df (valid TCs, unique channels):"); print(TC_df[['NI LV Channel', 'Shorthand Tag', 'X Position', 'IO Type']])
    RelevantColumns = TC_df['NI LV Channel'].to_list()
    if not RelevantColumns: graphLabel.configure(text=f"Error: No valid TCs in config."); return None, None, None
    unique_required_cols = list(dict.fromkeys(RelevantColumns + ["CV1-TC-Ambient"]))
    missing_cols = [col for col in unique_required_cols if col not in runDF_raw.columns]
    if missing_cols: graphLabel.configure(text=f"Error: Data missing cols: {missing_cols}"); return None, None, None
    try:
        runDF_selected = runDF_raw[unique_required_cols].iloc[8:]
        print("Converting temperature columns to numeric...")
        runDF_selected = runDF_selected.apply(pd.to_numeric, errors='coerce')
        # Use how='all' so only fully-empty rows are dropped.
        # how='any' (the default) discards every row that has a NaN in even
        # one TC channel, which empties the dataset if a single TC is missing.
        runDF_filtered = runDF_selected.dropna(how='all')
        n_dropped = len(runDF_selected) - len(runDF_filtered)
        if n_dropped: print(f'getData: dropped {n_dropped} fully-empty rows.')
        if runDF_filtered.empty: graphLabel.configure(text="Error: No valid data rows found after filtering & coercion."); return None, None, None
        print(f"Data rows remaining after filtering: {len(runDF_filtered)}")
        return runDF_filtered, TC_df, TC_positions
    except Exception as e: graphLabel.configure(text=f"Error filtering data: {e}"); return None, None, None


In [11]:
# getTempProfile removed — lookup now uses the central TC (tag "157") directly
# against the full lookup table index (column 1 of the CSV). See processData.


In [12]:
# --- interpolateTempProfile function ---
def interpolateTempProfile(current_temps, tc_positions_series):
    """
    Build a 1-mm-resolution temperature profile by linearly interpolating
    across ALL TC channels.

    Speed: sorted TC order, x_target, and _x_tc_sorted are pre-computed
    once before the main loop (_sorted_tc_idx, _x_tc_sorted, _x_target_interp)
    and reused every call.  Falls back to on-the-fly computation if those
    globals are not yet initialised (e.g. during unit testing).

    Previous behaviour used only three anchor points (min-pos / peak / max-pos),
    discarding all intermediate TCs.  This version uses every TC channel.
    """
    global _sorted_tc_idx, _x_tc_sorted, _x_target_interp
    try:
        y_raw = [float(t) for t in current_temps]
    except (ValueError, TypeError) as e:
        print(f"Error: Non-numeric temp reading in interpolateTempProfile: {e}"); return []

    # --- Use pre-computed arrays when available (normal run-time path) ---
    if _sorted_tc_idx is not None and _x_tc_sorted is not None and _x_target_interp is not None:
        if len(y_raw) != len(_sorted_tc_idx):
            print(f"Error: TC count mismatch in interpolateTempProfile "
                  f"({len(y_raw)} temps vs {len(_sorted_tc_idx)} positions)."); return []
        y_known = np.array(y_raw)[_sorted_tc_idx]
        return np.interp(_x_target_interp, _x_tc_sorted, y_known).tolist()

    # --- Fallback: compute on the fly (e.g. first call before pre-computation) ---
    try:
        x_known_values = tc_positions_series.astype(float).values
    except (AttributeError, ValueError) as e:
        print(f"Error: TC positions not convertible to float: {e}"); return []
    if len(y_raw) != len(x_known_values):
        print("Error: Temps/Positions length mismatch in interpolateTempProfile."); return []
    si = np.argsort(x_known_values)
    x_s = x_known_values[si]; y_s = np.array(y_raw)[si]
    if len(x_s) < 2:
        print(f"Warn: Only {len(x_s)} unique TC position(s) — cannot interpolate."); return []
    x_tgt = np.arange(int(x_s[0]), int(x_s[-1]) + 1)
    return np.interp(x_tgt, x_s, y_s).tolist()


In [13]:
def getPartVolProfile(data):
    APdata = data["Axial Position (mm)"]
    CSdata = data["Cross-Sectional Area (mm^2)"]
    TMdata = data["Transition Model"]
    
    areaProfile = []
    
    for i in range(len(APdata) - 1):
        # 1. Calculate the exact number of 0.1mm slices in this segment
        segment_length = APdata[i+1] - APdata[i]
        # Use round to handle float math like 0.700000000001
        num_slices = int(round(segment_length / 0.01))
        
        # 2. Create the x-array for the eval() functions
        # This creates exactly 'num_slices' points starting from the current position
        x = np.linspace(APdata[i], APdata[i+1], num_slices, endpoint=False)
        
        if TMdata[i] != 'Constant':
            try: 
                sectionAreaProfile = eval(TMdata[i])
                # Ensure the eval result matches our slice count
                if isinstance(sectionAreaProfile, np.ndarray):
                    if len(sectionAreaProfile) != num_slices:
                        # If eval returns the wrong size, force it to num_slices
                        sectionAreaProfile = np.resize(sectionAreaProfile, num_slices)
                else:
                    sectionAreaProfile = np.full(num_slices, sectionAreaProfile)
            except Exception as e: 
                print(f"Error eval TM '{TMdata[i]}': {e}")
                sectionAreaProfile = np.full(num_slices, CSdata[i])
        else: 
            # Constant area for the duration of this segment
            sectionAreaProfile = np.full(num_slices, CSdata[i])
            
        areaProfile.append(sectionAreaProfile)

    areaProfile = np.concatenate(areaProfile)
    
    # Each entry represents the volume of a 0.1mm thick slice
    volumeProfile = [0.01 * slice_area for slice_area in areaProfile]
    
    return volumeProfile

In [14]:
def safe_parse_profile(profile):
    if isinstance(profile, str): profile = profile.strip()
    if isinstance(profile, str) and (profile.endswith("'") or profile.endswith('"')): profile = profile[:-1]
    if isinstance(profile, str) and (profile.startswith("'") or profile.startswith('"')): profile = profile[1:]
    if isinstance(profile, str):
        try:
            parsed = ast.literal_eval(profile)
        except Exception as e:
            raise ValueError(f"Invalid profile string: {profile}") from e
        # Strip leading/trailing whitespace from all keys — guards against
        # accidental spaces in spreadsheet cells (e.g. ' Axial Position (mm)')
        if isinstance(parsed, dict):
            return {k.strip(): v for k, v in parsed.items()}
        return parsed
    # Already a dict (pre-parsed) — sanitise keys the same way
    if isinstance(profile, dict):
        return {k.strip(): v for k, v in profile.items()}
    return profile

In [15]:
import pandas as pd
import numpy as np


def _build_profile_sequence(relevantParts, planeTable):

    isReversed = False
    previousIsReversed = False
    cumulativeOffset = 0
    maxAxialPosition = 100000
    
    # Dictionary to hold Part Volume Series before building the final volumeDF
    part_series_dict = {} 

    # Initialize the output DataFrame structure
    colNames = ["Axial Position"]
    dummyarray = np.empty((maxAxialPosition,))
    dummyarray[:] = np.nan
    volumeDF = pd.DataFrame(dummyarray, columns=colNames)

    mate_positions = []
    placement_summaries = []

    if relevantParts.empty:
        return volumeDF, mate_positions, placement_summaries, "Error: No parts to process in sequence."

    # Initial part setup (Part 0)
    uid0 = str(relevantParts["UID"].iloc[0])

    mate_positions.append({
        "Part": f"Part_0_{uid0}",
        "Previous Mate Position": None,
        "Next Mate Position": None  # filled in at i=0
    })

    # --- Main Mating and Volume Calculation Loop ---
    for i in range(len(relevantParts) - 1):
        try:
            previousPart = safe_parse_profile(relevantParts["Profile"].iloc[i])
            nextPart = safe_parse_profile(relevantParts["Profile"].iloc[i + 1])
            
            if not isinstance(previousPart, dict) or not isinstance(nextPart, dict):
                raise ValueError(f"Parsed profile is not a dictionary at index {i} or {i+1}")
            
            previousPartPlanes = previousPart.get('Contact Plane')
            nextPartPlanes = nextPart.get('Contact Plane')
            if previousPartPlanes is None or nextPartPlanes is None:
                raise KeyError(f"'Contact Plane' missing in profile at index {i} or {i+1}")
                
            
            # Calculate the reversed version of the next part's planes
            nextPartPlanesReversed = [
                ''.join([c if c!='^' and c!='v' else '^' if c=='v' else 'v' for c in p]) 
                for p in reversed(nextPartPlanes)
            ]
            
            
            previousPartAxialPositions = previousPart.get('Axial Position (mm)')
            nextPartAxialPositions = nextPart.get('Axial Position (mm)')
            if previousPartAxialPositions is None or nextPartAxialPositions is None:
                raise KeyError(f"'Axial Position (mm)' missing in profile at index {i} or {i+1}")
            
            previousPartVolumeProfile = getPartVolProfile(previousPart)

            # --- Initial Part Placement ---
            if i == 0:
                colName0 = f"Part_0_{uid0}"
                first_part_slice_idx = list(range(len(previousPartVolumeProfile)))
                
                if first_part_slice_idx:
                    part_series_dict[colName0] = pd.Series(previousPartVolumeProfile, index=first_part_slice_idx)
                    cumulativeOffset = (len(previousPartVolumeProfile)) / 100.0
                else:
                    cumulativeOffset = 0.0

            # Reverse the previous part's data if it was reversed in the previous step
            if isReversed:
                previousPartPlanes = [
                    ''.join([c if c!='^' and c!='v' else '^' if c=='v' else 'v' for c in p]) 
                    for p in reversed(previousPartPlanes)
                ]
                previousPartAxialPositions.reverse()
                previousPartVolumeProfile.reverse()
                previousIsReversed = True
            else:
                previousIsReversed = False


            # --- Mating Logic (Finding Valid Mates) ---
            mateList = []
            for plane in previousPartPlanes:
                matches = planeTable.loc[planeTable['Plane'] == plane]
                if not matches.empty and 'Valid Mates' in matches.columns:
                    mates_str = matches['Valid Mates'].iloc[-1]
                    mateList.append(mates_str.split(', ') if pd.notna(mates_str) else [])
                else:
                    mateList.append([])
                    

            normalMates = []
            reversedMates = []
            
            # Loop backwards through previous part's planes (higher index first for 'top-most' logic)
            for j in reversed(range(len(mateList))):
                # Find normal orientation mates
                for k in range(len(nextPartPlanes)):
                    if nextPartPlanes[k] in mateList[j]:
                        normalMates.append([previousPartPlanes[j], j, nextPartPlanes[k], k])

                # Find reversed orientation mates
                for k in range(len(nextPartPlanesReversed)):
                    if nextPartPlanesReversed[k] in mateList[j]:
                        reversedMates.append([previousPartPlanes[j], j, nextPartPlanesReversed[k], k])

                    # Optimization for REST mates (only keep if no non-REST mate is found)
                    is_rest = lambda s: "REST" in str(s).upper()
                    nn = [c for c in normalMates if not is_rest(c[0])]
                    rr = [c for c in reversedMates if not is_rest(c[0])]
                    if nn or rr:
                        normalMates, reversedMates = nn, rr
            
            # --- Mate Selection Logic (Corrected Priority Structure) ---
            
            validMateList = None
            isReversed = False
            
            # --- 1. Priority 1: LID Mates (Highest Priority with Reversed Preference) ---
            
            # Find LID Mates (Logic remains the same)
            find_mate = lambda mates, names: next((m for m in mates if any(n in str(m[0]).upper() or n in str(m[2]).upper() for n in names)), None)
            lid_normal_mate = find_mate(normalMates, ['LID'])
            lid_reversed_mate = find_mate(reversedMates, ['LID'])
            
            # Reset validMateList here to ensure clean selection
            validMateList = None
            isReversed = False
            
            # A. Check for the CRITICAL Reversed LID mate first
            if lid_reversed_mate:
                # If a reversed LID mate exists, select it immediately as highest priority
                if not lid_normal_mate:
                    # Unique reversed LID mate
                    validMateList = lid_reversed_mate
                    isReversed = True
                    print(f"Reversed (Unique LID priority): {relevantParts['UID'].iloc[i+1]}")
                else:
                    # Both exist (Tie-breaker). Since the reversed mate is required, 
                    # we explicitly check the 'topmost' plane index (j) or prefer reverse.
                    jn_lid = lid_normal_mate[1]
                    jr_lid = lid_reversed_mate[1]
                    
                    # Explicitly prefer reverse if the reversed mate is equal or better 
                    # in terms of 'topmost' index (jr_lid <= jn_lid)
                    if jr_lid <= jn_lid:
                        validMateList = lid_reversed_mate
                        isReversed = True
                        print(f"Reversed (LID tie-breaker preference): {relevantParts['UID'].iloc[i+1]}")
                    else:
                        # Only use normal if reversed is demonstrably lower priority by index
                        validMateList = lid_normal_mate
                        isReversed = False
            
            # B. Fallback to Unique Normal LID mate if no reversed LID mate was selected
            elif lid_normal_mate:
                # Only a normal LID mate exists
                validMateList = lid_normal_mate
                isReversed = False
                
            # If validMateList is still None, the code proceeds to the 'if validMateList is None:' block below.

            # --- 2. Priority 2: Standard Multi-Mate Checks (Runs ONLY if no LID mate was found) ---
            
            # Only proceed if we haven't selected a mate yet AND we have multi-mates
            elif len(normalMates) > 0 and len(reversedMates) > 0:
                jn, jr = normalMates[0][1], reversedMates[0][1]

                # Priority 2a: Specific part UIDs
                # if any(x in relevantParts["UID"].iloc[i+1] for x in ["AC","GKT","GPA"]):
                #     validMateList = normalMates[0]
                #     isReversed = False
                # Priority 2b: Same mate plane index (prefer normal)
                if jr == jn:
                    validMateList = normalMates[0]
                    print(f"Normal and reversed for {relevantParts['UID'].iloc[i+1]} have same index; resorting to normal")
                    isReversed = False
                # Priority 2c: Honor topmost plane 
                elif jr != jn:
                    if jr > jn:
                        validMateList = reversedMates[0]
                        isReversed = True
                        print(f"Reversed to honor top-most plane: {relevantParts['UID'].iloc[i+1]}")
                    else:
                        validMateList = normalMates[0]
                        isReversed = False
                # Priority 2d: GASK/ORING planes 
                elif find_mate(normalMates, ['GASK','ORING']):
                    validMateList = normalMates[0]
                    isReversed = False
                    print(f"Normal mate for GASK/ORING mate: {relevantParts['UID'].iloc[i+1]}")
                elif find_mate(reversedMates, ['GASK','ORING']):
                    validMateList = reversedMates[0]
                    isReversed = True
                    print(f"Reversed (gask/oring): {relevantParts['UID'].iloc[i+1]}")
                # Default to normal match
                else:
                    validMateList = normalMates[0]
                    isReversed = False
                    print(f"Nothing else hit, resorting to first normal: {relevantParts['UID'].iloc[i+1]}")

            # --- 3. Priority 3: Simple Mate Cases (Runs ONLY if previous checks failed) ---
            
            # Case 3a: Only Normal Mates exist
            elif len(normalMates) > 0 and len(reversedMates) == 0:
                validMateList = normalMates[0]
                isReversed = False
                print(f"Only normal mates exist: {relevantParts['UID'].iloc[i+1]}")
            # Case 3b: Only Reversed Mates exist
            elif len(reversedMates) > 0 and len(normalMates) == 0:
                validMateList = reversedMates[0]
                isReversed = True
                print(f"Reversed: {relevantParts['UID'].iloc[i+1]}")

            # Final check before placement
            if validMateList is None:
                raise ValueError(f"No valid mates for {relevantParts['UID'].iloc[i+1]} to {relevantParts['UID'].iloc[i]}.")

            # --- Volume Profile Placement ---
            nextPartVolumeProfile = getPartVolProfile(nextPart)
            
            # Reverse next part's profile and axial positions if reversed mate was selected
            if isReversed:
                nextPartAxialPositions.reverse()
                x_max = max(nextPartAxialPositions)
                nextPartAxialPositions = [x_max - x for x in nextPartAxialPositions]
                nextPartVolumeProfile.reverse()

            j = validMateList[1]  # Index of the mating plane on the previous part
            k = validMateList[3]  # Index of the mating plane on the next part

            prev_part_end = max(previousPartAxialPositions)
            next_start = min(nextPartAxialPositions)

            # Calculate the distance of the previous mate plane from the previous part's end
            if previousIsReversed:
                d_prev = previousPartAxialPositions[j]
            else:
                d_prev = prev_part_end - previousPartAxialPositions[j]

            # Calculate the distance of the next mate plane from the next part's start
            d_next = nextPartAxialPositions[k]

            # Calculate the axial overlap/gap (part_Offset)
            if previousPartAxialPositions[j] == prev_part_end and nextPartAxialPositions[k] == next_start:
                part_Offset = 0.0
            else:
                part_Offset = d_prev + d_next

            part_length = (len(nextPartVolumeProfile)) / 100.0
            start_pos = cumulativeOffset - part_Offset
            end_pos = start_pos + part_length
            #next_mate = round(start_pos + nextPartAxialPositions[k], 3)

            # Insert next part's volume data into the part_series_dict
            next_part_uid = relevantParts['UID'].iloc[i + 1]
            colName = f"Part_{i + 1}_{next_part_uid}"
            
            slice_start_idx = int(round(100 * start_pos))
            effective_start_mm = slice_start_idx / 100.0
            next_mate = round(effective_start_mm + nextPartAxialPositions[k], 3)
            slice_end_idx = slice_start_idx + len(nextPartVolumeProfile)
            
            part_series_dict[colName] = pd.Series(nextPartVolumeProfile, index=range(slice_start_idx, slice_end_idx))

            # Update placement summaries and mate positions (includes reversal status)
            reversal_status = "Reversed" if isReversed else "Normal"
            uid_prev = relevantParts['UID'].iloc[i]
            summary = [
                (f"{uid_prev} mated to {next_part_uid} at {next_mate} mm ({reversal_status})"),
                (f"{validMateList[0]} → {validMateList[2]}"),
                (f"Calculated Length of {next_part_uid}: {part_length}")
            ]
            placement_summaries.append(summary)

            # Update previous part's next mate and append new part with correct interface position
            mate_positions[i]["Next Mate Position"] = next_mate
            mate_positions.append({
                "Part": f"Part_{i + 1}_{next_part_uid}",
                "Previous Mate Position": next_mate,
                "Next Mate Position": None  # filled in next iteration
            })

            cumulativeOffset = start_pos + part_length

        except (ValueError, KeyError, IndexError, TypeError) as e:
            error_msg = (
                f"Error processing part {i + 1} ({relevantParts['UID'].iloc[i + 1]} "
                f"mating with {relevantParts['UID'].iloc[i]}): {e}"
            )
            return pd.DataFrame(), [], [], error_msg

    # --- Final calculations and DataFrame Assembly ---
    
    # Determine max length for the final DataFrame
    max_idx = max(idx for series in part_series_dict.values() for idx in series.index) if part_series_dict else 0
    
    # Ensure the final DataFrame size is sufficient
    final_size = max(max_idx + 1, int(maxAxialPosition)) 

    indexList = [round(0.01 * i, 2) for i in range(final_size)]
    volumeDF = pd.DataFrame({'Axial Position': indexList})

    # Add all collected part series
    for colName, series in part_series_dict.items():
        volumeDF[colName] = series.reindex(volumeDF.index)

    volumeDF["Total Volume"] = volumeDF.filter(like='Part_').sum(axis=1)
    
    return volumeDF, mate_positions, placement_summaries, None

In [16]:
def _find_internal_alignment_position(relevantParts_df, safe_parse_profile, df_label="Unknown"):
    """
    Searches through all parts for 'MP-INTERNAL' planes.
    Reports findings to console for debugging.
    """
    print(f"\n--- Searching for MP-INTERNAL in {df_label} Sequence ---")
    
    for index, row in relevantParts_df.iterrows():
        uid = row['UID']
        try:
            profile_data = safe_parse_profile(row['Profile'])
            planes = profile_data.get('Contact Plane', [])
            positions = profile_data.get('Axial Position (mm)', [])
            
            # Debug: Log that we are checking this part
            # print(f"  Checking Part: {uid} (found {len(planes)} planes)")

            for plane_index, plane_name in enumerate(planes):
                # Ensure we have a string to check against
                p_name_str = str(plane_name).upper() if plane_name else ""
                
                if 'MP-INTERNAL' in p_name_str:
                    if plane_index < len(positions):
                        alignment_pos_mm = positions[plane_index]
                        
                        # --- THE DEBUG PRINT YOU REQUESTED ---
                        print(f"  [FOUND] '{plane_name}' on Part: {uid}")
                        print(f"  [INFO] Axial position: {alignment_pos_mm} mm (local to part)")
                        
                        return alignment_pos_mm
                    else:
                        print(f"  [ERROR] Found '{plane_name}' on {uid} but no axial position index {plane_index} exists.")
        
        except Exception as e:
            print(f"  [WARNING] Error parsing profile for part {uid}: {e}")
            continue
            
    print(f"  [RESULT] No 'MP-INTERNAL' plane found in {df_label} sequence.")
    return None

In [17]:
def getSystemVolumeProfile(partList, mating_table_path, fur_path, hps_path):
    global graphLabel
    
    # --- 1. File Reading and Part Profile Preparation ---
    # Authoritative sheet lists — only these contain valid UID/Profile geometry
    FUR_SHEETS = ['CRU', 'LID', 'PLU', 'VRE']
    HPS_SHEETS = ['AC', 'GKT', 'NZ', 'TU']

    def _load_sheets(path, sheet_names, label, material_col=None):
        """Load only the sheets that exist in the file; warn about any missing ones.
        If material_col is given, also returns a uid->material dict."""
        try:
            all_sheets = pd.ExcelFile(path).sheet_names
        except Exception as e:
            raise IOError(f'Cannot open {label}: {e}')
        available = [s for s in sheet_names if s in all_sheets]
        missing   = [s for s in sheet_names if s not in all_sheets]
        if missing:
            print(f'  Warning: {label} missing expected sheets: {missing} — skipped')
        if not available:
            raise ValueError(f'{label} contains none of the expected sheets: {sheet_names}')
        frames = []
        uid_mat = {}
        for s in available:
            df_s = pd.read_excel(path, sheet_name=s)
            if 'UID' in df_s.columns and 'Profile' in df_s.columns:
                frames.append(df_s[['UID', 'Profile']].dropna())
            else:
                print(f'  Warning: {label} sheet "{s}" missing UID/Profile columns — skipped')
            if material_col and material_col in df_s.columns and 'UID' in df_s.columns:
                for _, row in df_s[['UID', material_col]].dropna().iterrows():
                    uid_mat[str(row['UID'])] = str(row[material_col])
        if not frames:
            raise ValueError(f'{label}: no valid UID/Profile data found in sheets {available}')
        return pd.concat(frames, ignore_index=True), uid_mat

    try:
        planeTable   = pd.read_excel(mating_table_path)
        partsDF_Internal, uid_mat_fur = _load_sheets(fur_path, FUR_SHEETS, 'MLD-FUR', material_col='Chemicals')
        partsDF_External, uid_mat_hps = _load_sheets(hps_path, HPS_SHEETS, 'MLD-HPS', material_col='Material')
        uid_to_material = {**uid_mat_fur, **uid_mat_hps}
        _log_live(f'MLD-FUR: loaded {len(partsDF_Internal)} parts')
        _log_live(f'MLD-HPS: loaded {len(partsDF_External)} parts')

    except FileNotFoundError as e:
        print(f'Error: Could not find required geometry file: {e.filename}')
        graphLabel.configure(text=f'Error: File not found: {os.path.basename(e.filename)}')
        window.update_idletasks()
        return None
    except Exception as e:
        print(f'Error reading geometry files: {e}')
        graphLabel.configure(text=f'Error reading geometry files: {e}')
        window.update_idletasks()
        return None

    partsDF_All = pd.concat([partsDF_Internal, partsDF_External], ignore_index=True)



    # --- 2. Filter Input List to Create Master relevantParts List ---
    partList = partList.dropna()
    relevantParts = pd.DataFrame(columns=['UID', 'Profile'])

    for index, row in partList.iterrows():
        partProfile = partsDF_All.loc[partsDF_All['UID'] == row['UID']]
        if not partProfile.empty:
            relevantParts=pd.concat([relevantParts, partProfile], ignore_index=True)

    if relevantParts.empty:
        graphLabel.configure(text="Error: No relevant parts found based on Layout file.")
        return None

    # --- 3. Split Master List into Internal and External Sequences ---
    internal_uids = partsDF_Internal['UID'].tolist()
    
    relevantParts_Internal = relevantParts[relevantParts['UID'].isin(internal_uids)].reset_index(drop=True)
    relevantParts_External = relevantParts[~relevantParts['UID'].isin(internal_uids)].reset_index(drop=True)

    # --- 4. Generate Two Profiles using _build_profile_sequence ---

    _log_live("--- Building External Profile Sequence ---")
    external_volumeDF, external_mate_positions, external_placement_summaries, error_ext = \
        _build_profile_sequence(relevantParts_External, planeTable)
    
    if error_ext:
        error_msg = f"External Profile Error: {error_ext}"
        print(error_msg)
        graphLabel.configure(text=error_msg)
        window.update_idletasks()
        return None

    # Only attempt to build internal sequence if internal parts exist
    internal_volumeDF = None
    internal_mate_positions = []
    internal_placement_summaries = []
    if not relevantParts_Internal.empty:
        _log_live("--- Building Internal Profile Sequence ---")
        (internal_volumeDF, internal_mate_positions,
         internal_placement_summaries, error_int) = _build_profile_sequence(
            relevantParts_Internal, planeTable)
        if error_int:
            error_msg = f"Internal Profile Error: {error_int}"
            print(error_msg)
            graphLabel.configure(text=error_msg)
            window.update_idletasks()
            return None
    else:
        _log_live("--- No internal parts in layout, skipping internal profile sequence ---")

    # --- 5. Merge/Overlay Profiles based on MP-INTERNAL Alignment ---
    
    # Initialize variables for the final DataFrame
    volumeDF = None
    mate_positionsDF = pd.DataFrame()
    placement_summaries = []
    mate_positions = []
    
    if not relevantParts_Internal.empty:
        
        # Calculate alignment based on MP-INTERNAL plane
        external_alignment_pos = _find_internal_alignment_position(relevantParts_External, safe_parse_profile)
        internal_alignment_pos = _find_internal_alignment_position(relevantParts_Internal, safe_parse_profile)
        
        if external_alignment_pos is None or internal_alignment_pos is None:
            error_msg = "Error: Could not find 'MP-INTERNAL' plane in both external and internal part lists for alignment."
            print(error_msg)
            graphLabel.configure(text=error_msg)
            window.update_idletasks()
            return None
        
        # Calculate the required offset
        alignment_offset_mm = external_alignment_pos - internal_alignment_pos
        alignment_offset_slices = int(round(alignment_offset_mm * 100))
        
        _log_live(f"Alignment calculated: External pos={external_alignment_pos}mm, Internal pos={internal_alignment_pos}mm. Offset={alignment_offset_mm}mm.")

        # Determine the maximum length required for the final combined DataFrame
        max_idx_external = len(external_volumeDF)
        max_idx_internal = alignment_offset_slices + len(internal_volumeDF)
        max_len = max(max_idx_external, max_idx_internal)
        
        # Initialize final volumeDF
        volumeDF = pd.DataFrame(index=range(max_len))
        volumeDF["Axial Position"] = [round(0.01*i, 2) for i in range(max_len)]
        
        # Copy External Profile (starts at index 0)
        for col in external_volumeDF.columns:
            if col.startswith("Part_"):
                 volumeDF[f"External_{col}"] = external_volumeDF[col]
        
        # Copy Internal Profile (offset it by alignment_offset_slices)
        for col in internal_volumeDF.columns:
            if col.startswith("Part_"):
                 col_data = pd.Series(
                     internal_volumeDF[col].values, 
                     index=range(alignment_offset_slices, alignment_offset_slices + len(internal_volumeDF))
                 )
                 volumeDF[f"Internal_{col}"] = col_data
        
        # Fill NaN with 0 for summation and calculate Total Volume
        volumeDF = volumeDF.fillna(0)
        volumeDF["Total Volume"] = volumeDF.filter(like='Part_').sum(axis=1)
        
        # Combine summaries and adjust mate positions for plotting
        placement_summaries = external_placement_summaries + internal_placement_summaries
        
        # Adjust internal mate positions by the alignment offset
        adjusted_internal_mates = []
        for mate in internal_mate_positions:
            adjusted_mate = mate.copy()
            if adjusted_mate["Previous Mate Position"] is not None:
                adjusted_mate["Previous Mate Position"] = round(adjusted_mate["Previous Mate Position"] + alignment_offset_mm, 3)
            if adjusted_mate["Next Mate Position"] is not None:
                adjusted_mate["Next Mate Position"] = round(adjusted_mate["Next Mate Position"] + alignment_offset_mm, 3)
            adjusted_internal_mates.append(adjusted_mate)
            
        mate_positions = external_mate_positions + adjusted_internal_mates
        mate_positionsDF = pd.DataFrame(mate_positions).reset_index(drop=True)
        
    else:
        # If there are only external parts, use the external profile as the final result
        volumeDF = external_volumeDF
        mate_positionsDF = pd.DataFrame(external_mate_positions).reset_index(drop=True)
        placement_summaries = external_placement_summaries
        mate_positions = external_mate_positions


    # --- 6. Output and Plotting (Adjusted for combined/split profiles) ---

    print("\nPLACEMENT SUMMARY:")
    _log_live("-" * 60)
    # Ensure placement_summaries is a list of tuples (pos_str, mate_str) for printing
    for pos, mates_summary, length in placement_summaries:
        print(pos)
        print(mates_summary)
        print(length)
        _log_live("-" * 60)

    _log_live("\nMate Positions")
    _log_live(mate_positionsDF)

    _log_live("\nVolume per part (mL):")
    for col in volumeDF.columns:
        if col.startswith("External_Part_") or col.startswith("Internal_Part_"):
            v_mm3 = volumeDF[col].sum(skipna=True)
            _log_live(f"{col}: {v_mm3/1000:.3f} mL")


    print('\npartList:', partList)

    return volumeDF, uid_to_material

In [18]:
# --- processData function ---
def processData():
    global runData, baselineEdited, graphLabel, fileProgress, graphProgress, calc_step_var, output_dir, info_log, fit_csv_path, p_zero_start_var, p_zero_end_var, p_zero_manual_var
    global data_filename, temp_profile_file, layout_file, mating_table_file, fur_file, hps_file
    global z_temp_cutoff_var, z_offset_var, p_calc_start_time_var, interpolate_temps_var
    global _sorted_tc_idx, _x_tc_sorted, _x_target_interp
    global exportTempButton, tempViewButton, exportPlotButton, calculateFitButton, fit_results_label, texpansion_button
    global sigma_p_var, sigma_v_var, sigma_t_var # <-- NEW
    global gas_species_var

    info_log = []  # Reset for new run
    fit_csv_path = ""  # Reset — will be created after output_dir check
    _log = _log_live
    open_info_window(clear=True)  # Open/clear info window for new run
    if exportPlotButton: exportPlotButton.configure(state='disabled')
    if exportTempButton: exportTempButton.configure(state='disabled')
    if tempViewButton: tempViewButton.configure(state='disabled')
    if texpansion_button: texpansion_button.configure(state='disabled')
    if calculateFitButton: calculateFitButton.configure(state='disabled')
    if fit_results_label: fit_results_label.configure(text="")

    if not data_filename: graphLabel.configure(text="Error: Select Data File."); return
    use_interpolation = interpolate_temps_var.get()
    if not use_interpolation and not temp_profile_file: graphLabel.configure(text="Error: Select Temp Profile or check Interpolate."); return
    if not layout_file: graphLabel.configure(text="Error: Select Layout File."); return
    if not mating_table_file: graphLabel.configure(text="Error: Select Mating Table."); return
    if not fur_file: graphLabel.configure(text="Error: Select MLD-FUR File."); return
    if not hps_file: graphLabel.configure(text="Error: Select MLD-HPS File."); return
    if not output_dir:
        graphLabel.configure(text='Error: Set Output Folder before running.')
        return

    z_temp_cutoff_mm = None
    # Create the fit CSV path once for this run
    from datetime import datetime as _dt
    _ts_run = _dt.now().strftime('%y%m%d_%H%M%S')
    # Placeholder — will be finalised with run_id after getData
    _fit_csv_pending = True
    try: 
        z_cutoff_input = z_temp_cutoff_var.get();
        if z_cutoff_input: z_temp_cutoff_mm = int(float(z_cutoff_input)); print(f"Using Z-Temp Cutoff: {z_temp_cutoff_mm} mm")
    except ValueError: 
        graphLabel.configure(text="Error: Invalid Z-Temp Cutoff."); return

    z_offset_mm = 0
    try: 
        z_offset_input = z_offset_var.get();
        if z_offset_input: z_offset_mm = int(float(z_offset_input)); print(f"Applying Z-Offset: {z_offset_mm} mm")
    except ValueError: 
        graphLabel.configure(text="Error: Invalid Z-Offset."); return
    if z_offset_mm < 0:
        _log(f"Warning: Negative Z-Offset ({z_offset_mm} mm) not allowed — clamped to 0.")
        z_offset_mm = 0

    p_calc_start_hr = None
    try: 
        start_time_input = p_calc_start_time_var.get();
        if start_time_input: p_calc_start_hr = float(start_time_input); print(f"Using P_calc Start Time: {p_calc_start_hr} Hours")
        else: print("P_calc Start Time not provided.")
    except ValueError: 
        graphLabel.configure(text=f"Error: Invalid P_calc Start Time '{start_time_input}'."); return

    # --- NEW: Get Uncertainty Inputs ---
    try:
        sigma_p_percent = float(sigma_p_var.get()) / 100.0 # Convert from % to fraction
        sigma_v_percent = float(sigma_v_var.get()) / 100.0 # Convert from % to fraction
        sigma_t_base_K = float(sigma_t_var.get()) # Absolute K


        _log(f"Uncertainty Assumptions: P={sigma_p_percent*100}%, V={sigma_v_percent*100}%, T={sigma_t_base_K} K")
    except ValueError:
        graphLabel.configure(text="Error: Invalid uncertainty values. Must be numbers."); return
    # --- END NEW ---

    # --- Calculation Step Interval ---
    calc_step = 1  # default: compute every row
    try:
        step_input = calc_step_var.get().strip()
        if step_input:
            calc_step = max(1, int(float(step_input)))
        _log(f'Calculation step interval: every {calc_step} rows')
    except (ValueError, AttributeError):
        graphLabel.configure(text='Error: Invalid calc step interval.'); return



    fileProgress['value'] = 0; window.update_idletasks()

    try: df = pd.read_csv(data_filename, low_memory=False)
    except Exception as e: graphLabel.configure(text=f"Error reading Data: {e}"); return
    fileProgress['value'] = 10; window.update_idletasks()

    if df.shape[0] <= 12 or df.shape[1] <= 20: graphLabel.configure(text="Error: Data format unexpected."); return
    runID = df.iloc[12,10]; EXP_ID = df.iloc[6, 20]
    global run_id; run_id = str(runID).strip()
    if output_dir:
        from datetime import datetime as _dt2
        fit_csv_path = os.path.join(
            output_dir,
            f'{run_id}_{_dt2.now().strftime("%y%m%d_%H%M%S")}_FitResults.csv'
        )
        _log_live(f'Fit CSV: {os.path.basename(fit_csv_path)}')
    _log(f"Run ID: {runID}, EXP ID: {EXP_ID}")

    runDF_filtered, TC_df, TC_positions = getData(df.copy(), EXP_ID)
    if runDF_filtered is None: return

    print("Resetting runDF index..."); runDF = runDF_filtered.reset_index(drop=True); print("Index reset.")
    fileProgress['value'] = 20; window.update_idletasks()

    # Pressure Processing — column chosen in UI dropdown (populated from CSV headers)
    pressure_col = None; P_MPa = []; P_MPa_series = None
    _sel = pressure_col_var.get().strip() if pressure_col_var else ''
    if _sel and _sel not in ('', '(load data first)', '(none found)'):
        if _sel in df.columns:
            pressure_col = _sel
        else:
            graphLabel.configure(
                text=f'Error: Selected pressure column "{_sel}" not in data file.')
            return
    else:
        # Fallback: auto-detect first column whose name contains 'PT'
        for _c in df.columns:
            if 'PT' in _c.upper():
                pressure_col = _c
                _log(f'Auto-selected pressure column: {pressure_col}')
                break
    if pressure_col is None:
        graphLabel.configure(text='Error: No pressure (PT) column found — select one manually.')
        return
    try:
        _log(f"Using pressure column: {pressure_col}")
        P_MPa_series = df.loc[runDF_filtered.index, pressure_col].astype(float)
        if P_MPa_series.isna().all(): graphLabel.configure(text=f"Error: All pressure data is invalid."); return
        P_MPa = P_MPa_series.tolist()

    except Exception as e: graphLabel.configure(text=f"Error processing pressure: {e}"); return

    # ── Pressure zero offset ─────────────────────────────────────────────
    p_zero_offset = 0.0
    _zero_method = 'none'
    try:
        manual_in = p_zero_manual_var.get().strip() if p_zero_manual_var else ''
        zs_in     = p_zero_start_var.get().strip()  if p_zero_start_var  else ''
        ze_in     = p_zero_end_var.get().strip()    if p_zero_end_var    else ''

        if manual_in:
            # Manual override takes priority
            p_zero_offset = float(manual_in)
            _zero_method  = f'manual ({p_zero_offset:.6f} MPa)'

        elif zs_in and ze_in:
            # Auto: mean of P_meas over the vacuum window
            zs_hr = float(zs_in); ze_hr = float(ze_in)
            if zs_hr >= ze_hr:
                graphLabel.configure(text='Error: P zero window start must be before end.')
                return
            # Time series aligned with P_MPa (same index as runDF_filtered)
            time_series_tmp = df['Relative.2'].loc[runDF_filtered.index].astype(float) / 60.0
            mask_window = (time_series_tmp >= zs_hr) & (time_series_tmp <= ze_hr)
            p_window = P_MPa_series[mask_window].dropna()
            if len(p_window) < 2:
                graphLabel.configure(text='Warning: P zero window has < 2 points — offset not applied.')
            else:
                p_zero_offset = float(p_window.mean())
                p_zero_sigma  = float(p_window.std())
                _zero_method  = (
                    f'auto-vacuum window {zs_hr:.3f}–{ze_hr:.3f} hr '
                    f'(n={len(p_window)}, mean={p_zero_offset:.6f} MPa, '
                    f'sigma={p_zero_sigma:.4e} MPa)'
                )

        if p_zero_offset != 0.0:
            P_MPa = [v - p_zero_offset if pd.notna(v) else v for v in P_MPa]
            _log_live(f'Pressure zero offset applied: {_zero_method}')
        else:
            _log_live('Pressure zero offset: none applied')

    except Exception as _ze:
        _log_live(f'Warning: P zero offset error — {_ze}. Continuing without offset.')
    # ── End pressure zero offset ─────────────────────────────────────────

    _log(f"Pressure points: {len(P_MPa)}"); P_Pa=[float(p)*1e6 for p in P_MPa if pd.notna(p)]; fileProgress['value'] = 30; window.update_idletasks()

    # Time Processing (Aligned with runDF)
    if 'Relative.2' not in df.columns: graphLabel.configure(text="Error: Time column 'Relative.2' missing."); return
    Time_Minutes = []; Time_Seconds = pd.Series([], dtype=float); Time_Hours = pd.Series([], dtype=float)
    try:
        time_series_full = df['Relative.2']
        Time_Minutes_series = time_series_full.loc[runDF_filtered.index]
        if len(Time_Minutes_series) != len(runDF): graphLabel.configure(text="Error: Time/runDF length mismatch."); return
        Time_Minutes = Time_Minutes_series.tolist()
        Time_Seconds= pd.Series([float(min)*60 for min in Time_Minutes], index=runDF.index)
        Time_Hours= pd.Series([float(min)/60 for min in Time_Minutes], index=runDF.index)
        _log(f"Time points: {len(Time_Minutes)}")
    except Exception as e: graphLabel.configure(text=f"Error processing time: {e}"); return
    if not Time_Minutes: graphLabel.configure(text="Error: No time data after alignment."); return
    fileProgress['value'] = 40; window.update_idletasks()


    # Volume Profile Processing
    try: partList=pd.read_excel(layout_file)
    except Exception as e: graphLabel.configure(text=f"Error reading Layout: {e}"); return
    volumeDF, uid_to_material = getSystemVolumeProfile(partList, mating_table_file, fur_file, hps_file)
    if volumeDF is None: return

    # Material densities (g/cm³) keyed by material string from MLD files
    DENSITY_MAP = {
        "TZM B387 Type 364":      10.22,
        "304SS":                   7.93,
        "Molybdenum":             10.28,
        "Zr":                      6.52,
        "SS 316":                  8.00,
        "TZM B387-19, Type 364":  10.22,
    }

    # --- Capture volume before any correction ---
    _V_before_mL = volumeDF['Total Volume'].sum() / 1000.0

    # --- Furniture mass-based volume correction (from table) ---
    _any_fur = False
    for _uid, _mass_sv in fur_mass_entries.items():
        _mstr = _mass_sv.get().strip()
        if not _mstr: continue
        try: _mass_g = float(_mstr)
        except ValueError: _log(f'Warning: invalid mass for {_uid}: "{_mstr}" — skipped'); continue
        _is_int     = fur_internal_checks.get(_uid, tk.BooleanVar(value=False)).get()
        _col_prefix = 'Internal_' if _is_int else 'External_'
        _mat = uid_to_material.get(_uid)
        if not _mat: _log(f'Warning: no material for UID "{_uid}" — skipped'); continue
        _dens = DENSITY_MAP.get(_mat)
        if _dens is None: _log(f'Warning: no density for "{_mat}" (UID {_uid}) — add to DENSITY_MAP'); continue
        _V_mass_mm3 = (_mass_g / _dens) * 1000.0
        _found = False
        for _col in volumeDF.columns:
            if _col.startswith(_col_prefix) and _col.endswith(f'_{_uid}'):
                _V_cad = float(volumeDF[_col].sum())
                if abs(_V_cad) < 1e-9: _log(f'Warning: CAD vol ~0 for {_uid}'); _found = True; break
                _scale = _V_mass_mm3 / abs(_V_cad)
                volumeDF[_col] = volumeDF[_col] * _scale
                _V_corr = (_V_cad * _scale) / 1000.0  # preserves sign
                _delta  = _V_corr - _V_cad / 1000.0
                _log(f'Furniture: {_uid} ({_mat}, {"int" if _is_int else "ext"}): '
                     f'CAD={_V_cad/1000:.4f} mL → mass={_V_corr:.4f} mL '
                     f'(Δ={_delta:+.5f} mL, scale={_scale:.5f})')
                _found = True; _any_fur = True; break
        if not _found:
            _log(f'Warning: no {_col_prefix}Part_ col for UID "{_uid}" — check MLD file')
    if _any_fur:
        volumeDF['Total Volume'] = volumeDF.filter(like='Part_').fillna(0).sum(axis=1)
        _log('Total Volume recomputed after furniture corrections.')
    _V_after_fur_mL = volumeDF['Total Volume'].sum() / 1000.0

    # --- Chemical species volume (placed uniformly within CRU) ---
    # Densities (g/cm³) from literature / crystallographic data
    CHEMICAL_PROPS = {
        'Li3BN2':  1.85,   # lithium boron nitride (β-phase)
        'Li3N':    1.27,   # lithium nitride
        'Li3AlN2': 2.31,   # lithium aluminum nitride
        'Li3GaN2': 4.10,   # lithium gallium nitride
        'GaN':     6.15,   # gallium nitride (wurtzite)
    }
    _V_after_chem_mL = _V_after_fur_mL
    _chem_sp   = chem_species_var.get() if chem_species_var else 'None'
    _chem_ms   = chem_mass_var.get().strip() if chem_mass_var else ''
    if _chem_sp not in ('None', '') and _chem_ms:
        try:
            _chem_g = float(_chem_ms)
            _chem_dens = (float(chem_density_var.get().strip()) if chem_density_var and chem_density_var.get().strip() else None)\
                         if _chem_sp == 'Custom' else CHEMICAL_PROPS.get(_chem_sp)
            if _chem_dens and _chem_g > 0:
                _V_chem_mm3 = (_chem_g / _chem_dens) * 1000.0
                _cru_col = next((c for c in volumeDF.columns
                                 if c.startswith('Internal_') and 'CRU' in c.upper()), None)
                if _cru_col:
                    _nz = volumeDF[_cru_col][volumeDF[_cru_col] != 0]
                    if len(_nz) > 0:
                        _cru_s = int(_nz.index[0]); _cru_e = int(_nz.index[-1])
                        _n_sl  = _cru_e - _cru_s + 1
                        _vps   = -_V_chem_mm3 / _n_sl
                        volumeDF[f'Chem_{_chem_sp}'] = pd.Series([_vps]*_n_sl, index=range(_cru_s, _cru_e+1))
                        _chem_extra = volumeDF.filter(regex='^Chem_').fillna(0).sum(axis=1)
                        volumeDF['Total Volume'] = volumeDF.filter(like='Part_').fillna(0).sum(axis=1) + _chem_extra
                        _V_after_chem_mL = volumeDF['Total Volume'].sum() / 1000.0
                        _log(f'Chemical {_chem_sp}: {_chem_g:.3f} g, ρ={_chem_dens} g/cm³, '
                             f'V_chem={_V_chem_mm3/1000:.4f} mL (CRU slices {_cru_s}–{_cru_e})')
                    else: _log('Warning: CRU column all-zero — chemical not placed.')
                else: _log('Warning: no CRU Internal column found — chemical not placed.')
            else: _log(f'Warning: density not found for "{_chem_sp}" or mass≤0 — skipped.')
        except (ValueError, TypeError) as _ce:
            _log(f'Warning: chemical volume error: {_ce}')

    # --- Volume summary ---
    _vol_str = f'CAD vol: {_V_before_mL:.3f} mL'
    if _any_fur:
        _vol_str += f'  →  fur-corrected: {_V_after_fur_mL:.3f} mL  (Δ{_V_after_fur_mL-_V_before_mL:+.4f} mL)'
    if _chem_sp not in ('None','') and _chem_ms:
        _vol_str += f'  →  +chem: {_V_after_chem_mL:.3f} mL  (Δ{_V_after_chem_mL-_V_after_fur_mL:+.4f} mL)'
    _log(_vol_str)
    if vol_info_label: vol_info_label.configure(text=_vol_str)

    if 'Total Volume' not in volumeDF or volumeDF['Total Volume'].isna().all():
        graphLabel.configure(text='Error: Total Volume calc failed.'); return
    coarseVol_series = volumeDF['Total Volume']; remainder = len(coarseVol_series) % 100
    if remainder != 0: padding = pd.Series([0]*(100-remainder), index=range(len(coarseVol_series), len(coarseVol_series)+100-remainder)); coarseVol_series=pd.concat([coarseVol_series,padding])
    coarseVol = []; volumeProfile_padded = coarseVol_series.tolist()
    for i in range(int(len(volumeProfile_padded) / 100)): coarseVol.append(sum(volumeProfile_padded[100*i : 100*(i+1)]))
    if not coarseVol: graphLabel.configure(text="Error: Coarse volume calc failed."); return
    coarseVol = [vol / 1e9 for vol in coarseVol]; coarseVol_mL = [v*1e6 for v in coarseVol]
    V_total_m3 = sum(coarseVol)
    
    # Plot Coarse Volume Profile ---
    try:
        # Create an axial position array in mm (1mm steps)
        axial_coords_mm = np.arange(len(coarseVol))
        coarseVol_plot = [v * 1e9 for v in coarseVol]

        fig_vol = go.Figure()
        fig_vol.add_trace(go.Scatter(
            x=axial_coords_mm, 
            y=coarseVol_plot, 
            mode='lines',
            name='Coarse Volume',
            line=dict(color='darkgreen', width=2),
            fill='tozeroy'
        ))

        fig_vol.update_layout(
            title="System Volume Profile (1mm Resolution)",
            xaxis_title="Axial Position (mm)",
            yaxis_title="Volume (mm^3) per 1mm Slice",
            template='plotly_white',
            hovermode='x unified',
            width=1300,
            height=600
        )
        fig_vol.show()

    except Exception as e:
        print(f"Error plotting coarse volume: {e}")
    
    
    _log(f"\nTotal Vol (mL): {volumeDF['Total Volume'].sum()/1000:.3f} (DF), {sum(coarseVol_mL):.3f} (Coarse), Total Vol (m^3): {V_total_m3:.3e}")
    
    
    
    # --- Material property maps ---
    # Thermal expansion coefficients (m/m/K) keyed by material string from MLD files
    CTE_MAP = {
        "TZM B387 Type 364":     6.1e-6,
        "304SS":                 18e-6,
        "Molybdenum":            5.5e-6,
        "Zr":                    7e-6,
        "SS 316":                18e-6,
        "TZM B387-19, Type 364": 6.1e-6, 
    }

    T_ROOM_K = 295.15  # 22°C reference (temperature at which part profiles were measured)
    cte_sensitivity = np.zeros(len(coarseVol))
    for col in volumeDF.columns:
        if not (col.startswith("External_Part_") or col.startswith("Internal_Part_")):
            continue
        parts_split = col.split('_', 3)
        if len(parts_split) < 4:
            continue
        uid = parts_split[3]
        material = uid_to_material.get(uid)
        if not material:
            continue
        cte = CTE_MAP.get(material)
        if cte is None:
            continue
        part_vals = volumeDF[col].fillna(0).values
        remainder = len(part_vals) % 100
        if remainder:
            part_vals = np.concatenate([part_vals, np.zeros(100 - remainder)])
        part_coarse_m3 = part_vals.reshape(-1, 100).sum(axis=1) / 1e9  # mm³ -> m³
        n_contrib = min(500, len(part_coarse_m3), len(cte_sensitivity))
        cte_sensitivity[:n_contrib] += cte * part_coarse_m3[:n_contrib]
    print(f"Thermal expansion sensitivity built: {np.count_nonzero(cte_sensitivity)} nonzero 1mm positions in 0-500mm range")
    
    fileProgress['value'] = 50; window.update_idletasks()

    # Temperature Profile Setup (Conditional)
    lookupTable = None; central_tc_col = None; z_min_cal = None; z_max_cal = None
    if not use_interpolation:
        _log("Using Temperature Profile CSV...")
        try: lookupTable = pd.read_csv(temp_profile_file, index_col=0)
        except Exception as e: graphLabel.configure(text=f"Error reading Temp Profile: {e}"); return
        try: lookupTable.columns = [str(int(float(c))) for c in lookupTable.columns]
        except ValueError: graphLabel.configure(text="Error: Temp Profile columns not numbers."); return
        try:
            if len(lookupTable.columns) < 1: raise IndexError("Temp Profile has no position columns.")
            z_min_cal = int(lookupTable.columns[0]); z_max_cal = int(lookupTable.columns[-1])
            _log(f"Temp profile range: {z_min_cal}mm to {z_max_cal}mm")
        except (IndexError, ValueError) as e: graphLabel.configure(text=f"Error reading Temp Profile range: {e}"); return
        central_tc_row = TC_df[TC_df['Shorthand Tag'].str.contains('157', na=False)]
        if central_tc_row.empty: graphLabel.configure(text="Error: No TC with '157' in tag found."); return
        central_tc_col = central_tc_row.iloc[0]['NI LV Channel']
        _log(f"Central TC: {central_tc_row.iloc[0]['Shorthand Tag']} → channel {central_tc_col}")
    else:
        _log("Using Linear Interpolation for Temperature Profile...")
        if TC_positions.empty: graphLabel.configure(text="Error: No TC positions for interpolation."); return
        z_min_cal = int(TC_positions.min()); z_max_cal = int(TC_positions.max())
        _log(f"Interpolation range: {z_min_cal}mm to {z_max_cal}mm")
    fileProgress['value'] = 60; window.update_idletasks()

    # Main Calculation Loop
    graphProgress['value'] = 0; window.update_idletasks()
    n=[]; avgTempList = []; full_temp_profiles_K = []; n_error_list = [] # <-- NEW: Init error list
    R = 8.3145

    # Load Z (compressibility factor) lookup table for selected gas
    from scipy.interpolate import RegularGridInterpolator as _RGI
    _Z_TABLE_DIR = CANON_MLD_DIR
    _gas_file_map = {'Nitrogen': 'N2_Z.npz'}
    _gas_name = gas_species_var.get() if gas_species_var else 'Nitrogen'
    _gas_stem = _gas_file_map.get(_gas_name, 'N2_Z.npz')
    _z_interp = None; _z_T_min = 250.0; _z_T_max = 2000.0; _z_P_min = 0.1; _z_P_max = 50.0
    try:
        _zd = np.load(os.path.join(_Z_TABLE_DIR, _gas_stem))
        _z_interp = _RGI((_zd['T_K'], _zd['P_MPa']), _zd['Z'], bounds_error=False, fill_value=None)
        _z_T_min, _z_T_max = float(_zd['T_K'].min()), float(_zd['T_K'].max())
        _z_P_min, _z_P_max = float(_zd['P_MPa'].min()), float(_zd['P_MPa'].max())
        _log(f"Z table loaded: {_gas_name}")
    except Exception as _ze:
        _log(f"Warning: Z table not found ({_ze}). Using ideal gas Z=1.")

    required_tc_channels = TC_df['NI LV Channel'].tolist()
    runTempsDF = runDF[required_tc_channels]
    if runTempsDF.empty: graphLabel.configure(text="Error: No temperature data."); return

    # --- Pre-compute interpolation arrays (used by interpolateTempProfile) ---
    if use_interpolation:
        _sorted_tc_idx   = np.argsort(TC_positions.values.astype(float))
        _x_tc_sorted     = TC_positions.values.astype(float)[_sorted_tc_idx]
        _x_target_interp = np.arange(int(_x_tc_sorted[0]), int(_x_tc_sorted[-1]) + 1)
        _log(f'Interp pre-computation: {len(_x_tc_sorted)} TCs, '
             f'range {int(_x_tc_sorted[0])}–{int(_x_tc_sorted[-1])} mm '
             f'({len(_x_target_interp)} target points)')
    else:
        _sorted_tc_idx = _x_tc_sorted = _x_target_interp = None

    _log("Starting main calculation loop...")
    # Compute at every calc_step-th row; interp fills the rest
    calc_indices = list(range(0, len(runDF), calc_step))
    # Always include the last index so the full time range is covered
    if calc_indices[-1] != len(runDF) - 1:
        calc_indices.append(len(runDF) - 1)
    all_indices = list(range(len(runDF)))

    n_sparse = [];  avg_sparse = []; err_sparse = []; prof_sparse = []; t_sparse = []; dv_sparse = []
    sample_T_sparse = []   # temperature (degC) at user-specified sample z-position

    _sample_z_mm = None
    if sample_pos_var:
        try:
            _sz = sample_pos_var.get().strip()
            _sample_z_mm = int(float(_sz)) if _sz else None
        except (ValueError, AttributeError):
            _sample_z_mm = None
    _profile_start_z = z_min_cal if z_min_cal is not None else 0

    for i in calc_indices:
        t_sparse.append(i)
        try:
            temps = runTempsDF.iloc[i].tolist() # Already float
            if use_interpolation:
                tempProfile_raw = interpolateTempProfile(temps, TC_positions)
            else:
                central_val = float(runTempsDF[central_tc_col].iloc[i])
                if pd.isna(central_val): print(f"Warn: Central TC NaN at step {i}."); avg_sparse.append(np.nan); n_sparse.append(np.nan); prof_sparse.append([]); err_sparse.append(np.nan); dv_sparse.append(np.nan); sample_T_sparse.append(np.nan); continue
                row_idx = min(lookupTable.index.searchsorted(central_val), len(lookupTable) - 1)
                tempProfile_raw = lookupTable.iloc[row_idx].tolist()
            if not tempProfile_raw: print(f"Warn: Temp profile empty step {i}."); avg_sparse.append(np.nan); n_sparse.append(np.nan); prof_sparse.append([]); err_sparse.append(np.nan); dv_sparse.append(np.nan); sample_T_sparse.append(np.nan); continue # Append NaN to all
            if len(tempProfile_raw) > 0: avg_sparse.append(sum(tempProfile_raw)/len(tempProfile_raw))
            else: avg_sparse.append(np.nan)
            if _sample_z_mm is not None and tempProfile_raw:
                _si = max(0, min(_sample_z_mm - _profile_start_z, len(tempProfile_raw) - 1))
                sample_T_sparse.append(float(tempProfile_raw[_si]))
            else:
                sample_T_sparse.append(np.nan)
            tempProfile = [temp + 273.15 for temp in tempProfile_raw] # To Kelvin

            ambient_val = runDF["CV1-TC-Ambient"].iloc[i] # Already float/NaN
            ambient_temp = ambient_val + 273.15 if pd.notna(ambient_val) else 298.15

            tempProfile_cutoff = tempProfile[:]
            if z_temp_cutoff_mm is not None:
                profile_start_z = 0 if not use_interpolation else z_min_cal
                # z_temp_cutoff_mm is in V-coordinates.  When z_offset_mm > 0,
                # T is shifted forward by z_offset_mm relative to V, so the
                # cutoff in the T-array is at (z_temp_cutoff_mm + z_offset_mm).
                # For z_offset_mm <= 0 it is V that is trimmed, so no adjustment
                # to the T-array cutoff is needed.
                _zoff_corr = z_offset_mm if z_offset_mm > 0 else 0
                cutoff_index = z_temp_cutoff_mm + _zoff_corr - profile_start_z
                if cutoff_index < 0: cutoff_index = 0
                if cutoff_index < len(tempProfile_cutoff): tempProfile_cutoff = tempProfile_cutoff[:cutoff_index]
                elif cutoff_index > len(tempProfile_cutoff):
                    if tempProfile_cutoff: fill_needed = cutoff_index - len(tempProfile_cutoff); tempProfile_cutoff.extend([tempProfile_cutoff[-1]] * fill_needed)

            if not tempProfile_cutoff: last_cal_temp = ambient_temp
            else: last_cal_temp = float(tempProfile_cutoff[-1])
            temp_gradient = np.linspace(last_cal_temp, ambient_temp, 50); tempProfile_cutoff.extend(temp_gradient)
            _target_len = len(coarseVol) + z_offset_mm
            if _target_len > len(tempProfile_cutoff): tempProfile_cutoff += [ambient_temp] * (_target_len - len(tempProfile_cutoff))
            prof_sparse.append(tempProfile_cutoff[:])

            V_aligned = coarseVol
            T_aligned = tempProfile_cutoff[z_offset_mm:z_offset_mm + len(coarseVol)]
            if len(T_aligned) == 0: n_sparse.append(np.nan); err_sparse.append(np.nan); dv_sparse.append(np.nan); sample_T_sparse.append(np.nan); continue

            V_base = np.array(V_aligned); T_arr = np.array(T_aligned)
            if apply_smoothing_var and apply_smoothing_var.get() and len(T_arr) > 4:
                from scipy.signal import savgol_filter as _sgf
                _win = min(201, len(T_arr))
                if _win % 2 == 0: _win -= 1
                if _win >= 5:
                    T_arr = _sgf(T_arr, window_length=_win, polyorder=2)
            # Apply thermal expansion for axial positions 0-500mm
            _n_vol = len(coarseVol)
            cte_end = min(_n_vol, len(cte_sensitivity))
            cte_slice = cte_sensitivity[:cte_end]
            if len(cte_slice) < _n_vol:
                cte_slice = np.concatenate([cte_slice, np.zeros(_n_vol - len(cte_slice))])
            n_expand = min(500, _n_vol)
            V_arr = V_base.copy()
            V_arr[:n_expand] += 3.0 * cte_slice[:n_expand] * (T_arr[:n_expand] - T_ROOM_K)
            if i >= len(P_Pa): n_sparse.append(np.nan); err_sparse.append(np.nan); dv_sparse.append(np.nan); continue
            P = P_Pa[i]
            if np.any(T_arr <= 0): T_arr[T_arr <= 0] = 1e-6
            if np.isnan(P) or np.isnan(V_arr).any() or np.isnan(T_arr).any(): n_sparse.append(np.nan); err_sparse.append(np.nan); dv_sparse.append(np.nan); continue
            
            # --- Mole Calculation ---
            if _z_interp is not None:
                _P_clamp = np.clip(P / 1e6, _z_P_min, _z_P_max)
                _T_clamp = np.clip(T_arr, _z_T_min, _z_T_max)
                Z_arr = _z_interp(np.column_stack([_T_clamp, np.full(len(T_arr), _P_clamp)]))
                Z_arr = np.clip(Z_arr, 0.5, 2.0)
            else:
                Z_arr = np.ones(len(T_arr))
            nList = (P * V_arr) / (Z_arr * R * T_arr)
            n_total_step = np.sum(nList)
            n_sparse.append(n_total_step * 1000.0)
            
            # --- NEW: Uncertainty Calculation ---
            sigma_P = P * sigma_p_percent
            sigma_V_arr = V_arr * sigma_v_percent
            # Simple T uncertainty model: constant value
            # (A better model would increase error away from TCs if interpolating)
            sigma_T_arr = np.full_like(T_arr, sigma_t_base_K) 
            
            term_P_sq = ((n_total_step / P)**2) * (sigma_P**2)
            term_V_sq_sum = np.sum( (P / (Z_arr * R * T_arr))**2 * (sigma_V_arr**2) )
            term_T_sq_sum = np.sum( (P * V_arr / (Z_arr * R * T_arr**2))**2 * (sigma_T_arr**2) )
            
            total_variance = term_P_sq + term_V_sq_sum + term_T_sq_sum
            sigma_n = np.sqrt(total_variance)
            err_sparse.append(sigma_n * 1000.0)
            dv_sparse.append((np.sum(V_arr) - np.sum(V_base)) / V_total_m3 * 100.0)  # % of base volume
            # --- END NEW ---

            if i%1000 == 0:
                #print(f"Step {i}, Time: {Time_Hours.iloc[i]:.2f} hr, n: {n_total_step:.3e} ± {sigma_n:.2e} mol")
                graphProgress['value'] = int(100*i/len(runDF)); window.update_idletasks()
        except Exception as e: print(f"Error loop step {i}: {e}"); graphLabel.configure(text=f"Error step {i}: {e}"); n_sparse.append(np.nan); avg_sparse.append(np.nan); prof_sparse.append([]); err_sparse.append(np.nan); dv_sparse.append(np.nan); sample_T_sparse.append(np.nan)
    _log("Finished main calculation loop.")

    # ── Interpolate sparse results back to full resolution ──────────────
    _log('Interpolating sparse results to full resolution...')
    if len(t_sparse) >= 2:
        t_sp = np.array(t_sparse, dtype=float)
        n_sp  = np.array(n_sparse,   dtype=float)
        err_sp = np.array(err_sparse, dtype=float)
        avg_sp = np.array(avg_sparse, dtype=float)
        all_t  = np.array(all_indices, dtype=float)
        n           = list(np.interp(all_t, t_sp, n_sp))
        n_error_list = list(np.interp(all_t, t_sp, err_sp))
        avgTempList  = list(np.interp(all_t, t_sp, avg_sp))
        sample_temp_list = list(np.interp(all_t, t_sp, np.array(sample_T_sparse, dtype=float)))
        dv_sp = np.array(dv_sparse, dtype=float)
        delta_V_interp = list(np.interp(all_t, t_sp, dv_sp))
        # For full_temp_profiles_K: repeat nearest computed profile for gaps
        prof_map = {idx: prof for idx, prof in zip(calc_indices, prof_sparse)}
        full_temp_profiles_K = []
        last_prof = []
        for idx in all_indices:
            if idx in prof_map:
                last_prof = prof_map[idx]
            full_temp_profiles_K.append(last_prof)
    else:
        # Only one point computed — fill everything with it
        n            = n_sparse * len(all_indices)
        n_error_list = err_sparse * len(all_indices)
        avgTempList  = avg_sparse * len(all_indices)
        sample_temp_list = sample_T_sparse * len(all_indices)
        delta_V_interp = dv_sparse * len(all_indices)
        full_temp_profiles_K = [prof_sparse[0] if prof_sparse else []] * len(all_indices)
    _log(f'Interpolation complete. Total points: {len(n)}')

    

    # P_calc deferred — run via 'Calculate P_calc' button after plot
    P_calc_MPa_list = [np.nan] * len(runDF)

    # Show Moles Plot
    try: n_array=np.array(n); time_h_array=np.array(Time_Hours); mask_n_plot=~np.isnan(n_array); fig_n=go.Figure(); fig_n.add_trace(go.Scatter(x=time_h_array[mask_n_plot], y=n_array[mask_n_plot], name='Moles')); fig_n.update_layout(title="Moles vs Time", xaxis_title="Time (Hr)"); fig_n.show()
    except Exception as e: print(f"Plotly plot error: {e}")

    # Prepare Final Data
    rawData, baseline, baselineRange, baselineUnit, movingAvg, movingAvgRange, movingAvgUnit, tempRange, pressureRange, timeRange, nRange, dn_dtRange = (None,)*12
    final_expected_len = len(runDF)
    n_final = n + [np.nan] * (final_expected_len - len(n)); n_error_final = n_error_list + [np.nan] * (final_expected_len - len(n_error_list))
    delta_V_final = delta_V_interp + [np.nan] * (final_expected_len - len(delta_V_interp)) # <-- Pad errors
    avgTempList_final = avgTempList + [np.nan] * (final_expected_len - len(avgTempList))
    sample_temp_final = sample_temp_list + [np.nan] * (final_expected_len - len(sample_temp_list))
    P_MPa_final = P_MPa + [np.nan] * (final_expected_len - len(P_MPa)); P_calc_MPa_final = P_calc_MPa_list + [np.nan] * (final_expected_len - len(P_calc_MPa_list))
    Time_Seconds_final = Time_Seconds.tolist() + [np.nan] * (final_expected_len - len(Time_Seconds)); Time_Minutes_final = Time_Minutes + [np.nan] * (final_expected_len - len(Time_Minutes)); Time_Hours_final = Time_Hours.tolist() + [np.nan] * (final_expected_len - len(Time_Hours))
    control_tc_channel = "N/A"; control_temp_final = [np.nan] * final_expected_len
    if TC_df is not None and 'IO Type' in TC_df.columns:
        control_row = TC_df[TC_df['IO Type'].astype(str).str.strip().str.lower() == 'control']
        if not control_row.empty:
            control_tc_channel = control_row['NI LV Channel'].iloc[0]
            if runDF is not None and control_tc_channel in runDF.columns:
                control_data_raw = runDF[control_tc_channel]; control_data_numeric = pd.to_numeric(control_data_raw, errors='coerce').tolist(); control_temp_final = control_data_numeric + [np.nan] * (final_expected_len - len(control_data_numeric))
    plot_df = pd.DataFrame({'Time_Hours': Time_Hours_final,'Moles_mmol': n_final,'Moles_Uncertainty_mmol': n_error_final, 'Pressure_Meas_MPa': P_MPa_final,'Pressure_Calc_MPa': P_calc_MPa_final,'Temp_Avg_C': avgTempList_final,'Temp_Control_C': control_temp_final,'Control_TC_Channel': control_tc_channel,'Delta_V_pct': delta_V_final,'Temp_Sample_C': sample_temp_final}) # <-- Added Uncertainty
    if not (len(Time_Hours_final) == len(n_final) == len(P_MPa_final) == len(avgTempList_final) == len(P_calc_MPa_final) == len(n_error_final)): # <-- Added error length check
         print("Error: Length mismatch FINAL."); graphLabel.configure(text="Error: Final length mismatch."); return
    runData = [Time_Hours_final, Time_Minutes_final, Time_Seconds_final, n_final, P_MPa_final, avgTempList_final, runDF, TC_df, lookupTable, rawData, baseline, baselineRange, baselineUnit, baselineEdited, movingAvg, movingAvgRange, movingAvgUnit, df, data_filename, tempRange, pressureRange, timeRange, nRange, dn_dtRange, P_calc_MPa_final, full_temp_profiles_K, coarseVol, plot_df, n_error_final, delta_V_final] # item 29: thermal ΔV per timestep (% of base volume)

    # Call moleGraph

    try: moleGraph(runData)
    except Exception as e: graphLabel.configure(text=f"Error plotting: {e}"); print(f"Error moleGraph: {e}"); import traceback; traceback.print_exc(); return
    fileProgress['value'] = 100; graphProgress['value'] = 100

    # P_calc is run separately via 'Calculate P_calc' button
    if pcalc_button: pcalc_button.configure(state='normal')

    # ── Auto-save info file ────────────────────────────────────────────
    from datetime import datetime
    _ts = datetime.now().strftime('%y%m%d_%H%M%S')
    _safe_id = run_id if run_id else 'UnknownRun'
    if output_dir:
        _info_path = os.path.join(output_dir, f'{_safe_id}_{_ts}_Info.txt')
        try:
            with open(_info_path, 'w', encoding='utf-8') as _f:
                _f.write(f'MolCalc Run Info\n{"="*60}\n')
                _f.write('\n'.join(info_log))
            _log_live(f'Info saved: {os.path.basename(_info_path)}')
        except Exception as _e:
            print(f'Warning: could not save info file: {_e}')

    # ── Auto-save HTML plot ───────────────────────────────────────────────
    if output_dir:
        try:
            export_html_plot()
        except Exception as _e:
            print(f'Warning: auto HTML export failed: {_e}')

    graphLabel.configure(text="Graph Generated Successfully")
    if exportPlotButton: exportPlotButton.configure(state='normal')
    if exportHtmlButton: exportHtmlButton.configure(state='normal')
    if exportTempButton: exportTempButton.configure(state='normal')
    if tempViewButton: tempViewButton.configure(state='normal')
    if texpansion_button: texpansion_button.configure(state='normal')
    if calculateFitButton: calculateFitButton.configure(state='normal')
    if dn_dt_button: dn_dt_button.configure(state='normal')
    if save_plot_button: save_plot_button.configure(state='normal')
    window.update_idletasks()


In [19]:
# --- moleGraph function ---
def moleGraph(runData):
    global runBool, imageViewer, baselineEdited
    global plot_fig, plot_ax2, plot_canvas, plot_ax4
    [Time_Hours, Time_Minutes, Time_Seconds, n, P_MPa, avgTempList, runDF, TC_df, lookupTable,
     rawData, baseline, baselineRange, baselineUnit, baselineEdited, movingAvg, movingAvgRange,
     movingAvgUnit, df, fileName, tempRange, pressureRange, timeRange, nRange, dn_dtRange,
     P_calc_MPa, full_temp_profiles_K, coarseVol, plot_df, n_error, delta_V_pct] = runData  # 30 items
    if runBool==True:
        if imageViewer and imageViewer.winfo_exists(): imageViewer.destroy()
    runBool=True
    imageViewer = Toplevel(window); imageViewer.title(str(fileName))
    _iw, _ih = 1060, int(window.winfo_screenheight()*0.8)
    _ix = (window.winfo_screenwidth() - _iw) // 2
    _iy = max(0, (window.winfo_screenheight() - _ih) // 2)
    imageViewer.geometry(f"{_iw}x{_ih}+{_ix}+{_iy}")
    # ── Scientific plot styling ─────────────────────────────────────
    plt.rcParams.update({
        'font.family':           'DejaVu Sans',
        'font.size':             13,
        'axes.linewidth':        1.4,
        'axes.titlesize':        14,
        'axes.labelsize':        14,
        'xtick.labelsize':       12,
        'ytick.labelsize':       12,
        'xtick.direction':       'in',
        'ytick.direction':       'in',
        'xtick.minor.visible':   True,
        'ytick.minor.visible':   True,
        'xtick.major.size':      6,
        'xtick.minor.size':      3,
        'ytick.major.size':      6,
        'ytick.minor.size':      3,
        'xtick.top':             True,
        'ytick.right':           True,
        'legend.fontsize':       11,
        'legend.framealpha':     0.95,
        'legend.edgecolor':      '0.6',
        'lines.linewidth':       1.8,
        'figure.facecolor':      'white',
        'axes.facecolor':        'white',
        'axes.grid':             False,
        'axes.spines.top':       True,
        'axes.spines.right':     True,
    })

    fig = plt.figure(figsize=(10, 10))
    ax1 = fig.add_subplot(1, 1, 1)
    fig.subplots_adjust(left=0.09, right=0.82, top=0.88, bottom=0.09)
    plot_fig = fig

    n_error_np = None

    if plot_df is not None and isinstance(plot_df, pd.DataFrame):
        time_h_np = plot_df['Time_Hours'].to_numpy()
        n_np = plot_df['Moles_mmol'].to_numpy()
        p_mpa_np = plot_df['Pressure_Meas_MPa'].to_numpy()
        p_calc_mpa_np = plot_df['Pressure_Calc_MPa'].to_numpy()
        avg_temp_np = plot_df['Temp_Avg_C'].to_numpy()
        control_tc_data_numeric = plot_df['Temp_Control_C'].to_numpy()
        sample_tc_data_numeric = plot_df['Temp_Sample_C'].to_numpy() if 'Temp_Sample_C' in plot_df.columns else np.full(len(plot_df), np.nan)
        n_error_np = plot_df['Moles_Uncertainty_mmol'].to_numpy()
        control_tc_channel = plot_df['Control_TC_Channel'].iloc[0] if not plot_df.empty else 'N/A'
        mask_n = ~np.isnan(n_np); mask_p = ~np.isnan(p_mpa_np)
        mask_t = ~np.isnan(avg_temp_np); mask_p_calc = ~np.isnan(p_calc_mpa_np)
        mask_control_t = ~np.isnan(control_tc_data_numeric)
        time_control_t = time_h_np[mask_control_t] if np.any(mask_control_t) else np.array([])
    else:
        time_h_np = np.array(Time_Hours); n_np = np.array(n)
        p_mpa_np = np.array(P_MPa); avg_temp_np = np.array(avgTempList)
        p_calc_mpa_np = np.array(P_calc_MPa); n_error_np = np.array(n_error)
        control_tc_channel = None; control_tc_data_numeric = None
        sample_tc_data_numeric = np.full(len(time_h_np), np.nan)
        mask_control_t = np.array([False]*len(time_h_np)); time_control_t = np.array([])
        if TC_df is not None and 'IO Type' in TC_df.columns:
            ctrl = TC_df[TC_df['IO Type'].astype(str).str.strip().str.lower() == 'control']
            if not ctrl.empty:
                control_tc_channel = ctrl['NI LV Channel'].iloc[0]
                if runDF is not None and control_tc_channel in runDF.columns:
                    ctd = pd.to_numeric(runDF[control_tc_channel], errors='coerce').to_numpy()
                    control_tc_data_numeric = ctd
                    cl = min(len(ctd), len(time_h_np))
                    mask_control_t = ~np.isnan(ctd[:cl])
                    if np.any(mask_control_t): time_control_t = time_h_np[:cl][mask_control_t]
        mask_n = ~np.isnan(n_np); mask_p = ~np.isnan(p_mpa_np)
        mask_t = ~np.isnan(avg_temp_np); mask_p_calc = ~np.isnan(p_calc_mpa_np)

    mask_err = ~np.isnan(n_error_np)
    mask_n_with_err = mask_n & mask_err
    time_h_n   = time_h_np[mask_n]   if np.any(mask_n)   else np.array([])
    time_h_p   = time_h_np[mask_p]   if np.any(mask_p)   else np.array([])
    time_h_t   = time_h_np[mask_t]   if np.any(mask_t)   else np.array([])
    time_h_err = time_h_np[mask_n_with_err] if np.any(mask_n_with_err) else np.array([])

    # ── Axis 1: Moles (left) ─────────────────────────────────────────────
    handles = []; labels = []
    C_MOLES = '#1f77b4'   # blue
    C_PRES  = '#d62728'   # red
    C_TEMP  = '#2ca02c'   # green
    C_PCALC = '#ff7f0e'   # orange
    C_DNDT  = '#9467bd'   # purple
    _lw = float(line_thickness_var.get()) if line_thickness_var else 1.8

    if time_h_n.size > 0:
        p1, = ax1.plot(time_h_n, n_np[mask_n], color=C_MOLES,
                       linewidth=_lw, label='$n$ (mmol)')
        handles.append(p1); labels.append('$n$ (mmol)')
        if time_h_err.size > 0:
            _fb = ax1.fill_between(time_h_err,
                             n_np[mask_n_with_err] - n_error_np[mask_n_with_err],
                             n_np[mask_n_with_err] + n_error_np[mask_n_with_err],
                             color=C_MOLES, alpha=0.15, zorder=1,
                             label=r'$\pm1\sigma$ uncertainty')
            handles.append(_fb)
            labels.append(r'$\pm1\sigma$ uncertainty')
    ax1.set_ylabel('$n$ (mmol)', color=C_MOLES, labelpad=4)
    ax1.tick_params(axis='y', labelcolor=C_MOLES)
    ax1.yaxis.set_tick_params(which='both', direction='in')

    # ── Axis 2: Pressure (right 1) ───────────────────────────────────────
    ax2 = ax1.twinx(); plot_ax2 = ax2
    if time_h_p.size > 0:
        p2, = ax2.plot(time_h_p, p_mpa_np[mask_p], color=C_PRES,
                       linewidth=_lw, label='$P_{\\mathrm{meas}}$ (MPa)')
        handles.append(p2); labels.append('$P_{\\mathrm{meas}}$ (MPa)')
    ax2.set_ylabel('Pressure (MPa)', color=C_PRES, labelpad=4)
    ax2.tick_params(axis='y', labelcolor=C_PRES, direction='in', which='both')
    ax2.spines['right'].set_edgecolor(C_PRES)

    # ── Axis 3: Temperature (right 2) ────────────────────────────────────
    ax3 = ax1.twinx(); temp_label_text = 'Avg. $T$ (°C)'
    ax3.spines['right'].set_position(('outward', 60))
    ax3.spines['right'].set_edgecolor(C_TEMP)
    safe_mask_control_t = (mask_control_t[:len(control_tc_data_numeric)]
                           if control_tc_data_numeric is not None else mask_control_t)
    if (control_tc_channel and control_tc_channel != 'N/A'
            and control_tc_data_numeric is not None and time_control_t.size > 0):
        safe_len = min(len(control_tc_data_numeric), len(safe_mask_control_t))
        p3, = ax3.plot(time_control_t,
                       control_tc_data_numeric[:safe_len][safe_mask_control_t],
                       color=C_TEMP, linewidth=_lw,
                       label=f'$T_{{\\mathrm{{ctrl}}}}$ ({control_tc_channel}) (°C)')
        temp_label_text = f'$T_{{\\mathrm{{ctrl}}}}$ (°C)'
        handles.append(p3); labels.append(f'$T_{{\\mathrm{{ctrl}}}}$ (°C)')
    elif time_h_t.size > 0:
        p3a, = ax3.plot(time_h_t, avg_temp_np[mask_t], color=C_TEMP,
                        linewidth=_lw, linestyle='--', label='Avg. $T$ (°C)')
        handles.append(p3a); labels.append('Avg. $T$ (°C)')
    ax3.set_ylabel(temp_label_text, color=C_TEMP, labelpad=4)
    ax3.tick_params(axis='y', labelcolor=C_TEMP, direction='in', which='both')
    ax3.set_navigate(False)

    # ── Sample position temperature (second T trace on ax3) ───────────────
    C_SAMPLE = '#e6550d'   # burnt orange
    _spos_str = ''
    if sample_pos_var:
        try: _spos_str = sample_pos_var.get().strip()
        except Exception: pass
    if (sample_tc_data_numeric is not None
            and not np.all(np.isnan(sample_tc_data_numeric))):
        _safe_len_s = min(len(sample_tc_data_numeric), len(time_h_np))
        mask_sample_t = ~np.isnan(sample_tc_data_numeric[:_safe_len_s])
        if np.any(mask_sample_t):
            _lbl_s = (f'$T_{{\\mathrm{{sample}}}}$ @ {_spos_str} mm (°C)'
                      if _spos_str else '$T_{{\\mathrm{{sample}}}}$ (°C)')
            p3s, = ax3.plot(time_h_np[:_safe_len_s][mask_sample_t],
                            sample_tc_data_numeric[:_safe_len_s][mask_sample_t],
                            color=C_SAMPLE, linewidth=_lw, linestyle='--',
                            label=_lbl_s)
            handles.append(p3s)
            labels.append(_lbl_s)

    # ── Axis 4: dn/dt (right 3, hidden until used) ───────────────────────
    ax4 = ax1.twinx(); plot_ax4 = ax4
    ax4.yaxis.set_label_position('right')
    ax4.yaxis.tick_right()
    ax4.spines['right'].set_position(('outward', 120))
    ax4.spines['right'].set_edgecolor(C_DNDT)
    ax4.set_ylabel('d$n$/d$t$ (mmol h$^{-1}$)', color=C_DNDT, labelpad=4)
    ax4.tick_params(axis='y', labelcolor=C_DNDT, direction='in', which='both')
    ax4.set_visible(False)

    # ── Title & legend ────────────────────────────────────────────────────
    title_str = run_id if run_id else str(fileName)
    fig.suptitle(title_str, fontsize=14, fontweight='bold', x=0.455, y=0.97)
    if handles:
        ax1.legend(handles=handles, labels=labels,
                   loc='upper right',
                   framealpha=0.95, edgecolor='0.6',
                   fontsize=11, borderaxespad=1.0,
                   handlelength=2.0, labelspacing=0.5)

    ax1.tick_params(axis='x', direction='in', which='both')
    ax1.set_xlabel('Time (h)', labelpad=6)


    # ── Embed in Tkinter ──────────────────────────────────────────────────
    imageViewer.columnconfigure(1, weight=1)
    imageViewer.rowconfigure(2, weight=1)

    toolbarFrame = Frame(imageViewer)
    toolbarFrame.grid(row=1, column=1, sticky='ew')
    imageFrame = Frame(imageViewer)
    imageFrame.grid(row=2, column=1, sticky='nsew')

    canvas = FigureCanvasTkAgg(fig, master=imageFrame)
    canvas.draw()
    canvas.get_tk_widget().pack(fill='both', expand=True)
    plot_canvas = canvas
    NavigationToolbar2Tk(canvas, toolbarFrame)

    # ── Independent y-axis scrolling via mouse wheel over each axis ─────
    def _scroll_y(event):
        """Scroll/zoom the y-axis the cursor is currently over."""
        if event.inaxes is None: return
        ax = event.inaxes
        scale = 1.1 if event.button == 'down' else (1/1.1)
        y_min, y_max = ax.get_ylim()
        y_mid = event.ydata if event.ydata else (y_min + y_max) / 2
        ax.set_ylim(y_mid - (y_mid - y_min)*scale,
                    y_mid + (y_max - y_mid)*scale)
        canvas.draw_idle()

    def _scroll_x(event):
        """Shift x-axis on Shift+scroll."""
        if event.inaxes is None: return
        x_min, x_max = ax1.get_xlim()
        shift = (x_max - x_min) * 0.1
        if event.button == 'down':
            ax1.set_xlim(x_min + shift, x_max + shift)
        else:
            ax1.set_xlim(x_min - shift, x_max - shift)
        canvas.draw_idle()

    def _on_scroll(event):
        if event.key == 'shift':
            _scroll_x(event)
        else:
            _scroll_y(event)

    canvas.mpl_connect('scroll_event', _on_scroll)

    # ── Hint label ──────────────────────────────────────────────────────
    hintFrame = Frame(imageViewer)
    hintFrame.grid(row=3, column=1, sticky='ew')
    tk.Label(hintFrame,
             text='Scroll over an axis to zoom it independently  |  Shift+Scroll to pan X',
             font=('TkDefaultFont', 7), fg='gray', anchor='w').pack(side='left', padx=4)

    baselineEdited = False


In [20]:
# --- Temperature Profile Viewer Functions ---
def open_temp_viewer():
    global runData, temp_viewer_window, graphLabel
    if not runData or len(runData) < 27:
        graphLabel.configure(text="Error: Run 'Generate Graph' first."); return
    if runData[25] is None or runData[26] is None:
        graphLabel.configure(text="Error: Temp profiles N/A."); return
    if temp_viewer_window and temp_viewer_window.winfo_exists():
        temp_viewer_window.lift(); return

    time_hours_list = runData[0]      # list/Series of float hours, one per data row
    full_profiles   = runData[25]     # list of 1-mm T arrays (K), len == len(runDF)
    coarse_vol      = runData[26]     # list of floats (m³), 1-mm bins, AC coords

    # coarseVol is already 1-mm resolution (each bin = 100 x 0.01 mm volumeDF rows).
    # x_V_mm[j] = j mm in AC coordinates.
    vol_arr = np.array(coarse_vol, dtype=float)   # m³ per 1-mm slice
    x_V_mm  = np.arange(len(vol_arr), dtype=float)

    # Recover z_min_cal (start of T-profile in AC mm)
    _z_min_cal = 0
    try:
        _lut = runData[8]
        if _lut is not None and hasattr(_lut, 'columns') and len(_lut.columns) > 1:
            _z_min_cal = int(float(_lut.columns[1]))
        elif runData[7] is not None and 'X Position' in runData[7].columns:
            _tc_pos = pd.to_numeric(runData[7]['X Position'], errors='coerce').dropna()
            if not _tc_pos.empty:
                _z_min_cal = int(_tc_pos.min())
    except Exception:
        _z_min_cal = 0

    # Sample position (absolute AC mm) from GUI
    _sample_z_abs = None
    if sample_pos_var:
        try:
            _sz = sample_pos_var.get().strip()
            _sample_z_abs = float(_sz) if _sz else None
        except Exception:
            _sample_z_abs = None

    # Read z_offset so viewer shows the same T/V alignment used in the calculation
    _z_off = 0
    try:
        _zos = z_offset_var.get().strip()
        _z_off = int(float(_zos)) if _zos else 0
    except Exception:
        _z_off = 0

    t_max_hr  = max(time_hours_list) if time_hours_list else 0
    n_steps   = len(full_profiles)
    time_arr  = np.array(time_hours_list, dtype=float)

    temp_viewer_window = tk.Toplevel(window)
    temp_viewer_window.title("Temperature Profile Viewer")
    temp_viewer_window.geometry("860x640")

    # ── Control bar ────────────────────────────────────────────────────────
    ctrl = tk.Frame(temp_viewer_window); ctrl.pack(side=tk.TOP, fill=tk.X, pady=4, padx=6)

    tk.Label(ctrl, text=f'Time (h)  [0 – {t_max_hr:.3f}]:').pack(side=tk.LEFT, padx=(0,4))
    time_entry_var = tk.StringVar(value='0')
    time_entry = tk.Entry(ctrl, textvariable=time_entry_var, width=9)
    time_entry.pack(side=tk.LEFT, padx=(0,6))

    slider_var = tk.IntVar(value=0)
    _slide = ttk.Scale(ctrl, from_=0, to=max(n_steps-1,1), orient=tk.HORIZONTAL,
                       variable=slider_var, length=300)
    _slide.pack(side=tk.LEFT, padx=(0,8))

    idx_label = tk.Label(ctrl, text=f'Row index: 0 / {n_steps-1}', width=22, anchor=W)
    idx_label.pack(side=tk.LEFT)

    # ── Plot area ──────────────────────────────────────────────────────────
    plot_frame = tk.Frame(temp_viewer_window); plot_frame.pack(fill=tk.BOTH, expand=True)
    plt.rcParams.update({
        'font.family':           'DejaVu Sans',
        'font.size':             13,
        'axes.linewidth':        1.4,
        'axes.titlesize':        14,
        'axes.labelsize':        14,
        'xtick.labelsize':       12,
        'ytick.labelsize':       12,
        'xtick.direction':       'in',
        'ytick.direction':       'in',
        'xtick.minor.visible':   True,
        'ytick.minor.visible':   True,
        'xtick.major.size':      6,
        'xtick.minor.size':      3,
        'ytick.major.size':      6,
        'ytick.minor.size':      3,
        'xtick.top':             True,
        'ytick.right':           True,
        'legend.fontsize':       11,
        'legend.framealpha':     0.95,
        'legend.edgecolor':      '0.6',
        'lines.linewidth':       1.8,
        'figure.facecolor':      'white',
        'axes.facecolor':        'white',
        'axes.grid':             False,
        'axes.spines.top':       True,
        'axes.spines.right':     True,
    })
    fig_tv  = plt.figure(figsize=(8, 8))
    can_tv  = FigureCanvasTkAgg(fig_tv, master=plot_frame)
    tb_tv   = NavigationToolbar2Tk(can_tv, plot_frame)
    tb_tv.update(); tb_tv.pack(side=tk.BOTTOM, fill=tk.X)
    can_tv.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    # ── Core plot function ─────────────────────────────────────────────────
    def _draw(idx):
        idx = int(np.clip(idx, 0, n_steps-1))
        _lw = float(line_thickness_var.get()) if line_thickness_var else 1.8
        profile_K = full_profiles[idx]
        t_hr      = float(time_arr[idx]) if idx < len(time_arr) else float('nan')
        idx_label.configure(text=f'Row index: {idx} / {n_steps-1}')
        time_entry_var.set(f'{t_hr:.4f}')
        slider_var.set(idx)

        if not profile_K:
            return

        # T profile stays fixed at its calibrated AC coordinates.
        # V profile shifts by z_off so the user can see how the alignment moves.
        #   z_off > 0 → V shifts right (hot-zone moves toward sample)
        #   z_off < 0 → V shifts left
        T_use   = np.array(profile_K, dtype=float)
        if apply_smoothing_var and apply_smoothing_var.get() and len(T_use) > 4:
            from scipy.signal import savgol_filter as _sgf
            _win = min(201, len(T_use))
            if _win % 2 == 0: _win -= 1
            if _win >= 5:
                T_use = _sgf(T_use, window_length=_win, polyorder=2)
        x_T_mm  = _z_min_cal + np.arange(len(T_use), dtype=float)

        x_V_all = _z_off + np.arange(len(vol_arr), dtype=float)  # shifted V x-axis
        v_all   = vol_arr * 1e6                                    # m³/mm → mL/mm

        # Only plot the V range that overlaps with T for a clean comparison
        x_lo = max(x_T_mm[0],  x_V_all[0])
        x_hi = min(x_T_mm[-1], x_V_all[-1])
        if x_lo < x_hi:
            v_mask   = (x_V_all >= x_lo) & (x_V_all <= x_hi)
            x_V_plot = x_V_all[v_mask]
            v_plot   = v_all[v_mask]
        else:
            x_V_plot = x_V_all
            v_plot   = v_all

        fig_tv.clear()
        axT = fig_tv.add_subplot(111)
        axV = axT.twinx()

        pT, = axT.plot(x_T_mm, T_use,
                       color='steelblue', lw=_lw, label='T (K)')
        axV.fill_between(x_V_plot, v_plot, color='tomato', alpha=0.15)
        pV_line, = axV.plot(x_V_plot, v_plot,
                            color='tomato', lw=_lw, linestyle='--', alpha=0.80,
                            label='V (mL / 1-mm slice)')

        # Sample position vertical line
        _handles = [pT, pV_line]
        _labels  = ['T (K)', 'V (mL / 1-mm slice)']
        if _sample_z_abs is not None:
            _vl = axT.axvline(_sample_z_abs, color='#e6550d', lw=_lw,
                              linestyle=':', label=f'Sample @ {_sample_z_abs:.0f} mm')
            _handles.append(_vl)
            _labels.append(f'Sample @ {_sample_z_abs:.0f} mm')

        axT.set_xlabel('Axial Position (mm, AC coordinates)', labelpad=6)
        axT.set_ylabel('Temperature (K)', color='steelblue', labelpad=4)
        axT.tick_params(axis='x', direction='in', which='both')
        axT.tick_params(axis='y', labelcolor='steelblue', direction='in', which='both')
        axT.yaxis.set_tick_params(which='both', direction='in')
        _t_lo, _t_hi = axT.get_ylim()
        if _t_hi - _t_lo < 100:
            axT.set_ylim(_t_lo - 10, _t_lo + 90)
        axV.set_ylabel('Volume (mL / 1-mm slice)', color='tomato', labelpad=4)
        axV.tick_params(axis='y', labelcolor='tomato', direction='in', which='both')
        axV.spines['right'].set_edgecolor('tomato')
        _samp_info = f'  |  Sample @ {_sample_z_abs:.0f} mm' if _sample_z_abs is not None else ''
        _zoff_info = f'  |  z_off = {_z_off} mm' if _z_off != 0 else ''
        axT.set_title(
            f'T & V Profiles  |  t = {t_hr:.4f} h  |  z_min_cal = {_z_min_cal} mm{_zoff_info}{_samp_info}',
            fontsize=9)
        fig_tv.legend(handles=_handles, labels=_labels,
                      loc='upper left',
                      bbox_to_anchor=(0.76, 0.92),
                      bbox_transform=fig_tv.transFigure,
                      framealpha=0.95, edgecolor='0.6',
                      fontsize=11, borderaxespad=0,
                      handlelength=2.0, labelspacing=0.6)
        fig_tv.subplots_adjust(left=0.10, right=0.78, top=0.93, bottom=0.10)
        can_tv.draw()

    # ── Event handlers ─────────────────────────────────────────────────────
    def _on_time_entry(event=None):
        try:
            t_req = float(time_entry_var.get())
        except ValueError:
            return
        idx = int(np.argmin(np.abs(time_arr - t_req)))
        _draw(idx)

    def _on_slider(val):
        _draw(int(float(val)))

    time_entry.bind('<Return>', _on_time_entry)
    time_entry.bind('<FocusOut>', _on_time_entry)
    _slide.configure(command=_on_slider)

    tk.Button(ctrl, text='Export HTML',
              command=lambda: export_temp_profile_html(
                  full_profiles, coarse_vol, time_arr, _z_min_cal, _z_off, _sample_z_abs),
              width=12).pack(side=tk.LEFT, padx=(12, 0))

    _draw(0)


In [21]:
# --- Export Temperature Profile HTML ---
def export_temp_profile_html(full_profiles, coarse_vol, time_arr,
                              z_min_cal=0, z_off=0, sample_z_abs=None):
    """
    Export an interactive Plotly HTML showing T(z) and V(z) at every sampled
    time step — matching the GUI viewer exactly:
      - T stays fixed at z_min_cal + j AC mm
      - V shifts by z_off mm
      - Sample position shown as vertical line if set
      - Slider only (no play/pause)
    """
    import plotly.graph_objects as go
    from datetime import datetime

    global run_id, output_dir, graphLabel

    if not full_profiles or time_arr is None or len(time_arr) == 0:
        graphLabel.configure(text='Error: no profile data to export.'); return

    MAX_FRAMES = 500
    n_total = len(full_profiles)
    step    = max(1, n_total // MAX_FRAMES)
    indices = list(range(0, n_total, step))
    if indices[-1] != n_total - 1:
        indices.append(n_total - 1)
    print(f'Temp HTML export: {len(indices)} frames from {n_total} steps (step={step})')

    # coarse_vol is 1-mm resolution (m³/mm); V shifts by z_off, T fixed at z_min_cal
    vol_arr = np.array(coarse_vol, dtype=float)
    x_V_all = z_off + np.arange(len(vol_arr), dtype=float)

    # Global axis ranges
    all_T = [float(v) for i in indices for v in (full_profiles[i] or [])]
    T_min = float(np.nanmin(all_T)) if all_T else 0
    T_max = float(np.nanmax(all_T)) if all_T else 1
    V_min = float(np.nanmin(vol_arr * 1e6))
    V_max = float(np.nanmax(vol_arr * 1e6))
    T_pad = (T_max - T_min) * 0.05 or 1
    V_pad = (V_max - V_min) * 0.05 or 1e-10

    def _frame_traces(idx):
        prof  = full_profiles[idx] or []
        t_hr  = float(time_arr[idx])
        x_T   = (z_min_cal + np.arange(len(prof), dtype=float)).tolist()

        # Overlap region
        x_lo = max(x_T[0]  if x_T  else 0, float(x_V_all[0]))
        x_hi = min(x_T[-1] if x_T  else 0, float(x_V_all[-1]))
        if x_lo < x_hi:
            v_mask   = (x_V_all >= x_lo) & (x_V_all <= x_hi)
            x_V_plot = x_V_all[v_mask].tolist()
            v_plot   = (vol_arr[v_mask] * 1e6).tolist()
        else:
            x_V_plot = x_V_all.tolist()
            v_plot   = (vol_arr * 1e6).tolist()

        traces = [
            go.Scatter(x=x_T, y=[float(v) for v in prof],
                       mode='lines', name='T (K)', yaxis='y1',
                       line=dict(color='steelblue', width=1.5)),
            go.Scatter(x=x_V_plot, y=v_plot,
                       mode='lines', name='V (mL/mm)', yaxis='y2',
                       line=dict(color='tomato', width=1.1, dash='dash'),
                       fill='tozeroy', fillcolor='rgba(255,99,71,0.12)'),
        ]
        if sample_z_abs is not None:
            traces.append(go.Scatter(
                x=[sample_z_abs, sample_z_abs],
                y=[T_min - T_pad, T_max + T_pad],
                mode='lines', name=f'Sample @ {sample_z_abs:.0f} mm',
                yaxis='y1',
                line=dict(color='#e6550d', width=1.8, dash='dot'),
            ))
        return traces, f'{t_hr:.4f}'

    # Build frames
    frames = []
    slider_steps = []
    for fi, idx in enumerate(indices):
        traces, frame_name = _frame_traces(idx)
        frames.append(go.Frame(name=frame_name, data=traces))
        slider_steps.append(dict(
            args=[[frame_name],
                  dict(frame=dict(duration=0, redraw=True),
                       mode='immediate',
                       transition=dict(duration=0))],
            label=f'{float(time_arr[idx]):.3f}',
            method='animate'
        ))

    init_traces, _ = _frame_traces(indices[0])
    fig = go.Figure(data=init_traces, frames=frames)

    zoff_str   = f'z_off = {z_off} mm' if z_off != 0 else 'z_off = 0'
    samp_str   = f'  |  Sample @ {sample_z_abs:.0f} mm' if sample_z_abs is not None else ''
    fig.update_layout(
        title=dict(
            text=f'Temperature Profile — {run_id or ""}  |  z_min_cal = {z_min_cal} mm'
                 f'  |  {zoff_str}{samp_str}',
            font=dict(size=14)),
        xaxis=dict(title='Axial Position (mm, AC coordinates)', domain=[0, 0.76],
                   showline=True, linewidth=1.2, linecolor='black',
                   mirror=False, ticks='inside',
                   showgrid=False),
        yaxis=dict(
            title=dict(text='Temperature (K)', font=dict(color='steelblue')),
            tickfont=dict(color='steelblue'),
            showline=True, linewidth=1.2, linecolor='steelblue', ticks='inside',
            range=[T_min - T_pad, T_max + T_pad]
        ),
        yaxis2=dict(
            title=dict(text='Volume (mL / 1-mm slice)', font=dict(color='tomato')),
            tickfont=dict(color='tomato'),
            anchor='x', overlaying='y', side='right',
            showline=True, linewidth=1.2, linecolor='tomato', ticks='inside',
            range=[V_min - V_pad, V_max + V_pad]
        ),
        legend=dict(x=1.04, y=1.0, xanchor='left', yanchor='top',
                    bgcolor='rgba(255,255,255,0.95)',
                    bordercolor='#aaa', borderwidth=1, font=dict(size=13)),
        font=dict(family='Arial', size=13),
        template='simple_white',
        plot_bgcolor='white', paper_bgcolor='white',
        autosize=True, height=800,
        margin=dict(l=60, r=200, t=70, b=80),
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix='Time: ', suffix=' h',
                              font=dict(size=13), xanchor='center'),
            pad=dict(t=50, b=10),
            steps=slider_steps,
            x=0.0, len=1.0
        )]
    )

    ts      = datetime.now().strftime('%y%m%d_%H%M%S')
    safe_id = run_id if run_id else 'UnknownRun'
    fname   = f'{safe_id}_{ts}_TempProfile.html'
    if output_dir:
        fpath = os.path.join(output_dir, fname)
    else:
        fpath = filedialog.asksaveasfilename(
            title='Export Temperature Profile HTML',
            defaultextension='.html',
            initialfile=fname,
            filetypes=(('HTML files', '*.html'),)
        )
    if not fpath:
        graphLabel.configure(text='Temp profile HTML export cancelled.'); return

    fig.write_html(fpath, include_plotlyjs='cdn', full_html=True, config={'responsive': True})
    graphLabel.configure(text=f'Temp profile HTML saved: {os.path.basename(fpath)}')
    print(f'Temp profile HTML saved to {fpath}')


In [22]:
# --- Thermal Expansion Viewer ---
def open_texpansion_viewer():
    global runData, graphLabel
    if not runData or len(runData) < 30:
        graphLabel.configure(text="Error: Run 'Generate Graph' first."); return
    delta_V_pct = runData[29]
    time_hours  = runData[0]
    if delta_V_pct is None:
        graphLabel.configure(text="Error: Thermal expansion data N/A."); return

    win = Toplevel(window)
    win.title('Thermal Expansion — ΔV (%)')
    win.geometry('900x400')

    plt.rcParams.update({
        'font.family': 'DejaVu Sans', 'font.size': 13, 'axes.linewidth': 1.4,
        'axes.labelsize': 13, 'xtick.labelsize': 12, 'ytick.labelsize': 12,
        'legend.fontsize': 11,
        'axes.grid': True, 'grid.color': '0.88', 'grid.linewidth': 0.5,
        'xtick.direction': 'in', 'ytick.direction': 'in',
        'xtick.minor.visible': True, 'ytick.minor.visible': True,
        'figure.facecolor': 'white', 'axes.facecolor': 'white',
    })
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.set_box_aspect(1)
    fig.subplots_adjust(left=0.11, right=0.95, top=0.90, bottom=0.16)

    C_DV = '#17becf'
    time_np = np.array(time_hours, dtype=float)
    dv_np   = np.array(delta_V_pct, dtype=float)
    mask    = ~np.isnan(dv_np) & ~np.isnan(time_np)

    if np.any(mask):
        ax.plot(time_np[mask], dv_np[mask], color=C_DV, linewidth=1.5)
    ax.axhline(0, color='0.5', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Time (h)', labelpad=4)
    ax.set_ylabel('ΔV (%)', color=C_DV, labelpad=4)
    ax.set_title('Thermal Expansion — Volume Change (%)')
    ax.tick_params(axis='y', labelcolor=C_DV, direction='in', which='both')
    ax.tick_params(axis='x', direction='in', which='both')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.grid(True, color='0.88', linewidth=0.6)

    canvas = FigureCanvasTkAgg(fig, master=win)
    canvas.draw()
    canvas.get_tk_widget().pack(fill='both', expand=True)
    NavigationToolbar2Tk(canvas, win)


In [23]:
# --- Export Temp Profiles Function ---
def export_temp_profiles():
    global runData, graphLabel
    if not runData or len(runData) < 30: graphLabel.configure(text="Error: Run 'Generate Graph' first."); return
    if runData[25] is None: graphLabel.configure(text="Error: Temp profiles N/A."); return
    try:
        time_hours_list = runData[0]; full_profiles = runData[25]
        use_interpolation = runData[8] is None
        tc_positions = runData[7]['X Position'] if runData[7] is not None else None
        if not time_hours_list or not full_profiles: graphLabel.configure(text="Error: Time/profile data empty."); return
        max_len = 0; start_z = 0
        if full_profiles: max_len = max(len(p) for p in full_profiles if p)
        if use_interpolation and tc_positions is not None and not tc_positions.empty: start_z = int(tc_positions.min())
        elif not use_interpolation and runData[8] is not None:
            try: start_z = int(runData[8].columns[1])
            except (IndexError, ValueError): start_z = 0
        if max_len == 0: graphLabel.configure(text="Error: Profiles empty."); return
        file_path = filedialog.asksaveasfilename(title="Save Temp Profiles CSV", defaultextension=".csv", filetypes=(("CSV files", "*.csv"),))
        if not file_path: graphLabel.configure(text="Export cancelled."); return
        headers = ["Time_Hours"] + [f"Z_{start_z + z}_mm" for z in range(max_len)]
        data_to_export = []
        for i in range(len(time_hours_list)):
            time_val = time_hours_list[i]; profile = full_profiles[i] if i < len(full_profiles) else []
            padded_profile = profile + [np.nan] * (max_len - len(profile)); data_to_export.append([time_val] + padded_profile)
        export_df = pd.DataFrame(data_to_export, columns=headers)
        export_df.to_csv(file_path, index=False, float_format='%.3f')
        graphLabel.configure(text=f"Temp profiles exported.")
    except Exception as e: print(f"Error exporting profiles: {e}"); graphLabel.configure(text=f"Error exporting profiles: {e}")

In [24]:
# --- Info Window (live-updating) ---
def open_info_window(clear=False):
    """Open the Run Info window. If clear=True, wipe existing content first."""
    global info_log, info_window, info_text_widget
    from tkinter.scrolledtext import ScrolledText

    if info_window and info_window.winfo_exists():
        # Window already open — optionally clear for a fresh run
        if clear and info_text_widget:
            info_text_widget.config(state='normal')
            info_text_widget.delete('1.0', 'end')
            info_text_widget.config(state='normal')  # keep editable for live writes
        info_window.lift()
        return

    info_window = tk.Toplevel(window)
    info_window.title('Run Info')
    info_window.geometry('820x560')
    info_window.resizable(True, True)
    st = ScrolledText(info_window, wrap='word', font=('Courier New', 9),
                      bg='#1e1e1e', fg='#d4d4d4', insertbackground='white')
    st.pack(fill='both', expand=True, padx=6, pady=6)
    info_text_widget = st

def _log_live(text):
    """Append a line to info_log, print it, and push live to the info window."""
    line = str(text)
    info_log.append(line)
    print(line)
    if info_text_widget and info_window and info_window.winfo_exists():
        try:
            info_text_widget.config(state='normal')
            info_text_widget.insert('end', line + '\n')
            info_text_widget.see('end')
            info_text_widget.update_idletasks()
        except Exception:
            pass


In [25]:
# --- Calculate P_calc Function ---
def calculate_pcalc():
    global runData, graphLabel, plot_fig, plot_canvas, p_calc_start_time_var, p_offset_var, pcalc_info_label, p_calc_end_time_var, p_calc_step_var

    if not runData or len(runData) < 30:
        graphLabel.configure(text="Error: Run 'Generate Graph' first."); return

    # Pull required data from runData
    Time_Hours        = pd.Series(runData[0])
    n                 = runData[3]
    full_temp_profiles_K = runData[25]
    coarseVol         = runData[26]
    final_expected_len = len(runData[0])
    R = 8.3145

    # Total volume (m^3) from coarseVol
    V_total_m3 = sum(coarseVol) if coarseVol else 0.0

    # Z-offset from GUI
    try:
        z_off = z_offset_var.get().strip()
        z_offset_mm = int(float(z_off)) if z_off else 0
    except (ValueError, AttributeError):
        z_offset_mm = 0

    # P_calc start time from GUI
    try:
        start_input = p_calc_start_time_var.get().strip()
        if not start_input:
            graphLabel.configure(text="Error: Enter a P_calc Start Time (hr) first."); return
        p_calc_start_hr = float(start_input)
    except ValueError:
        graphLabel.configure(text="Error: P_calc Start Time must be numeric.")
        return

    # P_calc end time (optional)
    p_calc_end_hr = None
    try:
        end_input = p_calc_end_time_var.get().strip() if p_calc_end_time_var else ''
        if end_input:
            p_calc_end_hr = float(end_input)
            if p_calc_end_hr <= p_calc_start_hr:
                graphLabel.configure(text='Error: P_calc End Time must be after Start Time.')
                return
    except ValueError:
        graphLabel.configure(text='Error: P_calc End Time must be numeric.')
        return

    # P_calc step interval
    p_calc_step = 1
    try:
        step_in = p_calc_step_var.get().strip() if p_calc_step_var else ''
        if step_in:
            p_calc_step = max(1, int(float(step_in)))
    except (ValueError, AttributeError):
        pass
    _log_live(f'P_calc: step interval = every {p_calc_step} rows')

    graphLabel.configure(text='Calculating P_calc...')
    window.update_idletasks()

    try:
        n_start = np.nan
        P_calc_MPa_list = [np.nan] * final_expected_len
        molar_density = np.nan

        start_index_loc = (Time_Hours - p_calc_start_hr).abs().idxmin()
        if not (0 <= start_index_loc < len(n)):
            graphLabel.configure(text='Warn: P_calc start index out of range.'); return

        # n is in mmol — convert back to mol for molar density
        n_start = n[start_index_loc] / 1000.0
        actual_start_time = Time_Hours.iloc[start_index_loc]
        _log_live(f'P_calc: n_start = {n_start:.4e} mol at {actual_start_time:.3f} hr')

        if pd.isna(n_start):
            graphLabel.configure(text='Warn: n_start is NaN.'); return
        if V_total_m3 <= 0:
            graphLabel.configure(text='Warn: Total volume <= 0.'); return

        molar_density = n_start / V_total_m3
        _log_live(f'P_calc: molar density = {molar_density:.4e} mol/m³')

        P_calc_MPa_list = []
        # Determine index range for the P_calc window
        end_index_loc = final_expected_len
        if p_calc_end_hr is not None:
            end_index_loc = int((Time_Hours - p_calc_end_hr).abs().idxmin()) + 1
        _log_live(f'P_calc: computing from index {start_index_loc} to {end_index_loc-1}')

        # Sparse compute: only calculate at every p_calc_step-th index in window
        calc_indices = [i for i in range(start_index_loc, end_index_loc, p_calc_step)]
        if end_index_loc - 1 not in calc_indices:
            calc_indices.append(end_index_loc - 1)
        sparse_idx = []
        sparse_vals = []

        for i in calc_indices:
            if i >= len(full_temp_profiles_K): break
            temp_prof_K = full_temp_profiles_K[i]
            T_aligned = temp_prof_K
            V_aligned_ref = coarseVol
            if z_offset_mm > 0:
                T_aligned = temp_prof_K[z_offset_mm:] if z_offset_mm < len(temp_prof_K) else []
            elif z_offset_mm < 0:
                V_aligned_ref = coarseVol[abs(z_offset_mm):] if abs(z_offset_mm) < len(coarseVol) else []
            final_len_pcalc = min(len(T_aligned), len(V_aligned_ref))
            if final_len_pcalc == 0:
                sparse_idx.append(i); sparse_vals.append(np.nan); continue
            T_arr_pcalc = np.array(T_aligned[:final_len_pcalc])
            T_arr_pcalc[T_arr_pcalc <= 0] = 1e-6
            if np.isnan(T_arr_pcalc).any():
                sparse_idx.append(i); sparse_vals.append(np.nan); continue
            P_calc_Pa = np.nanmean(molar_density * R * T_arr_pcalc)
            sparse_idx.append(i)
            sparse_vals.append(np.nan if pd.isna(P_calc_Pa) else P_calc_Pa / 1e6)

        # Interpolate back to full resolution within the window
        if len(sparse_idx) >= 2:
            interp_vals = np.interp(
                np.arange(start_index_loc, end_index_loc),
                sparse_idx, sparse_vals
            )
            P_calc_MPa_list = ([np.nan] * start_index_loc +
                               interp_vals.tolist() +
                               [np.nan] * (final_expected_len - end_index_loc))
        elif len(sparse_idx) == 1:
            P_calc_MPa_list = [np.nan] * final_expected_len
            P_calc_MPa_list[sparse_idx[0]] = sparse_vals[0]
        else:
            P_calc_MPa_list = [np.nan] * final_expected_len

        _log_live('P_calc: calculation complete.')

        # ── Pressure offset ────────────────────────────────────────────
        # Auto-offset: difference at anchor point due to thermal gradient
        P_meas_at_start = runData[27]['Pressure_Meas_MPa'].iloc[start_index_loc]
        P_calc_at_start = P_calc_MPa_list[start_index_loc] if start_index_loc < len(P_calc_MPa_list) else np.nan
        auto_offset = (P_meas_at_start - P_calc_at_start
                       if not np.isnan(P_meas_at_start) and not np.isnan(P_calc_at_start)
                       else 0.0)

        # User offset (defaults to auto if field empty)
        user_offset = auto_offset
        try:
            ov = p_offset_var.get().strip() if p_offset_var else ''
            if ov: user_offset = float(ov)
        except ValueError:
            graphLabel.configure(text='Warning: Invalid P offset — using auto offset.')

        # Apply offset
        P_calc_MPa_list = [v + user_offset if not np.isnan(v) else np.nan
                           for v in P_calc_MPa_list]

        # Update info label
        info_txt = (f'Auto offset: {auto_offset:+.4f} MPa  |  '
                    f'Applied offset: {user_offset:+.4f} MPa  |  '
                    f'P_meas at t={actual_start_time:.3f} hr: {P_meas_at_start:.4f} MPa')
        _log_live(f'P_calc offset: {info_txt}')
        if pcalc_info_label:
            pcalc_info_label.configure(text=info_txt, fg='darkblue')
        # Pre-fill offset entry with auto value for user reference
        if p_offset_var and not p_offset_var.get().strip():
            p_offset_var.set(f'{auto_offset:.4f}')


        # Pad and update runData + plot_df
        P_calc_final = P_calc_MPa_list + [np.nan] * (final_expected_len - len(P_calc_MPa_list))
        if isinstance(runData[27], pd.DataFrame):
            runData[27]['Pressure_Calc_MPa'] = P_calc_final
        runData[24] = P_calc_final

        # Overlay on existing plot
        if plot_fig is not None and plot_canvas is not None:
            try:
                ax2_ref = plot_fig.axes[1]
                t_np  = np.array(runData[0])
                pc_np = np.array(P_calc_final)
                mask_pc = ~np.isnan(pc_np)
                if mask_pc.any():
                    for artist in ax2_ref.get_lines()[:]:
                        if artist.get_gid() == 'pcalc_line': artist.remove()
                    _lw_pc = float(line_thickness_var.get()) if line_thickness_var else 1.8
                    ax2_ref.plot(t_np[mask_pc], pc_np[mask_pc],
                                 color='darkorange', linestyle='--', linewidth=_lw_pc,
                                 label='P_calc (MPa)', gid='pcalc_line')
                    _rebuild_outside_legend(plot_fig)
                    plot_canvas.draw()
            except Exception as pe:
                print(f'Warning: could not overlay P_calc: {pe}')

        graphLabel.configure(text='P_calc complete.')

    except Exception as e:
        print(f'Error in calculate_pcalc: {e}')
        import traceback; traceback.print_exc()
        graphLabel.configure(text=f'Error in P_calc: {e}')


In [26]:
# --- Export Plot Data Function ---
def export_plot_data():
    global runData, graphLabel, output_dir, run_id
    from datetime import datetime
    if not runData or len(runData) < 30: graphLabel.configure(text="Error: Run 'Generate Graph' first."); return
    plot_df = runData[27]
    if not isinstance(plot_df, pd.DataFrame): graphLabel.configure(text='Error: plot_df missing.'); return
    try:
        if output_dir and run_id:
            ts = datetime.now().strftime('%y%m%d_%H%M%S')
            file_path = os.path.join(output_dir, f'{run_id}_{ts}_PlotData.csv')
        else:
            file_path = filedialog.asksaveasfilename(title='Save Plot Data CSV',
                defaultextension='.csv', filetypes=(('CSV files', '*.csv'),))
        if not file_path: graphLabel.configure(text='Export cancelled.'); return
        plot_df.to_csv(file_path, index=False, float_format='%.6e')
        graphLabel.configure(text=f'Plot data exported: {os.path.basename(file_path)}')
    except Exception as e: print(f'Error exporting plot data: {e}'); graphLabel.configure(text=f'Error: {e}')


In [27]:
# --- Export Interactive HTML Plot ---
def export_html_plot():
    global runData, graphLabel, run_id
    import plotly.graph_objects as go
    from datetime import datetime

    if not runData or len(runData) < 30:
        graphLabel.configure(text="Error: Run 'Generate Graph' first.")
        return

    plot_df = runData[27]
    if not isinstance(plot_df, pd.DataFrame):
        graphLabel.configure(text='Error: plot_df missing.')
        return

    try:
        # ── Build filename from RunID + timestamp ────────────────────────────
        timestamp = datetime.now().strftime('%y%m%d_%H%M%S')
        safe_run_id = run_id if run_id else 'UnknownRun'
        default_name = f'{safe_run_id}_{timestamp}.html'

        if output_dir:
            file_path = os.path.join(output_dir, default_name)
        else:
            file_path = filedialog.asksaveasfilename(
                title='Export Interactive HTML Plot',
                defaultextension='.html',
                initialfile=default_name,
                filetypes=(('HTML files', '*.html'),)
            )
        if not file_path:
            graphLabel.configure(text='HTML export cancelled.')
            return

        # ── Extract data arrays ──────────────────────────────────────────────
        t   = plot_df['Time_Hours'].to_numpy()
        n   = plot_df['Moles_mmol'].to_numpy()
        n_e = plot_df['Moles_Uncertainty_mmol'].to_numpy()
        p_m = plot_df['Pressure_Meas_MPa'].to_numpy()
        p_c = plot_df['Pressure_Calc_MPa'].to_numpy()

        # Prefer control TC; fall back to average
        ctrl_ch = plot_df['Control_TC_Channel'].iloc[0] if not plot_df.empty else 'N/A'
        if ctrl_ch and ctrl_ch != 'N/A' and 'Temp_Control_C' in plot_df.columns:
            temp     = plot_df['Temp_Control_C'].to_numpy()
            temp_lbl = f'Control T ({ctrl_ch}) (°C)'
        else:
            temp     = plot_df['Temp_Avg_C'].to_numpy()
            temp_lbl = 'Avg T (°C)'

        hover_t = [f'{v:.4f} hr' if not np.isnan(v) else 'N/A' for v in t]

        # ── Build figure ─────────────────────────────────────────────────────
        fig = go.Figure()

        # Uncertainty band (filled area, drawn first so it sits behind)
        valid_e = ~(np.isnan(n) | np.isnan(n_e))
        if valid_e.any():
            fig.add_trace(go.Scatter(
                x=np.concatenate([t[valid_e], t[valid_e][::-1]]),
                y=np.concatenate([n[valid_e] + n_e[valid_e], (n[valid_e] - n_e[valid_e])[::-1]]),
                fill='toself', fillcolor='rgba(0,0,255,0.12)',
                line=dict(color='rgba(255,255,255,0)'),
                hoverinfo='skip', showlegend=True,
                name='n (1\u03c3 uncertainty)',
                yaxis='y1'
            ))

        # Moles
        valid_n = ~np.isnan(n)
        fig.add_trace(go.Scatter(
            x=t[valid_n], y=n[valid_n],
            mode='lines', name='n (mmol)',
            line=dict(color='blue', width=1.5),
            yaxis='y1',
            hovertemplate='<b>n</b>: %{y:.4f} mmol<br>Time: %{x:.4f} hr<extra></extra>'
        ))

        # Measured pressure
        valid_pm = ~np.isnan(p_m)
        fig.add_trace(go.Scatter(
            x=t[valid_pm], y=p_m[valid_pm],
            mode='lines', name='P_meas (MPa)',
            line=dict(color='red', width=1.5),
            yaxis='y2',
            hovertemplate='<b>P_meas</b>: %{y:.4f} MPa<br>Time: %{x:.4f} hr<extra></extra>'
        ))

        # Calculated pressure
        valid_pc = ~np.isnan(p_c)
        if valid_pc.any():
            fig.add_trace(go.Scatter(
                x=t[valid_pc], y=p_c[valid_pc],
                mode='lines', name='P_calc (MPa)',
                line=dict(color='darkorange', width=1.5, dash='dash'),
                yaxis='y2',
                hovertemplate='<b>P_calc</b>: %{y:.4f} MPa<br>Time: %{x:.4f} hr<extra></extra>'
            ))

        # Temperature
        valid_t = ~np.isnan(temp)
        if valid_t.any():
            fig.add_trace(go.Scatter(
                x=t[valid_t], y=temp[valid_t],
                mode='lines', name=temp_lbl,
                line=dict(color='green', width=1.5),
                yaxis='y3',
                hovertemplate='<b>T</b>: %{y:.2f} °C<br>Time: %{x:.4f} hr<extra></extra>'
            ))

        # ── Layout ───────────────────────────────────────────────────────────
        fig.update_layout(
            title=dict(
                text=f'MolCalc — {safe_run_id}',
                font=dict(size=16)
            ),
            xaxis=dict(title='Time (h)', domain=[0, 0.70],
                       showline=True, linewidth=1.2, linecolor='black',
                       mirror=True, ticks='inside', showgrid=True,
                       gridcolor='rgba(0,0,0,0.08)'),
            yaxis=dict(
                title=dict(text='<i>n</i> (mmol)', font=dict(color='#1f77b4')),
                tickfont=dict(color='#1f77b4'),
                showline=True, linewidth=1.2, linecolor='#1f77b4',
                ticks='inside', mirror=False,
            ),
            yaxis2=dict(
                title=dict(text='Pressure (MPa)', font=dict(color='#d62728')),
                tickfont=dict(color='#d62728'),
                anchor='x', overlaying='y', side='right',
                showline=True, linewidth=1.2, linecolor='#d62728',
                ticks='inside',
            ),
            yaxis3=dict(
                title=dict(text=temp_lbl, font=dict(color='#2ca02c')),
                tickfont=dict(color='#2ca02c'),
                anchor='free', overlaying='y', side='right', position=0.86,
                showline=True, linewidth=1.2, linecolor='#2ca02c',
                ticks='inside',
            ),
            legend=dict(x=1.04, y=1.0, xanchor='left', yanchor='top',
                        bgcolor='rgba(255,255,255,0.95)',
                        bordercolor='#aaa', borderwidth=1,
                        font=dict(size=13)),
            font=dict(family='Arial', size=13),
            hovermode='x unified',
            template='simple_white',
            plot_bgcolor='white',
            paper_bgcolor='white',
            autosize=True, height=800,
            margin=dict(l=60, r=220, t=70, b=60),
        )

        # ── Write self-contained HTML ─────────────────────────────────────────
        fig.write_html(file_path, include_plotlyjs='cdn', full_html=True, config={'responsive': True})
        graphLabel.configure(text=f'HTML plot exported: {os.path.basename(file_path)}')
        print(f'HTML plot saved to {file_path}')

    except Exception as e:
        print(f'Error exporting HTML: {e}')
        import traceback; traceback.print_exc()
        graphLabel.configure(text=f'Error exporting HTML: {e}')


In [28]:
# --- Import Data Function ---
def import_data():
    global runData, graphLabel, tempViewButton, exportTempButton, exportPlotButton, calculateFitButton
    try:
        file_path = filedialog.askopenfilename(title="Import Plot Data CSV", filetypes=(("CSV files", "*.csv"),))
        if not file_path: graphLabel.configure(text="Import cancelled."); return
        imported_df = pd.read_csv(file_path)
        required_cols = ['Time_Hours', 'Moles_mmol', 'Moles_Uncertainty_mmol', 'Pressure_Meas_MPa', 'Pressure_Calc_MPa', 'Temp_Avg_C', 'Temp_Control_C', 'Control_TC_Channel']
        if not all(col in imported_df.columns for col in required_cols): graphLabel.configure(text=f"Error: Imported CSV missing columns."); print(f"Missing: {[c for c in required_cols if c not in imported_df.columns]}"); return
        Time_Hours_final = imported_df['Time_Hours'].tolist(); n_final = imported_df['Moles_mmol'].tolist()
        P_MPa_final = imported_df['Pressure_Meas_MPa'].tolist(); P_calc_MPa_final = imported_df['Pressure_Calc_MPa'].tolist()
        avgTempList_final = imported_df['Temp_Avg_C'].tolist(); control_temp_final = imported_df['Temp_Control_C'].tolist()
        n_error_final = imported_df['Moles_Uncertainty_mmol'].tolist() # <-- Import uncertainty
        control_tc_channel = imported_df['Control_TC_Channel'].iloc[0] if not imported_df.empty else "N/A"
        mock_runDF = pd.DataFrame({control_tc_channel: control_temp_final}) if control_tc_channel != "N/A" else pd.DataFrame()
        mock_TC_df = pd.DataFrame({'NI LV Channel': [control_tc_channel], 'IO Type': ['Control']}) if control_tc_channel != "N/A" else pd.DataFrame()
        mock_runData = [Time_Hours_final,None,None,n_final,P_MPa_final,avgTempList_final,mock_runDF,mock_TC_df,None,None,None,None,None,None,None,None,None,None,file_path,None,None,None,None,None,P_calc_MPa_final,None,None,imported_df,n_error_final] # <-- Add error to list
        runData = mock_runData
        if tempViewButton: tempViewButton.configure(state='disabled')
        if exportTempButton: exportTempButton.configure(state='disabled')
        if exportPlotButton: exportPlotButton.configure(state='normal')
        if exportHtmlButton: exportHtmlButton.configure(state='normal')
        if calculateFitButton: calculateFitButton.configure(state='normal')
        if dn_dt_button: dn_dt_button.configure(state='normal')
        moleGraph(mock_runData)
        graphLabel.configure(text=f"Imported & plotted: {os.path.basename(file_path)}")
    except Exception as e: print(f"Error importing data: {e}"); import traceback; traceback.print_exc(); graphLabel.configure(text=f"Error importing data: {e}")

In [29]:
# --- Import Temp Profile Viewer Function ---
def import_temp_profile_viewer():
    global graphLabel, temp_viewer_window
    try:
        file_path = filedialog.askopenfilename(title="Import Temperature Profile CSV", filetypes=(("CSV files", "*.csv"),))
        if not file_path: graphLabel.configure(text="Import cancelled."); return
        df = pd.read_csv(file_path)
        if 'Time_Hours' not in df.columns: graphLabel.configure(text="Error: Imported CSV missing 'Time_Hours'."); return
        z_cols = [col for col in df.columns if col.startswith('Z_') and col.endswith('_mm')]
        if not z_cols: graphLabel.configure(text="Error: Imported CSV missing 'Z_..._mm' cols."); return
        y_axis_time = df['Time_Hours']
        z_col_nums = sorted([(int(re.findall(r'\d+', col)[0]), col) for col in z_cols]); x_axis_z_positions = [z[0] for z in z_col_nums]; sorted_z_cols = [z[1] for z in z_col_nums]
        z_data_temps = df[sorted_z_cols].values
        
        if temp_viewer_window and temp_viewer_window.winfo_exists(): temp_viewer_window.destroy()
        temp_viewer_window = tk.Toplevel(window); temp_viewer_window.title(f"Imported Profile: {os.path.basename(file_path)}"); temp_viewer_window.geometry("800x600")
        
        fig_temp = plt.figure(); ax = fig_temp.add_subplot(111)
        im = ax.imshow(z_data_temps, aspect='auto', cmap='jet', extent=[min(x_axis_z_positions), max(x_axis_z_positions), y_axis_time.max(), y_axis_time.min()])
        ax.set_title(f"Temperature Profile (T vs. Z vs. Time)"); ax.set_xlabel("Z Position (mm)"); ax.set_ylabel("Time (Hours)")
        fig_temp.colorbar(im, ax=ax, label="Temp (K?)")
        
        plot_frame = tk.Frame(temp_viewer_window); plot_frame.pack(side=tk.TOP, fill=tk.BOTH, expand=True)
        canvas_temp = FigureCanvasTkAgg(fig_temp, master=plot_frame); canvas_temp.draw(); canvas_temp.get_tk_widget().pack(side=tk.TOP, fill=tk.BOTH, expand=True)
        toolbar_temp = NavigationToolbar2Tk(canvas_temp, plot_frame); toolbar_temp.update(); toolbar_temp.pack(side=tk.BOTTOM, fill=tk.X)
        graphLabel.configure(text=f"Temp profile viewer opened for {os.path.basename(file_path)}")
    except Exception as e: print(f"Error importing temp profile: {e}"); import traceback; traceback.print_exc(); graphLabel.configure(text=f"Error importing temp profile: {e}")

In [30]:
# --- Calculate Exponential Fit Function ---
def exp_fit_func(t, a, b, c):
    """Exponential decay function: a * exp(-b * t) + c"""
    return a * np.exp(-b * t) + c

def calculate_exponential_fit():
    global runData, fit_start_time_var, fit_end_time_var, fit_results_label
    
    if not runData or len(runData) < 30: # Check for 30 items
        fit_results_label.configure(text="Error: Must 'Generate Graph' or 'Import' data first.", fg='red')
        return
        
    plot_df = runData[27] # plot_df is at index 27
    if not isinstance(plot_df, pd.DataFrame) or 'Moles_Uncertainty_mmol' not in plot_df.columns:
        fit_results_label.configure(text="Error: Plot data/Uncertainty is missing. Re-run/Re-import.", fg='red')
        return
        
    try:
        start_h = float(fit_start_time_var.get())
        end_h = float(fit_end_time_var.get())
        if start_h >= end_h:
            fit_results_label.configure(text="Error: Start time must be before end time.", fg='red')
            return
    except ValueError:
        fit_results_label.configure(text="Error: Fit times must be numeric (e.g., 1.5).", fg='red')
        return

    try:
        df_fit = plot_df[(plot_df['Time_Hours'] >= start_h) & (plot_df['Time_Hours'] <= end_h)].copy()
        df_fit.dropna(subset=['Time_Hours', 'Pressure_Meas_MPa', 'Temp_Control_C', 'Moles_mmol', 'Moles_Uncertainty_mmol'], inplace=True)
        
        if len(df_fit) < 3:
            fit_results_label.configure(text="Error: Not enough valid data points in selected range.", fg='red')
            return

        t_data = df_fit['Time_Hours']; t_data_rel = t_data - t_data.iloc[0]; p_data = df_fit['Pressure_Meas_MPa']; t_control_data = df_fit['Temp_Control_C']
        t_sample_data = df_fit['Temp_Sample_C'] if 'Temp_Sample_C' in df_fit.columns else pd.Series([np.nan]*len(df_fit), index=df_fit.index)
        sample_temp_at_end = np.nan   # initialise; assigned after successful fit
        p0 = [p_data.iloc[0] - p_data.iloc[-1], 1.0, p_data.iloc[-1]]
        
        print(f"Fitting {len(t_data_rel)} points between {start_h} and {end_h} Hr..."); print(f"Initial guesses (a, b, c): {p0}")
        popt, pcov = curve_fit(exp_fit_func, t_data_rel, p_data, p0=p0, maxfev=5000)
        
        asymptote_c = popt[2]; temp_at_end = t_control_data.iloc[-1]
        _s_valid = t_sample_data.dropna()
        sample_temp_at_end = float(_s_valid.iloc[-1]) if not _s_valid.empty else np.nan
        
        # --- NEW: LOD Calculation ---
        delta_n_measured = df_fit['Moles_mmol'].iloc[-1] - df_fit['Moles_mmol'].iloc[0]
        sigma_n_start = df_fit['Moles_Uncertainty_mmol'].iloc[0]
        sigma_n_end = df_fit['Moles_Uncertainty_mmol'].iloc[-1]
        
        sigma_delta_n = np.sqrt(sigma_n_start**2 + sigma_n_end**2)
        LOD_3sigma_mol = 3 * sigma_delta_n
        # --- END NEW ---

        # --- Plot fit curve and asymptote onto the main pressure axis ---
        if plot_fig is not None and plot_ax2 is not None and plot_canvas is not None:
            try:
                t_fit_rel = np.linspace(0, float(t_data_rel.iloc[-1]), 500)
                p_fit_vals = exp_fit_func(t_fit_rel, *popt)
                t_fit_abs = t_fit_rel + float(t_data.iloc[0])
                # Remove any previous fit lines (tagged with gid='exp_fit')
                for artist in plot_ax2.get_lines():
                    if artist.get_gid() in ('exp_fit_curve', 'exp_fit_asymptote'):
                        artist.remove()
                _lw_fit = float(line_thickness_var.get()) if line_thickness_var else 1.8
                plot_ax2.plot(t_fit_abs, p_fit_vals, color='purple', linestyle='--',
                              linewidth=_lw_fit, label='Exp Fit', gid='exp_fit_curve')
                plot_ax2.axhline(y=asymptote_c, color='darkviolet', linestyle=':',
                                 linewidth=_lw_fit, label=f'Asymptote ({asymptote_c:.4e} MPa)',
                                 gid='exp_fit_asymptote')
                _rebuild_outside_legend(plot_fig)
                plot_canvas.draw()
                print('Fit curve and asymptote plotted on main graph.')
            except Exception as plot_err:
                print(f'Warning: Could not overlay fit on plot: {plot_err}')
        else:
            print('Warning: Main plot not available for fit overlay. Run Generate Graph first.')


        _spos_lbl = sample_pos_var.get().strip() if sample_pos_var else ""
        _samp_str = (f"Sample T @ {_spos_lbl} mm: {sample_temp_at_end:.2f} C" if (not np.isnan(sample_temp_at_end) and _spos_lbl) else ("Sample T: N/A" if np.isnan(sample_temp_at_end) else f"Sample T: {sample_temp_at_end:.2f} C"))
        result_text = f"Fit Asymptote: {asymptote_c:.4e} MPa | Control T @ {end_h:.2f} Hr: {temp_at_end:.2f} C  |  {_samp_str}\n"
        result_text += f"Measured $\Delta n$: {delta_n_measured:.3e} mol | LOD (3$\sigma$): {LOD_3sigma_mol:.3e} mol"
        
        fit_results_label.configure(text=result_text, fg='black')
        print(result_text)

        # Store fit result for manual export via 'Export Fit CSV' button
        # Write fit result directly to the run's fit CSV
        fit_row = {
            'Fit_Start_Time_hr':     start_h,
            'Fit_End_Time_hr':       end_h,
            'Fit_Param_a_MPa':       popt[0],
            'Fit_Param_b_per_hr':    popt[1],
            'Fit_Param_c_MPa':       popt[2],
            'Asymptote_MPa':         asymptote_c,
            'Control_Temp_at_End_C': temp_at_end,
            'Sample_Temp_at_End_C':  sample_temp_at_end,
            'Delta_n_mmol':          delta_n_measured,
            'Sigma_delta_n_mmol':    sigma_delta_n,
            'LOD_3sigma_mmol':       LOD_3sigma_mol,
        }
        if fit_csv_path and os.path.isdir(os.path.dirname(fit_csv_path)):
            try:
                write_header = not os.path.exists(fit_csv_path)
                pd.DataFrame([fit_row]).to_csv(
                    fit_csv_path, mode='a', index=False,
                    header=write_header, float_format='%.6e')
                graphLabel.configure(
                    text=f'Fit saved to {os.path.basename(fit_csv_path)}')
            except Exception as _csv_e:
                graphLabel.configure(text=f'Warning: fit CSV write failed: {_csv_e}')
        else:
            graphLabel.configure(
                text='Warning: Output folder not set — fit result not saved to CSV.')



    except RuntimeError:
        fit_results_label.configure(text="Error: Exponential fit failed to converge.", fg='red')
        print("Error: Exponential fit failed to converge.")
    except Exception as e:
        fit_results_label.configure(text=f"Error: {e}", fg='red')
        print(f"Error during fitting: {e}")
        import traceback; traceback.print_exc()


In [31]:
# --- Export Fit CSV & Save Plot Image ---
def export_fit_csv():
    global last_fit_result, output_dir, run_id, graphLabel, fit_csv_path
    if not last_fit_result:
        graphLabel.configure(text='No fit result to export. Run Calculate Fit first.')
        return
    if not output_dir or not run_id:
        graphLabel.configure(text='Set Output Folder before exporting.')
        return
    try:
        # Use a single persistent file for the whole run (append each fit)
        if not fit_csv_path:
            from datetime import datetime
            ts = datetime.now().strftime('%y%m%d_%H%M%S')
            fit_csv_path = os.path.join(output_dir, f'{run_id}_{ts}_FitResults.csv')
        write_header = not os.path.exists(fit_csv_path)
        pd.DataFrame([last_fit_result]).to_csv(
            fit_csv_path, mode='a', index=False,
            header=write_header, float_format='%.6e')
        graphLabel.configure(text=f'Fit appended to {os.path.basename(fit_csv_path)}')
    except Exception as e:
        graphLabel.configure(text=f'Error exporting fit: {e}')

def save_plot_image():
    global plot_fig, output_dir, run_id, graphLabel
    from datetime import datetime
    if plot_fig is None:
        graphLabel.configure(text="No plot to save. Run 'Generate Graph' first.")
        return
    if not output_dir or not run_id:
        graphLabel.configure(text='Set Output Folder before saving plot.')
        return
    try:
        ts = datetime.now().strftime('%y%m%d_%H%M%S')
        base = os.path.join(output_dir, f'{run_id}_{ts}_Plot')
        plot_fig.savefig(base + '.png', dpi=300, bbox_inches='tight', facecolor='white')
        plot_fig.savefig(base + '.svg', bbox_inches='tight', facecolor='white')
        graphLabel.configure(text=f'Plot saved: {os.path.basename(base)}.png/.svg')
    except Exception as e:
        graphLabel.configure(text=f'Error saving plot: {e}')


In [32]:
# --- Legend rebuild helper ---
def _rebuild_outside_legend(fig, canvas=None):
    """Collect all labeled artists from every axis and place legend outside the plot area."""
    seen = set()
    handles = []
    labels = []
    for ax in fig.axes:
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l and not l.startswith('_') and l not in seen:
                seen.add(l)
                handles.append(h)
                labels.append(l)
    for leg in list(fig.legends):
        leg.remove()
    if handles:
        fig.legend(handles=handles, labels=labels,
                   loc='upper left',
                   bbox_to_anchor=(0.745, 0.92),
                   bbox_transform=fig.transFigure,
                   framealpha=0.95, edgecolor='0.6',
                   fontsize=11, borderaxespad=0,
                   handlelength=2.0, labelspacing=0.6)
    if canvas is not None:
        canvas.draw()

# --- Clear Overlay Functions ---
def clear_dndt():
    global plot_ax4, plot_canvas, dn_dt_results_label
    if plot_ax4 is not None:
        plot_ax4.cla()
        plot_ax4.set_ylabel('dn/dt (mmol/hr)', color='darkorchid')
        plot_ax4.tick_params(axis='y', labelcolor='darkorchid', direction='in', which='both')
        plot_ax4.spines['right'].set_position(('outward', 120))
        plot_ax4.yaxis.set_label_position('right')
        plot_ax4.yaxis.tick_right()
        plot_ax4.set_visible(False)
        if plot_canvas: plot_canvas.draw()
    if dn_dt_results_label: dn_dt_results_label.configure(text='')

def clear_fit():
    global plot_ax2, plot_canvas, fit_results_label, last_fit_result, plot_fig
    if plot_ax2 is not None:
        for artist in plot_ax2.get_lines()[:]:
            if artist.get_gid() in ('exp_fit_curve', 'exp_fit_asymptote'):
                artist.remove()
        for coll in plot_ax2.collections[:]:
            if getattr(coll, '_gid', None) in ('exp_fit_curve', 'exp_fit_asymptote'):
                coll.remove()
        if plot_fig is not None:
            _rebuild_outside_legend(plot_fig)
        if plot_canvas: plot_canvas.draw()
    last_fit_result = None
    if fit_results_label: fit_results_label.configure(text='')

def clear_pcalc():
    global plot_ax2, plot_canvas, runData
    if plot_ax2 is not None:
        for artist in plot_ax2.get_lines()[:]:
            if artist.get_gid() == 'pcalc_line': artist.remove()
        if plot_canvas:
            _rebuild_outside_legend(plot_fig)
            plot_canvas.draw()()
    if runData and len(runData) >= 30 and isinstance(runData[27], pd.DataFrame):
        runData[27]['Pressure_Calc_MPa'] = np.nan
        runData[24] = [np.nan] * len(runData[0])
    graphLabel.configure(text='P_calc cleared.')


In [33]:
# --- Set Fit CSV Path Function ---
def set_fit_csv():
    global fit_results_csv_path, fitCsvLabel
    path = filedialog.asksaveasfilename(
        title='Choose / Create Fit Results CSV',
        defaultextension='.csv',
        filetypes=(('CSV files', '*.csv'),)
    )
    if path:
        fit_results_csv_path = path
        fitCsvLabel.configure(text=f'CSV: {os.path.basename(path)}')
        print(f'Fit results CSV set to: {path}')
    else:
        fitCsvLabel.configure(text='CSV: (not set)')

# --- Set dn/dt CSV Path Function ---
def set_dndt_csv():
    global dn_dt_csv_path, dn_dt_csv_label
    path = filedialog.asksaveasfilename(
        title='Choose / Create dn/dt Results CSV',
        defaultextension='.csv',
        filetypes=(('CSV files', '*.csv'),)
    )
    if path:
        dn_dt_csv_path = path
        if dn_dt_csv_label:
            dn_dt_csv_label.configure(text=f'CSV: {os.path.basename(path)}')
        print(f'dn/dt CSV set to: {path}')
    else:
        if dn_dt_csv_label:
            dn_dt_csv_label.configure(text='CSV: (not set)')


In [34]:
# --- Calculate dn/dt Function (interval sweep) ---
def calculate_dn_dt():
    global runData, dn_dt_start_var, dn_dt_end_var, dn_dt_interval_var
    global output_dir, run_id, graphLabel, dn_dt_results_label
    global plot_fig, plot_canvas, plot_ax4

    if not runData or len(runData) < 30:
        if dn_dt_results_label: dn_dt_results_label.configure(
            text="Error: Run 'Generate Graph' or 'Import' first.", fg='red')
        return

    plot_df = runData[27]
    if not isinstance(plot_df, pd.DataFrame):
        if dn_dt_results_label: dn_dt_results_label.configure(
            text='Error: No plot data available.', fg='red')
        return

    # ── Parse inputs ────────────────────────────────────────────────────────
    try:
        period_start = float(dn_dt_start_var.get())
        period_end   = float(dn_dt_end_var.get())
        interval_h   = float(dn_dt_interval_var.get())
        if period_start >= period_end:
            dn_dt_results_label.configure(
                text='Error: Start time must be before end time.', fg='red'); return
        if interval_h <= 0:
            dn_dt_results_label.configure(
                text='Error: Interval must be a positive number.', fg='red'); return
        if interval_h > (period_end - period_start):
            dn_dt_results_label.configure(
                text='Error: Interval is larger than the total period.', fg='red'); return
    except ValueError:
        dn_dt_results_label.configure(
            text='Error: Start, End, and Interval must all be numeric.', fg='red')
        return

    try:
        sigma_n_col = 'Moles_Uncertainty_mmol'
        results = []
        window_starts = np.arange(period_start, period_end, interval_h)

        # ── Clear previous dn/dt overlays ───────────────────────────────────
        if plot_fig is not None and plot_ax4 is not None:
            plot_ax4.cla()
            plot_ax4.yaxis.set_label_position('right')
            plot_ax4.yaxis.tick_right()
            plot_ax4.set_ylabel('dn/dt (mmol/hr)', color='darkorchid')
            plot_ax4.tick_params(axis='y', labelcolor='darkorchid')
            plot_ax4.spines['right'].set_position(('outward', 120))

        midpoints  = []
        rates      = []
        rate_errs  = []

        for idx, win_start in enumerate(window_starts):
            win_end = win_start + interval_h
            df_w = plot_df[
                (plot_df['Time_Hours'] >= win_start) &
                (plot_df['Time_Hours'] <= win_end)
            ].dropna(subset=['Time_Hours', 'Moles_mmol']).copy()

            if len(df_w) < 2:
                print(f'  Skipping interval [{win_start:.3f}, {win_end:.3f}]: < 2 points')
                continue

            t_arr = df_w['Time_Hours'].to_numpy()
            n_arr = df_w['Moles_mmol'].to_numpy()

            # Linear regression slope = dn/dt for this interval
            coeffs    = np.polyfit(t_arr, n_arr, 1)
            dn_dt_hr  = coeffs[0]          # mmol/hr
            dn_dt_s   = dn_dt_hr / 3600.0  # mmol/s
            delta_n   = n_arr[-1] - n_arr[0]
            delta_t_h = t_arr[-1] - t_arr[0]
            midpt     = (win_start + win_end) / 2.0

            # Uncertainty
            if sigma_n_col in df_w.columns:
                sig_s = df_w[sigma_n_col].iloc[0]
                sig_e = df_w[sigma_n_col].iloc[-1]
                sigma_dn_dt_hr = (np.sqrt(sig_s**2 + sig_e**2) / delta_t_h
                                  if delta_t_h > 0 else np.nan)
                sigma_dn_dt_s  = sigma_dn_dt_hr / 3600.0
            else:
                sigma_dn_dt_hr = np.nan
                sigma_dn_dt_s  = np.nan

            # Control temperature at midpoint
            temp_mid = np.nan
            if 'Temp_Control_C' in df_w.columns:
                vt = df_w['Temp_Control_C'].dropna()
                if not vt.empty: temp_mid = vt.iloc[len(vt)//2]

            midpoints.append(midpt)
            rates.append(dn_dt_hr)
            rate_errs.append(sigma_dn_dt_hr if not np.isnan(sigma_dn_dt_hr) else 0)

            results.append({
                'Window_Index':            idx + 1,
                'Start_Time_hr':           win_start,
                'End_Time_hr':             win_end,
                'Midpoint_Time_hr':        midpt,
                'dn_dt_mmol_per_hr':       dn_dt_hr,
                'dn_dt_mmol_per_s':        dn_dt_s,
                'sigma_dn_dt_mmol_per_hr': sigma_dn_dt_hr,
                'sigma_dn_dt_mmol_per_s':  sigma_dn_dt_s,
                'Delta_n_mmol':            delta_n,
                'Delta_t_hr':              delta_t_h,
                'Control_Temp_mid_C':      temp_mid,
            })

        if not results:
            dn_dt_results_label.configure(
                text='Error: No valid intervals found in the specified period.', fg='red')
            return

        # ── Plot dn/dt on dedicated 4th y-axis ──────────────────────────────
        if plot_fig is not None and plot_ax4 is not None and plot_canvas is not None:
            try:
                mid_arr  = np.array(midpoints)
                rate_arr = np.array(rates)
                err_arr  = np.array(rate_errs)
                _lw_dn = float(line_thickness_var.get()) if line_thickness_var else 1.8
                plot_ax4.errorbar(
                    mid_arr, rate_arr, yerr=err_arr,
                    fmt='o-', color='darkorchid', linewidth=_lw_dn,
                    markersize=5, capsize=3, label='dn/dt (mmol/hr)',
                    gid='dndt_axis'
                )
                plot_ax4.axhline(0, color='darkorchid', linewidth=0.6,
                                  linestyle=':', alpha=0.5)
                plot_ax4.set_visible(True)
                plot_ax4.relim()
                plot_ax4.autoscale_view()
                plot_fig.canvas.toolbar.update()  # reset home stack
                _rebuild_outside_legend(plot_fig)
                plot_canvas.draw()
            except Exception as plot_err:
                print(f'Warning: Could not plot dn/dt axis: {plot_err}')

        # ── Summary label (in dn/dt section) ────────────────────────────────
        mean_rate = np.mean(rates)
        min_rate  = np.min(rates)
        max_rate  = np.max(rates)
        result_text = (
            f"{len(results)} intervals  |  "
            f"Mean dn/dt: {mean_rate:.4e} mmol/hr  |  "
            f"Range: [{min_rate:.3e}, {max_rate:.3e}] mmol/hr\n"
            f"Period: {period_start:.3f}\u2013{period_end:.3f} hr  |  "
            f"Interval: {interval_h:.3f} hr"
        )
        dn_dt_results_label.configure(text=result_text, fg='darkorchid')
        print(result_text)

        # ── Auto-save to output_dir (new file each run) ───────────────
        from datetime import datetime
        if output_dir and run_id:
            ts = datetime.now().strftime('%y%m%d_%H%M%S')
            dndt_path = os.path.join(output_dir, f'{run_id}_{ts}_dndt.csv')
            try:
                pd.DataFrame(results).to_csv(dndt_path, index=False, float_format='%.6e')
                graphLabel.configure(text=f'dn/dt saved: {os.path.basename(dndt_path)}')
                _log_live(f'dn/dt: {len(results)} intervals saved to {os.path.basename(dndt_path)}')
            except Exception as csv_err:
                graphLabel.configure(text=f'Warning: dn/dt save failed: {csv_err}')
        else:
            graphLabel.configure(text='Warning: Set Output Folder before running dn/dt.')

    except Exception as e:
        dn_dt_results_label.configure(text=f'Error: {e}', fg='red')
        print(f'Error in calculate_dn_dt: {e}')
        import traceback; traceback.print_exc()


In [35]:
# --- Set Output Directory ---
def set_output_dir():
    global output_dir, outputDirLabel
    default = os.path.join(os.path.expanduser('~'),
        'Lehigh University Dropbox', 'ENG-MATSGroup',
        'MATS', 'Data', 'Processed - Run')
    start = default if os.path.isdir(default) else os.path.expanduser('~')
    path = filedialog.askdirectory(title='Select Output Folder', initialdir=start)
    if path:
        output_dir = path
        outputDirLabel.configure(
            text=f'Output: ...{os.sep}{os.path.basename(path)}', fg='darkgreen')
        print(f'Output directory set: {path}')
    else:
        outputDirLabel.configure(text='Output: (not set)', fg='gray')


In [36]:
# =============================================================================
# PRESSURE TRACE SYNTHESIZER
# Forward-calculates P(t) from user-defined T schedule + AC geometry.
# No raw LabVIEW data required.
# =============================================================================

def _synth_compute_volume():
    """Compute coarseVol from loaded geometry files without needing raw data."""
    global synth_coarse_vol, synth_vol_label
    global fur_file, hps_file, layout_file, mating_table_file
    global fur_mass_entries, fur_internal_checks
    global chem_species_var, chem_mass_var, chem_density_var

    if not layout_file:
        msg = 'Error: Layout file not loaded. Use Resolve Geometry or browse manually.'
        print('Synth:', msg)
        if synth_vol_label: synth_vol_label.configure(text=msg, fg='red')
        return
    if not mating_table_file or not fur_file or not hps_file:
        msg = 'Error: Geometry files missing. Use Resolve Geometry first.'
        print('Synth:', msg)
        if synth_vol_label: synth_vol_label.configure(text=msg, fg='red')
        return

    DENSITY_MAP = {
        "TZM B387 Type 364":      10.22,
        "304SS":                   7.93,
        "Molybdenum":             10.28,
        "Zr":                      6.52,
        "SS 316":                  8.00,
        "TZM B387-19, Type 364":  10.22,
    }
    CHEMICAL_PROPS = {
        'Li3BN2':  1.85,
        'Li3N':    1.27,
        'Li3AlN2': 2.31,
        'Li3GaN2': 4.10,
        'GaN':     6.15,
    }

    try:
        partList = pd.read_excel(layout_file)
    except Exception as e:
        print(f'Synth: Error reading layout: {e}')
        if synth_vol_label: synth_vol_label.configure(text=f'Error: {e}', fg='red')
        return

    volumeDF, uid_to_material = getSystemVolumeProfile(
        partList, mating_table_file, fur_file, hps_file)
    if volumeDF is None:
        print('Synth: getSystemVolumeProfile returned None.')
        if synth_vol_label: synth_vol_label.configure(text='Error: volume calc failed.', fg='red')
        return

    # --- Furniture mass correction (reuse fur_mass_entries from main UI) ---
    _any_fur = False
    for _uid, _mass_sv in fur_mass_entries.items():
        _mstr = _mass_sv.get().strip()
        if not _mstr: continue
        try:
            _mass_g = float(_mstr)
        except ValueError:
            continue
        _is_int     = fur_internal_checks.get(_uid, tk.BooleanVar(value=False)).get()
        _col_prefix = 'Internal_' if _is_int else 'External_'
        _mat  = uid_to_material.get(_uid)
        _dens = DENSITY_MAP.get(_mat) if _mat else None
        if not _dens: continue
        _V_mass_mm3 = (_mass_g / _dens) * 1000.0
        for _col in volumeDF.columns:
            if _col.startswith(_col_prefix) and _col.endswith(f'_{_uid}'):
                _V_cad = float(volumeDF[_col].sum())
                if abs(_V_cad) < 1e-9: break
                _scale = _V_mass_mm3 / abs(_V_cad)
                volumeDF[_col] = volumeDF[_col] * _scale
                _any_fur = True
                break
    if _any_fur:
        volumeDF['Total Volume'] = volumeDF.filter(like='Part_').fillna(0).sum(axis=1)

    # --- Chemical species volume ---
    _chem_sp = chem_species_var.get() if chem_species_var else 'None'
    _chem_ms = chem_mass_var.get().strip() if chem_mass_var else ''
    if _chem_sp not in ('None', '') and _chem_ms:
        try:
            _chem_g = float(_chem_ms)
            _chem_dens = (float(chem_density_var.get().strip())
                          if chem_density_var and chem_density_var.get().strip()
                          else None) if _chem_sp == 'Custom' else CHEMICAL_PROPS.get(_chem_sp)
            if _chem_dens and _chem_g > 0:
                _V_chem_mm3 = (_chem_g / _chem_dens) * 1000.0
                _cru_col = next((c for c in volumeDF.columns
                                 if c.startswith('Internal_') and 'CRU' in c.upper()), None)
                if _cru_col:
                    _nz = volumeDF[_cru_col][volumeDF[_cru_col] != 0]
                    if len(_nz) > 0:
                        _cru_s = int(_nz.index[0]); _cru_e = int(_nz.index[-1])
                        _n_sl  = _cru_e - _cru_s + 1
                        volumeDF[f'Chem_{_chem_sp}'] = pd.Series(
                            [-_V_chem_mm3 / _n_sl] * _n_sl,
                            index=range(_cru_s, _cru_e + 1))
                        _extra = volumeDF.filter(regex='^Chem_').fillna(0).sum(axis=1)
                        volumeDF['Total Volume'] = (
                            volumeDF.filter(like='Part_').fillna(0).sum(axis=1) + _extra)
        except Exception as _ce:
            print(f'Synth: chemical volume warning: {_ce}')

    # --- Coarsen to 100-mm bins -> m3 ---
    coarseVol_series = volumeDF['Total Volume']
    remainder = len(coarseVol_series) % 100
    if remainder:
        padding = pd.Series([0] * (100 - remainder),
                            index=range(len(coarseVol_series),
                                        len(coarseVol_series) + 100 - remainder))
        coarseVol_series = pd.concat([coarseVol_series, padding])
    raw = coarseVol_series.tolist()
    coarseVol = [sum(raw[100 * i:100 * (i + 1)]) / 1e9   # mm3 -> m3
                 for i in range(len(raw) // 100)]

    synth_coarse_vol = coarseVol
    V_tot_mL = sum(coarseVol) * 1e6
    vol_str = f'V = {V_tot_mL:.3f} mL  ({len(coarseVol)} bins x 100 mm)'
    if synth_vol_label:
        synth_vol_label.configure(text=vol_str, fg='darkgreen')
    print(f'Synth volume computed: {vol_str}')


# ---------------------------------------------------------------------------
# Temp-profile CSV lookup helpers
# ---------------------------------------------------------------------------

def _synth_load_lookup_table(path):
    """
    Load a MolCalc temp profile CSV as a lookup table for the synthesizer.
    Format: rows indexed by TC setpoint temperature (degC),
            columns are z positions in mm.
    Returns (lookup_table_df, z_positions_mm_array) or (None, None) on error.
    """
    global synth_lookup_table, synth_lookup_z_mm
    try:
        tbl = pd.read_csv(path, index_col=0)
        # Normalise column names to integer mm strings (same as processData)
        tbl.columns = [str(int(float(c))) for c in tbl.columns]
        # Row index should be numeric TC setpoint temperatures
        tbl.index = pd.to_numeric(tbl.index, errors='coerce')
        tbl = tbl[~tbl.index.isna()].sort_index()
        if tbl.empty:
            print('Synth: temp profile CSV is empty after parsing.')
            return None, None
        z_mm = np.array([int(c) for c in tbl.columns], dtype=float)
        synth_lookup_table = tbl
        synth_lookup_z_mm  = z_mm
        print(f'Synth: temp profile CSV loaded — '
              f'{len(tbl)} setpoint rows, z range {z_mm.min():.0f}–{z_mm.max():.0f} mm')
        return tbl, z_mm
    except Exception as e:
        print(f'Synth: error loading temp profile CSV: {e}')
        return None, None


def _synth_get_T_from_lookup(T_setpoint_C, n_bins, ambient_K):
    """
    Query the loaded temp profile lookup table at T_setpoint_C.
    Finds the closest row (TC setpoint), reads the full T(z) spatial profile,
    and resamples it to n_bins of 100 mm each (matching coarseVol resolution).
    Returns T array in K of length n_bins.
    """
    if synth_lookup_table is None or synth_lookup_z_mm is None:
        print('Synth: lookup table not loaded — falling back to uniform.')
        return np.full(n_bins, T_setpoint_C + 273.15)

    tbl  = synth_lookup_table
    z_mm = synth_lookup_z_mm

    # Find the closest row to T_setpoint_C (binary search on sorted index)
    idx = tbl.index.searchsorted(T_setpoint_C)
    idx = int(np.clip(idx, 0, len(tbl) - 1))
    T_z_C = tbl.iloc[idx].values.astype(float)

    # Resample from CSV z-positions to 100-mm bin centres used by coarseVol
    z_bin_centres = np.arange(n_bins) * 100.0 + 50.0   # mm centres
    T_resampled_C = np.interp(z_bin_centres, z_mm, T_z_C,
                               left=float(T_z_C[0]),
                               right=float(T_z_C[-1]))
    T_K = T_resampled_C + 273.15
    return np.clip(T_K, ambient_K, None)


def browse_synth_lookup_table():
    """Browse for a temp profile CSV to use in the synthesizer."""
    global synth_lookup_label
    p = filedialog.askopenfilename(
        initialdir=(os.path.dirname(temp_profile_file)
                    if temp_profile_file else os.path.expanduser('~')),
        title='Select Temp Profile CSV for Synthesizer',
        filetypes=(('CSV files', '*.csv'),))
    if not p:
        return
    tbl, z_mm = _synth_load_lookup_table(p)
    if tbl is not None and synth_lookup_label:
        synth_lookup_label.configure(
            text=f'{os.path.basename(p)}  '
                 f'({len(tbl)} setpoint rows, '
                 f'z {z_mm.min():.0f}–{z_mm.max():.0f} mm)',
            fg='black')
    elif synth_lookup_label:
        synth_lookup_label.configure(text='Error loading file.', fg='red')


def synth_use_loaded_temp_profile():
    """Load the temp profile CSV that is already selected in the File Selection section."""
    global synth_lookup_label
    if not temp_profile_file:
        if synth_lookup_label:
            synth_lookup_label.configure(
                text='No temp profile loaded in File Selection — browse below.',
                fg='darkorange')
        return
    tbl, z_mm = _synth_load_lookup_table(temp_profile_file)
    if tbl is not None and synth_lookup_label:
        synth_lookup_label.configure(
            text=f'{os.path.basename(temp_profile_file)}  '
                 f'({len(tbl)} setpoint rows, '
                 f'z {z_mm.min():.0f}–{z_mm.max():.0f} mm)',
            fg='darkgreen')
    elif synth_lookup_label:
        synth_lookup_label.configure(text='Error loading temp profile.', fg='red')


# ---------------------------------------------------------------------------
# Core T-profile builder
# ---------------------------------------------------------------------------

def _synth_build_T_profile(T_set_K, n_bins, mode, ambient_K, loaded_profile):
    """
    Return a numpy array of temperatures (K) of length n_bins for a given setpoint.

    mode == 'uniform'     : all bins at T_set_K
    mode == 'gradient'    : linear from ambient_K (z=0) to T_set_K (z=end)
    mode == 'profile_csv' : query synth_lookup_table at T_set_K-273.15
    mode == 'loaded'      : scale a loaded (z_arr, T_arr_K) shape proportionally
    """
    T_set_K = max(T_set_K, ambient_K)

    if mode == 'uniform':
        return np.full(n_bins, T_set_K)

    elif mode == 'gradient':
        return np.linspace(ambient_K, T_set_K, n_bins)

    elif mode == 'profile_csv':
        return _synth_get_T_from_lookup(T_set_K - 273.15, n_bins, ambient_K)

    elif mode == 'loaded' and loaded_profile is not None:
        z_ref, T_ref_K = loaded_profile
        src_idx  = np.linspace(0, len(T_ref_K) - 1, n_bins)
        T_interp = np.interp(src_idx, np.arange(len(T_ref_K)), T_ref_K)
        T_max    = float(np.max(T_interp))
        T_min_r  = float(np.min(T_interp))
        span     = T_max - T_min_r
        if span > 1.0:
            scale = (T_set_K - ambient_K) / span
            T_out = ambient_K + scale * (T_interp - T_min_r)
        else:
            T_out = np.full(n_bins, T_set_K)
        return np.clip(T_out, ambient_K, None)

    else:
        return np.full(n_bins, T_set_K)  # fallback


# ---------------------------------------------------------------------------
# Equilibrium pressure solver
# ---------------------------------------------------------------------------

def _synth_compute_P(n_mol, T_arr_K, coarse_vol, z_interp=None,
                     z_T_min=250, z_T_max=2000, z_P_min=0.1, z_P_max=50, R=8.3145):
    """
    Compute equilibrium pressure (Pa) for fixed n_mol using real-gas iteration.
    P = n / sum(Vi / (Zi * R * Ti))
    Converges in typically < 10 iterations.
    """
    V_arr = np.array(coarse_vol, dtype=float)
    n_b   = len(V_arr)
    T_in  = np.array(T_arr_K, dtype=float)
    if len(T_in) >= n_b:
        T_use = T_in[:n_b]
    else:
        T_use = np.concatenate([T_in, np.full(n_b - len(T_in),
                                               T_in[-1] if len(T_in) else 298.15)])
    T_use = np.clip(T_use, 1e-6, None)

    # Ideal-gas first guess
    P_guess = n_mol * R * float(np.mean(T_use)) / float(np.sum(V_arr))
    if not np.isfinite(P_guess) or P_guess <= 0:
        P_guess = 1e5

    for _ in range(20):
        if z_interp is not None:
            P_MPa_g = float(np.clip(P_guess / 1e6, z_P_min, z_P_max))
            T_cl    = np.clip(T_use, z_T_min, z_T_max)
            Z_arr   = z_interp(np.column_stack([T_cl, np.full(n_b, P_MPa_g)]))
            Z_arr   = np.clip(Z_arr, 0.5, 2.0)
        else:
            Z_arr = np.ones(n_b)
        denom = float(np.sum(V_arr / (Z_arr * R * T_use)))
        if denom <= 0:
            return np.nan
        P_new = n_mol / denom
        if not np.isfinite(P_new):
            return np.nan
        if abs(P_new - P_guess) / max(abs(P_guess), 1e-12) < 1e-8:
            break
        P_guess = P_new
    return P_new   # Pa


# ---------------------------------------------------------------------------
# Standalone plot window
# ---------------------------------------------------------------------------

def _synth_plot_standalone(time_arr, P_synth, sched_t, sched_T,
                           gas_name, T_amb_K, mode):
    """Open a standalone Toplevel window with the synthesized P(t) plot."""
    win = tk.Toplevel(window)
    win.title('Pressure Trace Synthesizer - Result')
    win.geometry('960x480')

    plt.rcParams.update({
        'font.family': 'DejaVu Sans', 'font.size': 13, 'axes.linewidth': 1.4,
        'axes.labelsize': 13, 'xtick.labelsize': 12, 'ytick.labelsize': 12,
        'legend.fontsize': 11,
        'axes.grid': True, 'grid.color': '0.88', 'grid.linewidth': 0.5,
        'xtick.direction': 'in', 'ytick.direction': 'in',
        'xtick.minor.visible': True, 'ytick.minor.visible': True,
        'figure.facecolor': 'white', 'axes.facecolor': 'white',
    })
    fig, ax_p = plt.subplots(figsize=(8, 8))
    ax_p.set_box_aspect(1)
    fig.subplots_adjust(left=0.10, right=0.88, top=0.91, bottom=0.18)

    C_P = '#d62728'
    C_T = '#ff7f0e'

    mask = ~np.isnan(P_synth)
    ax_p.plot(time_arr[mask], P_synth[mask], color=C_P, linewidth=1.8,
              label='P_synth (MPa)')
    ax_p.set_xlabel('Time (h)', labelpad=4)
    ax_p.set_ylabel('Pressure (MPa)', color=C_P, labelpad=4)
    ax_p.tick_params(axis='y', labelcolor=C_P, direction='in', which='both')
    ax_p.tick_params(axis='x', direction='in', which='both')
    ax_p.spines['top'].set_visible(False)

    ax_t = ax_p.twinx()
    ax_t.plot(sched_t, sched_T, color=C_T, linestyle='--', linewidth=1.4,
              marker='o', markersize=4, label='T setpoint (degC)')
    ax_t.set_ylabel('T setpoint (degC)', color=C_T, labelpad=4)
    ax_t.tick_params(axis='y', labelcolor=C_T, direction='in', which='both')
    ax_t.spines['top'].set_visible(False)

    T_amb_C = T_amb_K - 273.15
    ax_p.set_title(
        f'Synthesized Pressure Trace  |  Gas: {gas_name}  |  Mode: {mode}  |  '
        f'T_amb = {T_amb_C:.0f} degC',
        fontsize=9, pad=5)

    lines1, labs1 = ax_p.get_legend_handles_labels()
    lines2, labs2 = ax_t.get_legend_handles_labels()
    fig.legend(lines1 + lines2, labs1 + labs2,
               loc='upper center', bbox_to_anchor=(0.5, 0.06),
               ncol=2, fontsize=9, framealpha=0.95, edgecolor='0.6',
               bbox_transform=fig.transFigure)

    canvas = FigureCanvasTkAgg(fig, master=win)
    canvas.draw()
    canvas.get_tk_widget().pack(fill='both', expand=True)
    NavigationToolbar2Tk(canvas, win)


# ---------------------------------------------------------------------------
# Browse for simple 2-col spatial profile (legacy mode)
# ---------------------------------------------------------------------------

def browse_synth_spatial_profile():
    """Browse for a 1-D spatial profile CSV (columns: z_mm, T_C) — legacy mode."""
    global synth_spatial_file, synth_loaded_profile, synth_spatial_label
    p = filedialog.askopenfilename(
        initialdir=os.path.expanduser('~'),
        title='Select Spatial Profile CSV  (columns: z_mm, T_C)',
        filetypes=(('CSV files', '*.csv'),))
    if not p:
        return
    try:
        df_sp = pd.read_csv(p)
        num_cols = [c for c in df_sp.columns
                    if pd.api.types.is_numeric_dtype(df_sp[c])]
        if len(num_cols) < 2:
            print(f'Synth spatial CSV: need >= 2 numeric cols; got {df_sp.columns.tolist()}')
            return
        z_arr   = df_sp[num_cols[0]].values.astype(float)
        T_arr_C = df_sp[num_cols[1]].values.astype(float)
        T_arr_K = T_arr_C + 273.15
        synth_loaded_profile = (z_arr, T_arr_K)
        synth_spatial_file   = p
        if synth_spatial_label:
            synth_spatial_label.configure(
                text=f'{os.path.basename(p)}  ({len(z_arr)} pts, '
                     f'T range {T_arr_C.min():.0f}-{T_arr_C.max():.0f} degC)',
                fg='black')
        print(f'Synth spatial profile loaded: {p}  ({len(z_arr)} pts)')
    except Exception as e:
        print(f'Error loading spatial profile: {e}')
        if synth_spatial_label:
            synth_spatial_label.configure(text=f'Error: {e}', fg='red')


# ---------------------------------------------------------------------------
# Main synthesizer
# ---------------------------------------------------------------------------

def synthesize_pressure_trace():
    """
    Forward-calculate a synthetic pressure trace P(t) from:
      - AC geometry (layout + MLD files already loaded)
      - User-defined furnace temperature schedule
      - Initial conditions P0 (MPa), T0 (degC)
      - Gas species
    No raw LabVIEW data needed.
    """
    global synth_coarse_vol, synth_status_label, synth_n_label
    global gas_species_var, plot_fig, plot_ax2, plot_canvas, output_dir

    # -- 1. Parse scalar inputs -----------------------------------------------
    try:
        P0_MPa   = float(synth_p0_var.get())
        T0_C     = float(synth_t0_var.get())
        T_amb_C  = float(synth_ambient_var.get().strip() or '25')
        total_hr = float(synth_total_time_var.get())
        dt_hr    = float(synth_dt_var.get())
    except (ValueError, AttributeError) as e:
        if synth_status_label:
            synth_status_label.configure(text=f'Input error: {e}', fg='red')
        return

    if dt_hr <= 0 or total_hr <= 0:
        synth_status_label.configure(
            text='Error: Total time and time step must be > 0.', fg='red')
        return

    # -- 2. Parse furnace schedule ---------------------------------------------
    try:
        raw_sched = synth_schedule_text.get('1.0', 'end').strip()
        sched_t_list, sched_T_list = [], []
        for line in raw_sched.splitlines():
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.replace(';', ',').split(',')
            if len(parts) < 2:
                continue
            sched_t_list.append(float(parts[0].strip()))
            sched_T_list.append(float(parts[1].strip()))
        if not sched_t_list:
            synth_status_label.configure(
                text='Error: Enter at least one schedule line (time_hr, T_degC).',
                fg='red')
            return
        sched_t = np.array(sched_t_list, dtype=float)
        sched_T = np.array(sched_T_list, dtype=float)
    except (ValueError, AttributeError) as e:
        synth_status_label.configure(text=f'Schedule parse error: {e}', fg='red')
        return

    # -- 3. Ensure volume profile is available --------------------------------
    if synth_coarse_vol is None:
        synth_status_label.configure(text='Computing volume profile...', fg='darkorange')
        synth_status_label.update_idletasks()
        _synth_compute_volume()
        if synth_coarse_vol is None:
            synth_status_label.configure(
                text='Error: Volume not computed. Load geometry files then click '
                     '"Compute Volume Profile".', fg='red')
            return
    coarse_vol = synth_coarse_vol
    n_bins = len(coarse_vol)
    V_arr  = np.array(coarse_vol, dtype=float)
    if V_arr.sum() <= 0:
        synth_status_label.configure(text='Error: Total volume is zero.', fg='red')
        return

    # -- 4. Load Z table -------------------------------------------------------
    from scipy.interpolate import RegularGridInterpolator as _RGI
    _z_interp = None
    _z_T_min, _z_T_max, _z_P_min, _z_P_max = 250.0, 2000.0, 0.1, 50.0
    _gas_name = gas_species_var.get() if gas_species_var else 'Nitrogen'
    _gas_file_map = {'Nitrogen': 'N2_Z.npz'}
    _gas_stem = _gas_file_map.get(_gas_name, 'N2_Z.npz')
    try:
        _zd = np.load(os.path.join(CANON_MLD_DIR, _gas_stem))
        _z_interp = _RGI((_zd['T_K'], _zd['P_MPa']), _zd['Z'],
                          bounds_error=False, fill_value=None)
        _z_T_min = float(_zd['T_K'].min())
        _z_T_max = float(_zd['T_K'].max())
        _z_P_min = float(_zd['P_MPa'].min())
        _z_P_max = float(_zd['P_MPa'].max())
        print(f'Synth: Z table loaded ({_gas_name})')
    except Exception as _ze:
        print(f'Synth: Z table not found ({_ze}) -- using ideal gas Z=1.')

    R       = 8.3145
    T0_K    = T0_C + 273.15
    T_amb_K = T_amb_C + 273.15
    mode    = synth_profile_mode_var.get() if synth_profile_mode_var else 'uniform'

    # -- 5. Validate profile_csv mode is ready --------------------------------
    if mode == 'profile_csv' and synth_lookup_table is None:
        synth_status_label.configure(
            text='Error: No temp profile CSV loaded for Synthesizer. '
                 'Use "Use Loaded Profile" or browse for a file.',
            fg='red')
        return

    # -- 6. Derive n from initial conditions ----------------------------------
    T0_profile = _synth_build_T_profile(T0_K, n_bins, mode, T_amb_K, synth_loaded_profile)
    P0_Pa = P0_MPa * 1e6
    if _z_interp is not None:
        Z0_arr = _z_interp(np.column_stack([
            np.clip(T0_profile, _z_T_min, _z_T_max),
            np.full(n_bins, float(np.clip(P0_Pa / 1e6, _z_P_min, _z_P_max)))]))
        Z0_arr = np.clip(Z0_arr, 0.5, 2.0)
    else:
        Z0_arr = np.ones(n_bins)
    n_mol = P0_Pa * float(np.sum(V_arr / (Z0_arr * R * T0_profile)))

    if synth_n_label:
        synth_n_label.configure(
            text=f'n = {n_mol * 1000:.4f} mmol  ({n_mol:.4e} mol)  |  '
                 f'V_total = {V_arr.sum() * 1e6:.3f} mL')
    print(f'Synth: n={n_mol:.4e} mol  P0={P0_MPa} MPa  T0={T0_C} degC  '
          f'V={V_arr.sum()*1e6:.3f} mL  mode={mode}')

    # -- 7. Main time loop ----------------------------------------------------
    synth_status_label.configure(text='Running synthesizer...', fg='darkorange')
    synth_status_label.update_idletasks()

    time_arr = np.arange(0.0, total_hr + dt_hr / 2, dt_hr)
    P_synth  = np.empty(len(time_arr))

    for idx, t_hr in enumerate(time_arr):
        T_set_C = float(np.interp(t_hr, sched_t, sched_T,
                                   left=float(sched_T[0]),
                                   right=float(sched_T[-1])))
        T_set_K  = T_set_C + 273.15
        T_prof_K = _synth_build_T_profile(
            T_set_K, n_bins, mode, T_amb_K, synth_loaded_profile)
        P_Pa = _synth_compute_P(
            n_mol, T_prof_K, coarse_vol,
            z_interp=_z_interp,
            z_T_min=_z_T_min, z_T_max=_z_T_max,
            z_P_min=_z_P_min, z_P_max=_z_P_max, R=R)
        P_synth[idx] = P_Pa / 1e6 if np.isfinite(P_Pa) else np.nan

    P_valid = P_synth[~np.isnan(P_synth)]
    result_str = (f'{len(time_arr)} steps  |  '
                  f'P range [{P_valid.min():.3f}, {P_valid.max():.3f}] MPa'
                  if len(P_valid) else 'No valid P values.')
    print(f'Synth complete: {result_str}')

    # -- 8. Plot --------------------------------------------------------------
    _pf = globals().get('plot_fig')
    _pa = globals().get('plot_ax2')
    _pc = globals().get('plot_canvas')
    if _pf is not None and _pa is not None and _pc is not None:
        for artist in _pa.get_lines()[:]:
            if artist.get_gid() == 'synth_P_line':
                artist.remove()
        _lw_sy = float(line_thickness_var.get()) if line_thickness_var else 1.8
        _pa.plot(time_arr, P_synth,
                      color='#9467bd', linestyle='-.', linewidth=_lw_sy,
                      label='P_synth (MPa)', gid='synth_P_line')
        _rebuild_outside_legend(_pf)
        _pc.draw()
    else:
        _synth_plot_standalone(time_arr, P_synth, sched_t, sched_T,
                               _gas_name, T_amb_K, mode)

    # -- 9. Auto-export CSV ---------------------------------------------------
    if output_dir:
        from datetime import datetime
        ts = datetime.now().strftime('%y%m%d_%H%M%S')
        csv_path = os.path.join(output_dir, f'Synth_{ts}.csv')
        try:
            pd.DataFrame({
                'Time_Hours':   time_arr,
                'P_Synth_MPa':  P_synth
            }).to_csv(csv_path, index=False, float_format='%.6f')
            result_str += f'  |  Saved: {os.path.basename(csv_path)}'
            print(f'Synth CSV saved: {csv_path}')
        except Exception as _e:
            print(f'Warning: synth CSV save failed: {_e}')

    synth_status_label.configure(text=result_str, fg='darkgreen')


def clear_synth_trace():
    """Remove synthesizer overlay from the main plot."""
    global plot_ax2, plot_canvas, plot_fig, synth_status_label, synth_n_label
    if plot_fig is not None and plot_ax2 is not None:
        for artist in plot_ax2.get_lines()[:]:
            if artist.get_gid() == 'synth_P_line':
                artist.remove()
        if plot_canvas:
            ax1_ref = _pf.axes[0]
            h, lb = [], []
            for ax in plot_fig.axes:
                for line in ax.get_lines():
                    lbl = line.get_label()
                    if lbl and not lbl.startswith('_') and 'P_synth' not in lbl:
                        h.append(line); lb.append(lbl)
            ax1_ref.legend(handles=h, labels=lb, loc='upper left',
                           bbox_to_anchor=(0.01, 0.99), framealpha=0.85,
                           edgecolor='0.7', fontsize=8)
            plot_canvas.draw()
    if synth_status_label:
        synth_status_label.configure(text='Synth trace cleared.', fg='gray')
    if synth_n_label:
        synth_n_label.configure(text='')


In [37]:
# --- MAIN GUI ---
window = tk.Tk()
window.title('MoleGrapher 1.4.0')
window.geometry('760x880')
window.resizable(True, True)

# ── Status bar (always visible at bottom) ────────────────────────────────
statusFrame = tk.Frame(window, bd=1, relief=tk.SUNKEN)
statusFrame.pack(side=tk.BOTTOM, fill=tk.X, padx=6, pady=(0, 4))
graphLabel = tk.Label(statusFrame,
                       text="Press 'Generate Graph' or 'Import...' to begin.",
                       anchor=W, fg='navy')
graphLabel.pack(side=tk.LEFT, padx=4)

# ── Run-finder helpers ────────────────────────────────────────────────────
_PROCESSED_RUN_BASE = os.path.join(
    os.path.expanduser('~'),
    'Lehigh University Dropbox', 'ENG-MATSGroup', 'MATS', 'Data', 'Processed - Run'
)

def _open_run_folder(run_id, folder_path):
    files = sorted(f for f in os.listdir(folder_path)
                   if os.path.isfile(os.path.join(folder_path, f)))
    if not files:
        find_status_label.configure(text='No files found in that folder.')
        return
    win = tk.Toplevel(window)
    win.title(f'Run: {run_id}')
    win.geometry('520x380')
    win.lift()
    tk.Label(win, text=f'Run: {run_id}',
             font=('TkDefaultFont', 12, 'bold')).pack(pady=(12, 2))
    tk.Label(win, text=folder_path, fg='gray', wraplength=480,
             font=('TkDefaultFont', 8)).pack(pady=(0, 8))
    _lf = tk.Frame(win); _lf.pack(fill=tk.BOTH, expand=True, padx=10)
    _sb = tk.Scrollbar(_lf); _sb.pack(side=tk.RIGHT, fill=tk.Y)
    _lb = tk.Listbox(_lf, yscrollcommand=_sb.set, font=('TkDefaultFont', 10),
                     activestyle='dotbox', selectmode=tk.SINGLE)
    _lb.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
    _sb.config(command=_lb.yview)
    for fname in files:
        _lb.insert(tk.END, fname)
    def _open_file(event=None):
        sel = _lb.curselection()
        if not sel:
            return
        fpath = os.path.join(folder_path, files[sel[0]])
        import subprocess, sys
        if sys.platform == 'darwin':
            subprocess.Popen(['open', fpath])
        elif sys.platform == 'win32':
            os.startfile(fpath)
        else:
            subprocess.Popen(['xdg-open', fpath])
    _lb.bind('<Double-Button-1>', _open_file)
    tk.Button(win, text='Open', command=_open_file).pack(pady=(6, 10))

def search_run_id():
    run_id = run_id_search_var.get().strip()
    find_status_label.configure(text='')
    if len(run_id) < 6:
        find_status_label.configure(text='Run ID must be at least 6 characters.')
        return
    folder_path = os.path.join(_PROCESSED_RUN_BASE, run_id[:2], run_id[2:4], run_id)
    if not os.path.isdir(folder_path):
        find_status_label.configure(text=f'Folder not found: ...{os.sep}{run_id[:2]}{os.sep}{run_id[2:4]}{os.sep}{run_id}')
        return
    _open_run_folder(run_id, folder_path)

def go_to_main():
    landing_frame.pack_forget()
    _outer.pack(fill=tk.BOTH, expand=True)
    window.update_idletasks()
    _rw = min(mainFrame.winfo_reqwidth() + 40, window.winfo_screenwidth() - 40)
    _rh = min(int(window.winfo_screenheight() * 0.85), window.winfo_screenheight() - 60)
    _rx = (window.winfo_screenwidth() - _rw) // 2
    _ry = (window.winfo_screenheight() - _rh) // 2
    window.geometry(f'{_rw}x{_rh}+{_rx}+{_ry}')

def go_to_landing():
    _outer.pack_forget()
    landing_frame.pack(fill=tk.BOTH, expand=True)
    window.update_idletasks()
    _lw, _lh = 500, 420
    _lx = (window.winfo_screenwidth() - _lw) // 2
    _ly = (window.winfo_screenheight() - _lh) // 2
    window.geometry(f'{_lw}x{_lh}+{_lx}+{_ly}')

# ── Landing page ──────────────────────────────────────────────────────────
landing_frame = tk.Frame(window, bg='white')
landing_frame.pack(fill=tk.BOTH, expand=True)

tk.Label(landing_frame, text='MolCalc',
         font=('TkDefaultFont', 28, 'bold'), bg='white').pack(pady=(70, 30))

tk.Button(landing_frame, text='Process a Run',
          font=('TkDefaultFont', 14), width=22,
          command=go_to_main).pack(pady=8)

tk.Frame(landing_frame, height=2, bg='#cccccc').pack(fill=tk.X, padx=80, pady=22)

_find_outer = tk.Frame(landing_frame, bg='white')
_find_outer.pack()
tk.Label(_find_outer, text='Find a Processed Run',
         font=('TkDefaultFont', 13, 'bold'), bg='white').pack(pady=(0, 12))
_find_row = tk.Frame(_find_outer, bg='white'); _find_row.pack()
tk.Label(_find_row, text='Run ID:', bg='white',
         font=('TkDefaultFont', 11)).pack(side=tk.LEFT, padx=(0, 6))
run_id_search_var = tk.StringVar()
_run_id_entry = tk.Entry(_find_row, textvariable=run_id_search_var, width=16,
                          font=('TkDefaultFont', 11))
_run_id_entry.pack(side=tk.LEFT, padx=(0, 6))
_run_id_entry.bind('<Return>', lambda e: search_run_id())
tk.Button(_find_row, text='Search', command=search_run_id,
          font=('TkDefaultFont', 11)).pack(side=tk.LEFT)
find_status_label = tk.Label(_find_outer, text='', fg='red', bg='white', wraplength=420)
find_status_label.pack(pady=(8, 0))

# ── Scrollable main area ──────────────────────────────────────────────────
_outer = tk.Frame(window)  # packed by go_to_main()

_vscroll = tk.Scrollbar(_outer, orient='vertical')
_vscroll.pack(side=tk.RIGHT, fill=tk.Y)
_hscroll = tk.Scrollbar(_outer, orient='horizontal')
_hscroll.pack(side=tk.BOTTOM, fill=tk.X)

_canvas_scroll = tk.Canvas(_outer, yscrollcommand=_vscroll.set,
                           xscrollcommand=_hscroll.set,
                           highlightthickness=0)
_canvas_scroll.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

_vscroll.config(command=_canvas_scroll.yview)
_hscroll.config(command=_canvas_scroll.xview)

mainFrame = tk.Frame(_canvas_scroll)
_canvas_scroll.create_window((0, 0), window=mainFrame, anchor='nw')

def _on_frame_configure(event):
    _canvas_scroll.configure(scrollregion=_canvas_scroll.bbox('all'))

mainFrame.bind('<Configure>', _on_frame_configure)

# ── Back button (top of main UI) ──────────────────────────────────────────
_back_row = tk.Frame(mainFrame)
_back_row.pack(fill=tk.X, padx=6, pady=(4, 0))
tk.Button(_back_row, text='← Back to MolCalc', command=go_to_landing,
          font=('TkDefaultFont', 9)).pack(side=tk.LEFT)

# Allow mousewheel scrolling
def _on_mousewheel(event):
    try:
        if event.widget.winfo_toplevel() is not window:
            return
    except Exception:
        return
    _canvas_scroll.yview_scroll(int(-1*(event.delta/120)), 'units')
window.bind_all('<MouseWheel>', _on_mousewheel)
# Linux mousewheel
window.bind_all('<Button-4>', lambda e: _canvas_scroll.yview_scroll(-1, 'units'))
window.bind_all('<Button-5>', lambda e: _canvas_scroll.yview_scroll( 1, 'units'))

PAD = dict(padx=6, pady=3)

# ── Section 1: File Selection ─────────────────────────────────────────────
fileFrame = tk.LabelFrame(mainFrame, text='File Selection', font=('TkDefaultFont', 9, 'bold'))
fileFrame.pack(fill=tk.X, **PAD)

row0 = tk.Frame(fileFrame); row0.pack(fill=tk.X, padx=4, pady=2)
fileBrowseButton = tk.Button(row0, text='Browse Data...', command=lambda: (browseFiles(), resolve_geometry_files()), width=16)
fileBrowseButton.pack(side=tk.LEFT, padx=(0,6))
fileLabel = tk.Label(row0, text='Data: (Not Selected)', anchor=W, fg='gray')
fileLabel.pack(side=tk.LEFT, fill=tk.X, expand=True)

# Pressure column selector (populated by _populate_pt_dropdown after Browse)
ptRow = tk.Frame(fileFrame); ptRow.pack(fill=tk.X, padx=4, pady=2)
ptLabel = tk.Label(ptRow, text='Pressure Col: (load data file first)',
                   anchor=W, fg='gray', width=34)
ptLabel.pack(side=tk.LEFT)
pressure_col_var = tk.StringVar(value='')
pt_menu = tk.OptionMenu(ptRow, pressure_col_var, '(load data first)')
pt_menu.config(width=26)
pt_menu.pack(side=tk.LEFT)

row1 = tk.Frame(fileFrame); row1.pack(fill=tk.X, padx=4, pady=2)
interpolate_temps_var = tk.BooleanVar(value=False)
interpolateCheckbox = tk.Checkbutton(row1, text='Interpolate Temps (No CSV)',
                                     variable=interpolate_temps_var,
                                     command=toggle_temp_file_selection)
interpolateCheckbox.pack(side=tk.LEFT, padx=(0,6))
tempLabel = tk.Label(row1, text='Temp Profile: (Select via Browse)', anchor=W, fg='gray')
tempLabel.pack(side=tk.LEFT, fill=tk.X, expand=True)

row2 = tk.Frame(fileFrame); row2.pack(fill=tk.X, padx=4, pady=2)
importPlotButton = tk.Button(row2, text='Import Plot CSV', command=import_data, width=16)
importPlotButton.pack(side=tk.LEFT, padx=(0,6))
importTempButton = tk.Button(row2, text='Import Temp CSV',
                             command=import_temp_profile_viewer, width=16)
importTempButton.pack(side=tk.LEFT)

outDirRow = tk.Frame(fileFrame); outDirRow.pack(fill=tk.X, padx=4, pady=2)
tk.Button(outDirRow, text='Set Output Folder...', command=set_output_dir,
          width=18).pack(side=tk.LEFT, padx=(0,8))
outputDirLabel = tk.Label(outDirRow, text='Output: (NOT SET — required)',
                          anchor=W, fg='red')
outputDirLabel.pack(side=tk.LEFT, fill=tk.X, expand=True)

# ── Section 2: Geometry Files ─────────────────────────────────────────────
geoFrame = tk.LabelFrame(mainFrame, text='Geometry Files', font=('TkDefaultFont', 9, 'bold'))
geoFrame.pack(fill=tk.X, **PAD)

def _geo_row(parent, label_text, browse_cmd):
    f = tk.Frame(parent); f.pack(fill=tk.X, padx=4, pady=2)
    btn = tk.Button(f, text='Browse...', command=browse_cmd, width=10)
    btn.pack(side=tk.RIGHT, padx=(4, 0))
    lbl = tk.Label(f, text=label_text, anchor=W, fg='gray')
    lbl.pack(side=tk.LEFT, fill=tk.X, expand=True)
    return lbl

layoutLabel  = _geo_row(geoFrame, 'Layout:       (Not Selected)', browse_layout)
matingLabel  = _geo_row(geoFrame, 'Mating Table: (Not Selected)', browse_mating)
furLabel     = _geo_row(geoFrame, 'MLD-FUR:      (Not Selected)', browse_fur)
hpsLabel     = _geo_row(geoFrame, 'MLD-HPS:      (Not Selected)', browse_hps)


# ── Section 3: Calc Parameters + Uncertainty (side by side) ──────────────
midRow = tk.Frame(mainFrame); midRow.pack(fill=tk.X, **PAD)

calcFrame = tk.LabelFrame(midRow, text='Calculation Parameters',
                          font=('TkDefaultFont', 9, 'bold'))
calcFrame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True, padx=(0,4))

def _param_row(parent, label_text, default=''):
    f = tk.Frame(parent); f.pack(fill=tk.X, padx=4, pady=2)
    tk.Label(f, text=label_text, width=24, anchor=W).pack(side=tk.LEFT)
    sv = tk.StringVar(value=default)
    tk.Entry(f, textvariable=sv, width=10).pack(side=tk.LEFT)
    return sv

z_temp_cutoff_var   = _param_row(calcFrame, 'Z-Temp Cutoff (mm):')
z_offset_var        = _param_row(calcFrame, 'Z-Offset (mm):')
sample_pos_var      = _param_row(calcFrame, 'Sample Position (mm):', '')
calc_step_var       = _param_row(calcFrame, 'Calc Step (every N rows):', '1')

# Gas species dropdown
_gas_row = tk.Frame(calcFrame); _gas_row.pack(fill=tk.X, padx=4, pady=2)
tk.Label(_gas_row, text='Gas Species:', width=24, anchor=W).pack(side=tk.LEFT)
gas_species_var = tk.StringVar(value='Nitrogen')
tk.OptionMenu(_gas_row, gas_species_var, 'Nitrogen').pack(side=tk.LEFT)

sigmaFrame = tk.LabelFrame(midRow, text='Uncertainty Inputs',
                           font=('TkDefaultFont', 9, 'bold'))
sigmaFrame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

sigma_p_var = _param_row(sigmaFrame, 'σ_P (% Error):', '0.25')
sigma_v_var = _param_row(sigmaFrame, 'σ_V (% Error):', '5.0')
sigma_t_var = _param_row(sigmaFrame, 'σ_T (K Error):', '2.2')

# ── Section 3b: Furniture Masses & Chemical Sample ─────────────────────
furChemFrame = tk.LabelFrame(mainFrame, text='Furniture Masses & Chemical Sample',
                              font=('TkDefaultFont', 9, 'bold'))
furChemFrame.pack(fill=tk.X, **PAD)

# Top row: build-table button + volume info label
_fcTopRow = tk.Frame(furChemFrame); _fcTopRow.pack(fill=tk.X, padx=4, pady=(4,0))
tk.Button(_fcTopRow, text='Build Table from Layout',
          command=build_fur_table_from_layout, width=24).pack(side=tk.LEFT)
vol_info_label = tk.Label(_fcTopRow, text='',
                           fg='darkblue', font=('TkDefaultFont', 8), anchor=W)
vol_info_label.pack(side=tk.LEFT, padx=(10, 0))

# Column headers
_fcHdr = tk.Frame(furChemFrame); _fcHdr.pack(fill=tk.X, padx=4, pady=(3,0))
for _ht, _hw in [('UID', 16), ('Material', 16), ('Internal', 9), ('Mass (g)', 9)]:
    tk.Label(_fcHdr, text=_ht, width=_hw, anchor=W,
             font=('TkDefaultFont', 8, 'bold')).pack(side=tk.LEFT)
tk.Label(_fcHdr, text='  ← enter measured part mass; Internal = subtracts from gas volume',
         font=('TkDefaultFont', 7), fg='gray').pack(side=tk.LEFT)

# Scrollable table body
_fcCanvasFrame = tk.Frame(furChemFrame); _fcCanvasFrame.pack(fill=tk.X, padx=4, pady=2)
_fcCanvas  = tk.Canvas(_fcCanvasFrame, height=130, bg='#f5f5f5', highlightthickness=0)
_fcScrollb = ttk.Scrollbar(_fcCanvasFrame, orient=tk.VERTICAL, command=_fcCanvas.yview)
fur_table_inner = tk.Frame(_fcCanvas, bg='#f5f5f5')
fur_table_inner.bind('<Configure>',
    lambda e: _fcCanvas.configure(scrollregion=_fcCanvas.bbox('all')))
_fcCanvas.create_window((0, 0), window=fur_table_inner, anchor='nw')
_fcCanvas.configure(yscrollcommand=_fcScrollb.set)
_fcCanvas.pack(side=tk.LEFT, fill=tk.X, expand=True)
_fcScrollb.pack(side=tk.RIGHT, fill=tk.Y)
_fcCanvas.bind('<Enter>',
    lambda e: _fcCanvas.bind_all('<MouseWheel>',
        lambda ev: _fcCanvas.yview_scroll(-1*(1 if ev.delta>0 else -1), 'units')))
_fcCanvas.bind('<Leave>', lambda e: window.bind_all('<MouseWheel>', _on_mousewheel))

# Chemical species row
_chemRow = tk.Frame(furChemFrame); _chemRow.pack(fill=tk.X, padx=4, pady=(6,2))
tk.Label(_chemRow, text='Chemical Species:', width=18, anchor=W).pack(side=tk.LEFT)
chem_species_var = tk.StringVar(value='None')
_chem_opts = ['None','Li3BN2','Li3N','Li3AlN2','Li3GaN2','GaN','Custom']
tk.OptionMenu(_chemRow, chem_species_var, *_chem_opts).pack(side=tk.LEFT, padx=(0,8))
tk.Label(_chemRow, text='Mass (g):', width=9, anchor=W).pack(side=tk.LEFT)
chem_mass_var = tk.StringVar(value='')
tk.Entry(_chemRow, textvariable=chem_mass_var, width=8).pack(side=tk.LEFT, padx=(0,10))
tk.Label(_chemRow, text='Custom Density (g/cm³):', width=22, anchor=W).pack(side=tk.LEFT)
chem_density_var = tk.StringVar(value='')
tk.Entry(_chemRow, textvariable=chem_density_var, width=8).pack(side=tk.LEFT)
tk.Label(_chemRow, text='(Custom species only)',
         fg='gray', font=('TkDefaultFont', 8)).pack(side=tk.LEFT, padx=(4,0))

# ── Section 3c: Pressure Zero Offset ────────────────────────────────────
pzeroFrame = tk.LabelFrame(mainFrame, text='Pressure Zero Offset',
                           font=('TkDefaultFont', 9, 'bold'))
pzeroFrame.pack(fill=tk.X, **PAD)

pzeroRow1 = tk.Frame(pzeroFrame); pzeroRow1.pack(fill=tk.X, padx=4, pady=2)
tk.Label(pzeroRow1, text='Vacuum Window Start (hr):', width=26, anchor=W).pack(side=tk.LEFT)
p_zero_start_var = tk.StringVar(value='')
tk.Entry(pzeroRow1, textvariable=p_zero_start_var, width=8).pack(side=tk.LEFT, padx=(0,16))
tk.Label(pzeroRow1, text='End (hr):', width=10, anchor=W).pack(side=tk.LEFT)
p_zero_end_var = tk.StringVar(value='')
tk.Entry(pzeroRow1, textvariable=p_zero_end_var, width=8).pack(side=tk.LEFT)

pzeroRow2 = tk.Frame(pzeroFrame); pzeroRow2.pack(fill=tk.X, padx=4, pady=2)
tk.Label(pzeroRow2, text='Manual Override (MPa):', width=26, anchor=W).pack(side=tk.LEFT)
p_zero_manual_var = tk.StringVar(value='')
tk.Entry(pzeroRow2, textvariable=p_zero_manual_var, width=8).pack(side=tk.LEFT, padx=(0,10))
tk.Label(pzeroRow2,
         text='Override takes priority over vacuum window. Leave both blank for no offset.',
         anchor=W, fg='gray', font=('TkDefaultFont', 8)).pack(side=tk.LEFT)

# ── Section 4: Action Buttons ─────────────────────────────────────────────
actFrame = tk.LabelFrame(mainFrame, text='Actions', font=('TkDefaultFont', 9, 'bold'))
actFrame.pack(fill=tk.X, **PAD)

actRow = tk.Frame(actFrame); actRow.pack(fill=tk.X, padx=4, pady=4)
graphButton = tk.Button(actRow, text='Generate Graph', command=lambda: processData(), width=16)
graphButton.pack(side=tk.LEFT, padx=(0,4))
tempViewButton = tk.Button(actRow, text='View Temp Profiles', command=open_temp_viewer,
                           width=16, state='disabled')
tempViewButton.pack(side=tk.LEFT, padx=(0,4))
texpansion_button = tk.Button(actRow, text='T Expansion', command=open_texpansion_viewer,
                              width=12, state='disabled')
texpansion_button.pack(side=tk.LEFT, padx=(0,4))
exportPlotButton = tk.Button(actRow, text='Export Plot Data', command=export_plot_data,
                             width=16, state='disabled')
exportPlotButton.pack(side=tk.LEFT, padx=(0,4))
exportTempButton = tk.Button(actRow, text='Export Temp Profiles', command=export_temp_profiles,
                             width=18, state='disabled')
exportTempButton.pack(side=tk.LEFT)
exportHtmlButton = tk.Button(actRow, text='Export HTML Plot', command=export_html_plot,
                             width=16, state='disabled')
exportHtmlButton.pack(side=tk.LEFT, padx=(4,0))


save_plot_button = tk.Button(actRow, text='Save Plot',
                             command=save_plot_image, width=12, state='disabled')
save_plot_button.pack(side=tk.LEFT, padx=(4,0))


# ── Section 5: P_calc ───────────────────────────────────────────────────
pcalcFrame = tk.LabelFrame(mainFrame, text='Back-Calculated Pressure  P_calc',
                           font=('TkDefaultFont', 9, 'bold'))
pcalcFrame.pack(fill=tk.X, **PAD)

pcalcTimeRow = tk.Frame(pcalcFrame); pcalcTimeRow.pack(fill=tk.X, padx=4, pady=2)
tk.Label(pcalcTimeRow, text='Start Time (hr):', width=18, anchor=W).pack(side=tk.LEFT)
p_calc_start_time_var = tk.StringVar(value='')
tk.Entry(pcalcTimeRow, textvariable=p_calc_start_time_var, width=8).pack(side=tk.LEFT, padx=(0,12))
tk.Label(pcalcTimeRow, text='End Time (hr):', width=14, anchor=W).pack(side=tk.LEFT)
p_calc_end_time_var = tk.StringVar(value='')
tk.Entry(pcalcTimeRow, textvariable=p_calc_end_time_var, width=8).pack(side=tk.LEFT)
tk.Label(pcalcTimeRow,
         text='  (leave End blank to calculate over full run)',
         anchor=W, fg='gray', font=('TkDefaultFont', 8)).pack(side=tk.LEFT)
tk.Label(pcalcTimeRow, text='  Step (every N rows):', width=20, anchor=W).pack(side=tk.LEFT)
p_calc_step_var = tk.StringVar(value='1')
tk.Entry(pcalcTimeRow, textvariable=p_calc_step_var, width=6).pack(side=tk.LEFT)

pcalcRow = tk.Frame(pcalcFrame); pcalcRow.pack(fill=tk.X, padx=4, pady=4)
pcalc_button = tk.Button(pcalcRow, text='Calculate P_calc',
                         command=calculate_pcalc, width=18, state='disabled')
pcalc_button.pack(side=tk.LEFT, padx=(0,6))
tk.Button(pcalcRow, text='Clear P_calc', command=clear_pcalc,
          width=12).pack(side=tk.LEFT)

pcalcOffsetRow = tk.Frame(pcalcFrame); pcalcOffsetRow.pack(fill=tk.X, padx=4, pady=2)
tk.Label(pcalcOffsetRow, text='P Offset (MPa):', width=18, anchor=W).pack(side=tk.LEFT)
p_offset_var = tk.StringVar(value='')
tk.Entry(pcalcOffsetRow, textvariable=p_offset_var, width=10).pack(side=tk.LEFT, padx=(0,8))
tk.Label(pcalcOffsetRow,
         text='Leave blank to use auto offset. Re-run Calculate P_calc after changing.',
         anchor=W, fg='gray', font=('TkDefaultFont', 8)).pack(side=tk.LEFT)

pcalc_info_label = tk.Label(pcalcFrame, text='', anchor=W, justify=LEFT,
                             fg='darkblue', wraplength=680,
                             font=('TkDefaultFont', 8))
pcalc_info_label.pack(fill=tk.X, padx=6, pady=(0,4))


# ── Section 5: Exponential Fit ────────────────────────────────────────────
fitFrame = tk.LabelFrame(mainFrame, text='Exponential Fit', font=('TkDefaultFont', 9, 'bold'))
fitFrame.pack(fill=tk.X, **PAD)

fitRow1 = tk.Frame(fitFrame); fitRow1.pack(fill=tk.X, padx=4, pady=2)
tk.Label(fitRow1, text='Fit Start Time (hr):', width=20, anchor=W).pack(side=tk.LEFT)
fit_start_time_var = tk.StringVar(value='')
tk.Entry(fitRow1, textvariable=fit_start_time_var, width=10).pack(side=tk.LEFT, padx=(0,20))
tk.Label(fitRow1, text='Fit End Time (hr):', width=18, anchor=W).pack(side=tk.LEFT)
fit_end_time_var = tk.StringVar(value='')
tk.Entry(fitRow1, textvariable=fit_end_time_var, width=10).pack(side=tk.LEFT)

fitRow2 = tk.Frame(fitFrame); fitRow2.pack(fill=tk.X, padx=4, pady=2)
export_fit_button = tk.Button(fitRow2, text='Export Fit CSV',
    command=export_fit_csv, width=16, state='disabled')
export_fit_button.pack(side=tk.LEFT)

fitRow3 = tk.Frame(fitFrame); fitRow3.pack(fill=tk.X, padx=4, pady=2)
calculateFitButton = tk.Button(fitRow3, text='Calculate Fit',
                               command=calculate_exponential_fit, width=14, state='disabled')
calculateFitButton.pack(side=tk.LEFT)
tk.Button(fitRow3, text='Clear Fit', command=clear_fit,
          width=10).pack(side=tk.LEFT, padx=(6,0))

fit_results_label = tk.Label(fitFrame, text='', anchor=W, justify=LEFT,
                             fg='darkgreen', wraplength=680)
fit_results_label.pack(fill=tk.X, padx=6, pady=(0,4))

# ── Section 5b: Plot Options ─────────────────────────────────────────────
plotOptFrame = tk.LabelFrame(mainFrame, text='Plot Options',
                              font=('TkDefaultFont', 9, 'bold'))
plotOptFrame.pack(fill=tk.X, **PAD)

thicknessRow = tk.Frame(plotOptFrame); thicknessRow.pack(fill=tk.X, padx=4, pady=4)
tk.Label(thicknessRow, text='Curve Thickness:', width=18, anchor='w').pack(side=tk.LEFT)
line_thickness_var = tk.DoubleVar(value=1.8)
tk.Scale(thicknessRow, from_=0.5, to=5.0, resolution=0.25,
         orient=tk.HORIZONTAL, variable=line_thickness_var,
         length=220, showvalue=True).pack(side=tk.LEFT)
tk.Label(thicknessRow, text='pt  (applies to all curves on next Generate Graph / overlay)',
         fg='gray', font=('TkDefaultFont', 8)).pack(side=tk.LEFT, padx=(6,0))


# ── Section 5c: Apply Smoothing ──────────────────────────────────────────
smoothFrame = tk.LabelFrame(mainFrame, text='Apply Smoothing',
                             font=('TkDefaultFont', 9, 'bold'))
smoothFrame.pack(fill=tk.X, **PAD)
_smRow = tk.Frame(smoothFrame); _smRow.pack(fill=tk.X, padx=4, pady=4)
apply_smoothing_var = tk.BooleanVar(value=False)
tk.Checkbutton(_smRow,
               text='Smooth temperature profile  (2nd-degree polynomial, 200 mm window)',
               variable=apply_smoothing_var).pack(side=tk.LEFT)

# ── Section 6: dn/dt Calculator ─────────────────────────────────────────
dndtFrame = tk.LabelFrame(mainFrame, text='Rate of Change  dn/dt',
                          font=('TkDefaultFont', 9, 'bold'))
dndtFrame.pack(fill=tk.X, **PAD)

dndtRow1 = tk.Frame(dndtFrame); dndtRow1.pack(fill=tk.X, padx=4, pady=2)
tk.Label(dndtRow1, text='Period Start (hr):', width=18, anchor=W).pack(side=tk.LEFT)
dn_dt_start_var = tk.StringVar(value='')
tk.Entry(dndtRow1, textvariable=dn_dt_start_var, width=8).pack(side=tk.LEFT, padx=(0,12))
tk.Label(dndtRow1, text='Period End (hr):', width=16, anchor=W).pack(side=tk.LEFT)
dn_dt_end_var = tk.StringVar(value='')
tk.Entry(dndtRow1, textvariable=dn_dt_end_var, width=8).pack(side=tk.LEFT, padx=(0,12))
tk.Label(dndtRow1, text='Interval (hr):', width=13, anchor=W).pack(side=tk.LEFT)
dn_dt_interval_var = tk.StringVar(value='')
tk.Entry(dndtRow1, textvariable=dn_dt_interval_var, width=8).pack(side=tk.LEFT)



dndtRow2 = tk.Frame(dndtFrame); dndtRow2.pack(fill=tk.X, padx=4, pady=2)
dn_dt_button = tk.Button(dndtRow2, text='Calculate dn/dt',
                         command=calculate_dn_dt, width=16, state='disabled')
dn_dt_button.pack(side=tk.LEFT)
tk.Button(dndtRow2, text='Clear dn/dt', command=clear_dndt,
          width=12).pack(side=tk.LEFT, padx=(6,0))

dn_dt_results_label = tk.Label(dndtFrame, text='', anchor=W,
                               justify=LEFT, fg='darkorchid',
                               wraplength=680)
dn_dt_results_label.pack(fill=tk.X, padx=6, pady=(0,4))


# ── Progress Bars ─────────────────────────────────────────────────────────
progFrame = tk.Frame(mainFrame); progFrame.pack(fill=tk.X, padx=6, pady=2)
tk.Label(progFrame, text='File:', anchor=W, width=5).pack(side=tk.LEFT)
fileProgress = Progressbar(progFrame, orient=HORIZONTAL, length=150, mode='determinate')
fileProgress.pack(side=tk.LEFT, padx=(0,16))
tk.Label(progFrame, text='Graph:', anchor=W, width=6).pack(side=tk.LEFT)
graphProgress = Progressbar(progFrame, orient=HORIZONTAL, length=250, mode='determinate')
graphProgress.pack(side=tk.LEFT)

window.attributes('-topmost', False)

# == Section 8: Pressure Trace Synthesizer ====================================
synthFrame = tk.LabelFrame(mainFrame, text='Pressure Trace Synthesizer',
                            font=('TkDefaultFont', 9, 'bold'))
synthFrame.pack(fill=tk.X, **PAD)

tk.Label(synthFrame,
         text='Compute a forward P(t) trace from AC geometry + temperature '
              'schedule.  No raw data file required.',
         anchor=W, fg='#444', font=('TkDefaultFont', 8),
         wraplength=680).pack(fill=tk.X, padx=6, pady=(2, 0))

# -- Volume computation row ---------------------------------------------------
synthVolRow = tk.Frame(synthFrame); synthVolRow.pack(fill=tk.X, padx=4, pady=2)
tk.Button(synthVolRow, text='Compute Volume Profile',
          command=_synth_compute_volume, width=22).pack(side=tk.LEFT)
synth_vol_label = tk.Label(synthVolRow,
                            text='(click after loading geometry files)',
                            anchor=W, fg='gray', font=('TkDefaultFont', 8))
synth_vol_label.pack(side=tk.LEFT, padx=(8, 0))

# -- Initial conditions -------------------------------------------------------
synthIC = tk.Frame(synthFrame); synthIC.pack(fill=tk.X, padx=4, pady=2)
tk.Label(synthIC, text='P0 (MPa):', width=10, anchor=W).pack(side=tk.LEFT)
synth_p0_var = tk.StringVar(value='')
tk.Entry(synthIC, textvariable=synth_p0_var, width=8).pack(side=tk.LEFT, padx=(0, 10))
tk.Label(synthIC, text='T0 (degC):', width=10, anchor=W).pack(side=tk.LEFT)
synth_t0_var = tk.StringVar(value='25')
tk.Entry(synthIC, textvariable=synth_t0_var, width=7).pack(side=tk.LEFT, padx=(0, 10))
tk.Label(synthIC, text='T_ambient (degC):', width=17, anchor=W).pack(side=tk.LEFT)
synth_ambient_var = tk.StringVar(value='25')
tk.Entry(synthIC, textvariable=synth_ambient_var, width=7).pack(side=tk.LEFT, padx=(0, 10))

# -- Time parameters ----------------------------------------------------------
synthTimeRow = tk.Frame(synthFrame); synthTimeRow.pack(fill=tk.X, padx=4, pady=2)
tk.Label(synthTimeRow, text='Total time (hr):', width=15, anchor=W).pack(side=tk.LEFT)
synth_total_time_var = tk.StringVar(value='')
tk.Entry(synthTimeRow, textvariable=synth_total_time_var,
         width=8).pack(side=tk.LEFT, padx=(0, 12))
tk.Label(synthTimeRow, text='Time step (hr):', width=14, anchor=W).pack(side=tk.LEFT)
synth_dt_var = tk.StringVar(value='0.05')
tk.Entry(synthTimeRow, textvariable=synth_dt_var, width=8).pack(side=tk.LEFT)

# -- Furnace schedule text widget ---------------------------------------------
synthSchedOuter = tk.LabelFrame(
    synthFrame,
    text='Furnace Schedule  (one breakpoint per line:  time_hr, T_setpoint_degC)',
    font=('TkDefaultFont', 8))
synthSchedOuter.pack(fill=tk.X, padx=6, pady=(4, 2))
synth_schedule_text = tk.Text(synthSchedOuter, height=5, width=32,
                               font=('Courier New', 9), relief='sunken', bd=1)
synth_schedule_text.pack(side=tk.LEFT, padx=(4, 6), pady=4)
synth_schedule_text.insert('1.0',
    '# time_hr, T_setpoint_degC\n'
    '0.0, 25\n'
    '0.5, 25\n'
    '1.5, 700\n'
    '5.0, 700\n'
    '6.0, 25\n')
tk.Label(synthSchedOuter,
         text='Linear interpolation between breakpoints.\n'
              'Lines starting with # are ignored.',
         anchor=W, justify=tk.LEFT, fg='gray',
         font=('TkDefaultFont', 8)).pack(side=tk.LEFT, padx=(0, 6))

# -- Spatial temperature distribution mode ------------------------------------
synthModeFrame = tk.LabelFrame(
    synthFrame,
    text='Spatial Temperature Distribution along Autoclave',
    font=('TkDefaultFont', 8))
synthModeFrame.pack(fill=tk.X, padx=6, pady=(4, 2))
synth_profile_mode_var = tk.StringVar(value='profile_csv')

# Option 1: Uniform
tk.Radiobutton(synthModeFrame,
               text='Uniform  (entire gas volume at T_setpoint)',
               variable=synth_profile_mode_var,
               value='uniform').pack(anchor=W, padx=6)

# Option 2: Gradient
tk.Radiobutton(synthModeFrame,
               text='Gradient  (linear: T_ambient at z=0  to  T_setpoint at z=end)',
               variable=synth_profile_mode_var,
               value='gradient').pack(anchor=W, padx=6)

# Option 3: Temp Profile CSV (the proper calibrated spatial shape)
synthCsvRow = tk.Frame(synthModeFrame); synthCsvRow.pack(anchor=W, padx=6, pady=(0, 2))
tk.Radiobutton(synthCsvRow,
               text='Temp Profile CSV  (calibrated T(z) lookup — same format as File Selection)',
               variable=synth_profile_mode_var,
               value='profile_csv').pack(side=tk.LEFT)

synthCsvBtnRow = tk.Frame(synthModeFrame); synthCsvBtnRow.pack(anchor=W, padx=22, pady=(0, 4))
tk.Button(synthCsvBtnRow, text='Use Loaded Profile',
          command=synth_use_loaded_temp_profile,
          width=18).pack(side=tk.LEFT, padx=(0, 4))
tk.Button(synthCsvBtnRow, text='Browse Different File...',
          command=browse_synth_lookup_table,
          width=22).pack(side=tk.LEFT, padx=(0, 6))
synth_lookup_label = tk.Label(synthCsvBtnRow,
                               text='(no profile CSV loaded for synthesizer)',
                               anchor=W, fg='gray',
                               font=('TkDefaultFont', 8))
synth_lookup_label.pack(side=tk.LEFT)

# Option 4: Simple 2-col shape CSV (legacy)
synthLoadedRow = tk.Frame(synthModeFrame); synthLoadedRow.pack(anchor=W, padx=6, pady=(0, 4))
tk.Radiobutton(synthLoadedRow,
               text='Simple shape CSV  (z_mm, T_C — scaled proportionally)',
               variable=synth_profile_mode_var,
               value='loaded').pack(side=tk.LEFT)
tk.Button(synthLoadedRow, text='Browse...',
          command=browse_synth_spatial_profile,
          width=10).pack(side=tk.LEFT, padx=(6, 4))
synth_spatial_label = tk.Label(synthLoadedRow, text='(none loaded)',
                                anchor=W, fg='gray', font=('TkDefaultFont', 8))
synth_spatial_label.pack(side=tk.LEFT)

# -- n readout ----------------------------------------------------------------
synthNRow = tk.Frame(synthFrame); synthNRow.pack(fill=tk.X, padx=4, pady=(0, 2))
synth_n_label = tk.Label(synthNRow, text='', anchor=W, fg='#004080',
                          font=('TkDefaultFont', 8))
synth_n_label.pack(side=tk.LEFT)

# -- Action buttons -----------------------------------------------------------
synthActRow = tk.Frame(synthFrame); synthActRow.pack(fill=tk.X, padx=4, pady=4)
synth_run_button = tk.Button(synthActRow, text='Run Synthesizer',
                              command=synthesize_pressure_trace, width=18)
synth_run_button.pack(side=tk.LEFT, padx=(0, 6))
tk.Button(synthActRow, text='Clear Synth Trace',
          command=clear_synth_trace, width=16).pack(side=tk.LEFT)

# -- Status label -------------------------------------------------------------
synth_status_label = tk.Label(synthFrame, text='', anchor=W, justify=tk.LEFT,
                               fg='darkgreen', wraplength=680,
                               font=('TkDefaultFont', 8))
synth_status_label.pack(fill=tk.X, padx=6, pady=(0, 4))
# Size window for landing page
window.update_idletasks()
_lw, _lh = 500, 420
_lx = (window.winfo_screenwidth() - _lw) // 2
_ly = (window.winfo_screenheight() - _lh) // 2
window.geometry(f'{_lw}x{_lh}+{_lx}+{_ly}')
window.mainloop()
